# Primer Entregable: base de datos, EDA y modelo base

**Predicción de brotes de dengue en Colombia con cuatro semanas de anticipación**

**Maestría en Matemáticas · Proyecto de Aprendizaje Automático · Daniel Serrano Romero**

Este libro establece las bases del proyecto: (i) la selección y documentación de la base de datos, (ii) un análisis
exploratorio exhaustivo que guía cada decisión de modelado, y (iii) un modelo base (regresión logística) comparado con
líneas base triviales. Todos los capítulos son cuadernos ejecutados de principio a fin; sus figuras son interactivas.

## Ruta de trabajo

El objetivo es categórico (brote / no brote en la semana $t+4$), así que se sigue la **ruta A (clasificación)** de la
guía. Como el dataset tiene tiempo y, tras agregar los centroides municipales del DANE, también coordenadas, se añaden
las secciones **2.6 (temporal), 2.7 (espacial) y 2.8 (espacio-temporal)** con la partición correspondiente: cronológica
para el futuro y por bloques espaciales para zonas nuevas.

**Dataset y problema** → **reserva de la prueba 2023–2024** (purga de 4 semanas) → **EDA** del objetivo, uni-, bi- y
multivariado → **auditoría de fuga** → componentes **temporal, espacial y espacio-temporal** → **preprocesamiento** en
`Pipeline` → **regresión logística frente a líneas base** → **nota crítica** y diagnóstico de residuos.

## Resumen

| Elemento | Resultado |
|---|---|
| Datos | Panel de 376 municipios × 782 semanas (2010–2024): 294 032 filas, 21 columnas, sin nulos ni duplicados |
| Objetivo | Brote en $t+4$ según el canal endémico; ≈ 10 % de brotes en el entrenamiento, con fuerte variación anual |
| Partición | Entrenamiento 2015–2021 · validación 2022 · prueba 2023–2024, con purga de 4 semanas en cada frontera |
| Hallazgos clave | La historia de casos domina; el clima actúa con rezago y de forma no lineal; El Niño cuadruplica el riesgo; autocorrelación espacial (Moran ≈ 0,31) e interacción espacio-tiempo (Knox) |
| Faltantes | La temperatura está imputada en el 64 % de las semanas (MAR estructural: municipios pequeños sin estación) |
| Modelo base | Regresión logística L2: AUC-ROC ≈ 0,885 y AUC-PR ≈ 0,81 en la prueba, frente a 0,79 / 0,59 de la persistencia y 0,50 / 0,30 del Dummy |

## Correspondencia con la guía del entregable

| Requisito de la guía | Dónde está |
|---|---|
| 1. Problema, justificación, fuente y licencia, diccionario, estructura | Capítulo 1 |
| 1. Tamaño de la muestra, $n/p$, casos por clase, entidades independientes | Capítulo 1 (incluye tamaño efectivo por ICC) |
| 1. Calidad: faltantes (patrón y mecanismo, prueba de Little), duplicados, atípicos, valores imposibles, sesgos | Capítulo 1 |
| 1. Consideraciones éticas | Capítulo 1 |
| Reservar la prueba antes de cualquier decisión | Capítulo 2 |
| 2.1 Variable objetivo | Capítulo 2 |
| 2.2 Análisis unidimensional | Capítulo 3 |
| 2.3 Análisis bidimensional (con tamaños de efecto, Holm y FDR, información mutua, VIF) | Capítulo 4 |
| 2.4 Análisis multivariado (PCA, t-SNE, Mahalanobis, Isolation Forest, agrupamiento) | Capítulo 5 |
| 2.5 Auditoría de fuga | Capítulo 6 |
| 2.6 Componente temporal (STL, ADF/KPSS, ACF/PACF, regímenes, deriva) | Capítulo 7 |
| 2.7 Componente espacial (CRS, mapas, Clark-Evans, Ripley, DBSCAN, Moran, LISA, Gi*, MAUP) | Capítulo 8 |
| 2.8 Componente espacio-temporal (focos emergentes, Knox) | Capítulo 9 |
| 2.9 Preprocesamiento en Pipeline | Capítulo 10 |
| 3. Modelo base, líneas base, métricas con IC, residuos, curva de aprendizaje, coeficientes | Capítulo 11 |
| Nota crítica (desempeño ≥ 80 %) | Capítulo 11 |
| Entregables: Jupyter Book, base de datos, dependencias y semillas | Conclusiones |


# Base de datos

Este capítulo cubre la sección 1 de la guía: problema de investigación, justificación, fuente y licencia, diccionario de variables, estructura, tamaño de la muestra, calidad de los datos y consideraciones éticas.

In [1]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
raw = pd.read_csv(cf.RUTA_DATOS)  # Archivo original, sin transformar
raw["week_start"] = pd.to_datetime(raw["week_start"])  # Fecha de inicio de la semana epidemiológica
m = d.municipios()  # 376 municipios con coordenadas DANE y región
print(f"Archivo: {cf.RUTA_DATOS.name} · {raw.shape[0]:,} filas × {raw.shape[1]} columnas")  # Dimensión

Archivo: final_dataset.csv · 294,032 filas × 21 columnas


## Problema de investigación

**Pregunta.** ¿Es posible anticipar, con cuatro semanas de margen, si un municipio colombiano entrará en situación de brote de dengue, usando solo la información disponible en la semana actual (historia de casos, clima, vegetación, El Niño, población y movilidad)?

**Definición operativa.** Una semana es de brote si los casos superan el tercer cuartil de las mismas semanas (± 2) de los cinco años anteriores y hay al menos tres casos (canal endémico, método de cuartiles). El objetivo de la fila $(m, t)$ es el brote de la semana $t+4$:

$$y_{m,t} = \mathbb{1}\left[c_{m,t+4} > Q_{0.75}\{c_{m,\,t+4-52k+j}\}_{k=1..5,\;j=-2..2} \;\wedge\; c_{m,t+4} \ge 3\right].$$

**Tipo de tarea.** Clasificación binaria probabilística sobre un panel espacio-temporal (ruta A de la guía, con las secciones 2.6 a 2.8 por tener tiempo y espacio).

## Justificación de la selección del dataset

* **Relevancia.** El dengue es la arbovirosis de mayor carga en Colombia; 2024 fue el año con más casos de la serie. Las secretarías de salud deciden cada semana dónde fumigar, eliminar criaderos y reforzar servicios: una alerta con cuatro semanas de margen tiene uso directo.
* **Granularidad.** Municipio × semana epidemiológica, la misma resolución de la vigilancia oficial.
* **Riqueza.** Combina la notificación de casos con clima, vegetación (NDVI), el índice ONI de El Niño, población y flujo de personas, lo que permite estudiar componentes temporales, espaciales y espacio-temporales.
* **Tamaño.** Muy por encima del mínimo de 20 000 observaciones exigido por la guía (ver abajo).
* **Dificultad adecuada.** El objetivo no es trivial: la persistencia ("si hay brote hoy, habrá brote en cuatro semanas") ya acierta mucho, y el reto es mejorar el **ordenamiento del riesgo**, no la exactitud.

## Fuente de los datos y licencia

| Componente | Origen | Licencia / condiciones de uso |
|---|---|---|
| Panel integrado `final_dataset.csv` | Archivo del proyecto (casos, clima, NDVI, ONI, población y flujo ya integrados por municipio y semana) | Ver nota |
| Casos de dengue | Sistema Nacional de Vigilancia en Salud Pública (SIVIGILA), Instituto Nacional de Salud | Datos abiertos del Estado colombiano (Ley 1712 de 2014) |
| Índice ONI (El Niño) | NOAA Climate Prediction Center | Dominio público (gobierno de EE. UU.) |
| Centroides municipales | DANE, DIVIPOLA, corte 30-dic-2024: [datos.gov.co/resource/gdxc-w37w](https://www.datos.gov.co/Mapas-Nacionales/DIVIPOLA-C-digos-municipios/gdxc-w37w) | CC BY-SA 4.0 |

:::{note}
Las coordenadas no venían en el archivo original: se agregaron desde la DIVIPOLA del DANE (unión por el código municipal; los 376 municipios del panel se encontraron) para poder hacer el análisis espacial de las secciones 2.7 y 2.8. El panel integrado se distribuye con el repositorio del proyecto en `data/final_dataset.csv`.
:::

## Diccionario de variables

In [2]:
dic = pd.DataFrame([  # Tipo, unidad y significado de las 21 columnas originales
    ("COD_MUN_N", "Identificador", "código DANE", "Municipio (5 dígitos; los dos primeros son el departamento)"),
    ("ANO", "Temporal", "año", "Año epidemiológico"),
    ("SEMANA", "Temporal", "semana", "Semana epidemiológica (1–52; 53 en 2015 y 2020)"),
    ("casos_totales", "Conteo", "casos/semana", "Casos de dengue notificados en la semana"),
    ("casos_0_4", "Conteo", "casos/semana", "Casos en menores de 5 años"),
    ("casos_5_14", "Conteo", "casos/semana", "Casos de 5 a 14 años"),
    ("casos_15_64", "Conteo", "casos/semana", "Casos de 15 a 64 años"),
    ("casos_65_plus", "Conteo", "casos/semana", "Casos de 65 años o más"),
    ("casos_m", "Conteo", "casos/semana", "Casos en hombres"),
    ("casos_f", "Conteo", "casos/semana", "Casos en mujeres"),
    ("week_start", "Temporal", "fecha", "Lunes de inicio de la semana epidemiológica"),
    ("MES", "Temporal", "mes", "Mes de la semana (1–12)"),
    ("temp_mean", "Continua", "°C", "Temperatura media semanal (imputada si temp_mean_missing = 1)"),
    ("prec_total", "Continua", "mm/semana", "Precipitación acumulada en la semana"),
    ("ndvi_mean", "Continua", "índice [−1, 1]", "Índice de vegetación de diferencia normalizada (valor mensual)"),
    ("oni_anom", "Continua", "°C", "Anomalía del Índice Oceánico de El Niño (> 0,5 El Niño; < −0,5 La Niña)"),
    ("oni_total", "Continua", "°C", "Temperatura superficial del mar en la región Niño 3.4"),
    ("Flujo_in", "Continua", "personas (escala del dataset)", "Flujo de personas que ingresan al municipio"),
    ("temp_mean_missing", "Binaria", "0/1", "1 si la temperatura de la semana fue imputada"),
    ("time_idx", "Temporal", "índice", "Número de la semana desde enero de 2010 (0–781)"),
    ("poblacion", "Conteo", "habitantes", "Población del municipio en el año"),
], columns=["variable", "tipo", "unidad", "significado"])  # Tabla
dic["dtype"] = dic["variable"].map(raw.dtypes.astype(str))  # Tipo de dato en pandas
dic["valores únicos"] = dic["variable"].map(raw.nunique())  # Cardinalidad
dic

,variable,tipo,unidad,significado,dtype,valores únicos
0,COD_MUN_N,Identificador,código DANE,Municipio (5 dígitos; los dos primeros son el ...,int64,376
1,ANO,Temporal,año,Año epidemiológico,int64,15
2,SEMANA,Temporal,semana,Semana epidemiológica (1–52; 53 en 2015 y 2020),int64,53
3,casos_totales,Conteo,casos/semana,Casos de dengue notificados en la semana,int64,543
4,casos_0_4,Conteo,casos/semana,Casos en menores de 5 años,int64,85
5,casos_5_14,Conteo,casos/semana,Casos de 5 a 14 años,int64,240
6,casos_15_64,Conteo,casos/semana,Casos de 15 a 64 años,int64,381
7,casos_65_plus,Conteo,casos/semana,Casos de 65 años o más,int64,71
8,casos_m,Conteo,casos/semana,Casos en hombres,int64,346
9,casos_f,Conteo,casos/semana,Casos en mujeres,int64,343


Además del archivo original, el análisis usa tres columnas **derivadas**: el umbral del canal endémico, el indicador de brote y el objetivo `y` (brote en $t+4$), y las 34 variables predictoras del capítulo de preprocesamiento. Las coordenadas, el nombre, el departamento y la región vienen de la DIVIPOLA.

## Estructura de los datos

In [3]:
n_mun, n_sem = raw["COD_MUN_N"].nunique(), raw["time_idx"].nunique()  # Entidades y periodos
balanceado = (raw.groupby("COD_MUN_N").size() == n_sem).all()  # ¿Todas las entidades tienen todas las semanas?
pd.Series({  # Resumen estructural
    "Unidad de observación": "municipio × semana epidemiológica",
    "Nivel de agregación": "conteos semanales por municipio (no hay registros individuales)",
    "Tipo de datos": "panel espacio-temporal (longitudinal con coordenadas)",
    "Municipios (entidades)": n_mun,
    "Departamentos": raw["COD_MUN_N"].floordiv(1000).nunique(),
    "Semanas": n_sem,
    "Periodo": f"{raw['week_start'].min().date()} a {raw['week_start'].max().date()}",
    "Panel balanceado": balanceado,
}, name="valor").to_frame()

,valor
Unidad de observación,municipio × semana epidemiológica
Nivel de agregación,conteos semanales por municipio (no hay regist...
Tipo de datos,panel espacio-temporal (longitudinal con coord...
Municipios (entidades),376
Departamentos,32
Semanas,782
Periodo,2010-01-04 a 2024-12-23
Panel balanceado,True


## Tamaño de la muestra

In [4]:
df = d.panel()  # Panel con canal endémico, objetivo, variables y periodo de cada fila
mod = d.modelado(df)  # Filas con objetivo y variables completas (2015–2024, sin filas purgadas)
tr, va, te = d.particion(mod)  # Entrenamiento, validación y prueba
p = len(cf.FEATURES)  # Número de variables predictoras
pd.Series({  # Tamaños relevantes
    "Filas del archivo": f"{len(raw):,}",
    "Columnas originales": raw.shape[1],
    "Filas aptas para modelar (2015–2024)": f"{len(mod):,}",
    "Filas de entrenamiento (2015–2021)": f"{len(tr):,}",
    "Variables predictoras (p)": p,
    "Relación n/p (entrenamiento)": f"{len(tr) / p:,.0f}",
    "Brotes en entrenamiento (clase minoritaria)": f"{int(tr[cf.TARGET].sum()):,}",
    "No brotes en entrenamiento": f"{int((1 - tr[cf.TARGET]).sum()):,}",
}, name="valor").to_frame()

,valor
Filas del archivo,"294,032"
Columnas originales,21
Filas aptas para modelar (2015–2024),"188,376"
Filas de entrenamiento (2015–2021),"132,728"
Variables predictoras (p),34
Relación n/p (entrenamiento),"3,904"
Brotes en entrenamiento (clase minoritaria),"13,644"
No brotes en entrenamiento,"119,084"


### Tamaño efectivo: las filas no son independientes

Las 782 semanas de un municipio están correlacionadas. Con el coeficiente de correlación intraclase (ICC) del brote dentro de cada municipio, el **efecto de diseño** $\text{DEFF} = 1 + (\bar m - 1)\,\text{ICC}$ indica cuántas filas equivalen a una observación independiente.

In [5]:
def icc_anova(valores, grupos):  # ICC(1) por análisis de varianza de un factor
    g = pd.DataFrame({"v": valores, "g": grupos})  # Tabla
    k = g.groupby("g").size().mean()  # Tamaño medio de grupo
    medias = g.groupby("g")["v"].transform("mean")  # Media de cada grupo
    msb = ((medias - g["v"].mean()) ** 2).sum() / (g["g"].nunique() - 1)  # Cuadrado medio entre grupos
    msw = ((g["v"] - medias) ** 2).sum() / (len(g) - g["g"].nunique())  # Cuadrado medio dentro de grupos
    return (msb - msw) / (msb + (k - 1) * msw), k  # ICC y tamaño medio

icc, k = icc_anova(tr[cf.TARGET], tr["COD_MUN_N"])  # ICC del objetivo en entrenamiento
deff = 1 + (k - 1) * icc  # Efecto de diseño
print(f"ICC del brote dentro de cada municipio: {icc:.3f}")  # Correlación intraclase
print(f"Semanas por municipio en entrenamiento: {k:.0f} · efecto de diseño: {deff:.1f}")  # DEFF
print(f"Tamaño efectivo aproximado: {len(tr) / deff:,.0f} observaciones independientes (de {len(tr):,} filas)")  # n efectivo

ICC del brote dentro de cada municipio: 0.075
Semanas por municipio en entrenamiento: 353 · efecto de diseño: 27.5
Tamaño efectivo aproximado: 4,823 observaciones independientes (de 132,728 filas)


**Interpretación.** El panel supera con holgura el mínimo de 20 000 observaciones y la relación $n/p$ es muy alta, así que el sobreajuste por falta de datos no es una preocupación para un modelo lineal. Pero el tamaño efectivo es mucho menor que el número de filas: las semanas de un mismo municipio se parecen entre sí. Por eso la incertidumbre de las métricas se estima remuestreando **municipios completos** (bootstrap por conglomerados) y no filas, y la validación nunca mezcla semanas al azar.

## Calidad de los datos

### Valores faltantes: patrón

In [6]:
print(f"Celdas nulas en el archivo: {int(raw.isna().sum().sum())}")  # No hay nulos explícitos
print(f"Semanas con temperatura imputada: {raw['temp_mean_missing'].mean():.1%}")  # Faltantes ocultos
frac = raw.groupby("COD_MUN_N")["temp_mean_missing"].mean()  # Fracción imputada por municipio
grupo = pd.cut(frac, [-0.01, 0, 0.99, 1], labels=["nunca imputada", "parcialmente", "siempre imputada"])  # Tres patrones
grupo.value_counts().rename("municipios").to_frame()

Celdas nulas en el archivo: 0
Semanas con temperatura imputada: 64.3%


,municipios
temp_mean_missing,
siempre imputada,226
parcialmente,129
nunca imputada,21


In [7]:
matriz = raw.pivot_table(index="COD_MUN_N", columns="ANO", values="temp_mean_missing", aggfunc="mean")  # Municipio × año
matriz = matriz.loc[frac.sort_values().index]  # Ordena por fracción imputada
fig = px.imshow(matriz, aspect="auto", color_continuous_scale=["#f8fafc", ROJO], zmin=0, zmax=1,  # Mapa de calor
                labels=dict(x="Año", y="Municipio (ordenado)", color="Fracción imputada"))
fig.update_yaxes(showticklabels=False)  # Sin etiquetas de municipio
fig.update_layout(title="Matriz de faltantes: temperatura imputada por municipio y año")  # Título
mostrar(fig, 520)  # Muestra

In [8]:
imp = raw.loc[raw["temp_mean_missing"] == 1, "temp_mean"]  # Temperaturas imputadas
obs = raw.loc[raw["temp_mean_missing"] == 0, "temp_mean"]  # Temperaturas observadas
bordes = np.linspace(9, 35, 105)  # Intervalos de 0,25 °C (histograma precalculado: la figura no embebe 294 000 valores)
centros = (bordes[:-1] + bordes[1:]) / 2  # Centro de cada intervalo
fig = go.Figure([go.Bar(x=centros, y=np.histogram(obs, bordes, density=True)[0], name="Observada", marker_color=AZUL, opacity=0.6),  # Observada
                 go.Bar(x=centros, y=np.histogram(imp, bordes, density=True)[0], name="Imputada", marker_color=ROJO, opacity=0.6)])  # Imputada
fig.update_layout(barmode="overlay", bargap=0, title="Distribución de la temperatura observada frente a la imputada",  # Formato
                  xaxis_title="°C", yaxis_title="densidad")
mostrar(fig, 380)  # Muestra
print(f"Imputada: media {imp.mean():.2f} °C, desviación {imp.std():.2f} · Observada: media {obs.mean():.2f} °C, desviación {obs.std():.2f}")  # Resumen

Imputada: media 22.49 °C, desviación 1.25 · Observada: media 24.48 °C, desviación 4.50


**Patrón.** No hay celdas nulas, pero la temperatura está imputada en el 64 % de las semanas y el patrón es **por municipio**, no por semana: 226 municipios nunca tienen temperatura medida, solo 21 la tienen siempre y 129 la tienen parcialmente. La fracción imputada es casi idéntica todos los años. Los valores imputados se concentran alrededor de 22,4 °C con muy poca variación: la imputación original es prácticamente una constante y no conserva la diferencia climática entre municipios.

### Valores faltantes: mecanismo (MCAR, MAR o MNAR)

Se marcan como faltantes las temperaturas imputadas y se aplican tres evidencias: la prueba de Little (H0: MCAR), la comparación de distribuciones entre registros completos e incompletos (diferencia estandarizada) y un modelo que intenta predecir qué municipios carecen de temperatura a partir de variables observadas.

In [9]:
eda = d.entrenamiento_eda(df)  # Decisiones del EDA: solo semanas 2010–2021 (la prueba queda sellada)
X = pd.DataFrame({"temp_mean": eda["temp_mean"].where(eda["temp_mean_missing"] == 0),  # Temperatura con faltantes
                  "prec_total": eda["prec_total"], "ndvi_mean": eda["ndvi_mean"], "oni_anom": eda["oni_anom"],  # Observadas
                  "log_poblacion": np.log(eda["poblacion"]), "log_casos": np.log1p(eda["casos_totales"])})  # Observadas
muestra = X.sample(20000, random_state=cf.SEMILLA)  # Muestra para la prueba (con n enorme siempre rechaza)
d2, gl, p_little, patrones = es.prueba_little(muestra.to_numpy())  # Prueba de Little
print(f"Prueba de Little (n = 20 000): d² = {d2:,.1f}, gl = {gl}, p = {p_little:.2e}, patrones = {patrones}")  # Resultado
falta = X["temp_mean"].isna()  # Indicador de faltante
pd.DataFrame({"diferencia estandarizada (faltante − observado)": {c: es.diferencia_estandarizada(X.loc[falta, c], X.loc[~falta, c])  # d de Cohen
                                                                 for c in X.columns[1:]}})

Prueba de Little (n = 20 000): d² = 2,304.8, gl = 5, p = 0.00e+00, patrones = 2


,diferencia estandarizada (faltante − observado)
log_casos,-0.3347
log_poblacion,-0.6801
ndvi_mean,0.2808
oni_anom,0.0080
prec_total,-0.0149


In [10]:
from sklearn.linear_model import LogisticRegression  # Modelo del mecanismo
from sklearn.model_selection import cross_val_score  # Validación cruzada
from sklearn.pipeline import make_pipeline  # Encadena
from sklearn.preprocessing import StandardScaler  # Estandariza
perfil = eda.groupby("COD_MUN_N").agg(log_poblacion=("poblacion", lambda s: np.log(s.mean())), prec=("prec_total", "mean"),  # Perfil del municipio
                                      ndvi=("ndvi_mean", "mean"), log_casos=("casos_totales", lambda s: np.log1p(s.mean())),
                                      falta=("temp_mean_missing", "mean")).join(m.set_index("COD_MUN_N")[["region", "latitud", "longitud"]])
Z = pd.get_dummies(perfil.drop(columns="falta"), columns=["region"], dtype=float)  # Variables observadas
auc_mec = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), Z, (perfil["falta"] > 0.5).astype(int),  # ¿Se predice el faltante?
                          cv=5, scoring="roc_auc").mean()
print(f"AUC al predecir qué municipio carece de temperatura con variables observadas: {auc_mec:.3f}")  # 0,5 = imposible de predecir
pd.crosstab(m.set_index("COD_MUN_N")["region"], grupo, margins=True)  # Patrón de faltantes por región (todos los años)

AUC al predecir qué municipio carece de temperatura con variables observadas: 0.666


temp_mean_missing,nunca imputada,parcialmente,siempre imputada,All
region,,,,
Amazonía,0,9,10,19
Andina,10,63,100,173
Caribe,2,34,63,99
Orinoquía,4,11,26,41
Pacífica,5,12,27,44
All,21,129,226,376


**Mecanismo.** Los faltantes **no son MCAR**: la prueba de Little los rechaza con claridad, y la diferencia más grande está en el tamaño del municipio (los municipios sin temperatura son mucho más pequeños, $d \approx -0{,}7$) y en la carga de casos ($d \approx -0{,}3$). La falta de dato se explica en parte con variables observadas (AUC ≈ 0,67 con población, región, coordenadas y clima) y no cambia con el tiempo, lo que es coherente con **MAR estructural**: los municipios pequeños no tienen estación meteorológica. No se puede descartar un componente **MNAR** (si las estaciones faltan justo donde el clima es más extremo), y eso no es verificable con estos datos.

**Consecuencia.** No se reimputa la temperatura con medias de otros municipios (226 municipios no tienen ninguna medición propia y la altitud cambia mucho entre vecinos). Se conserva la bandera `temp_missing` como variable (método del indicador de faltante), de modo que el modelo pueda descontar la señal térmica cuando es imputada; el `SimpleImputer` del Pipeline queda como protección ante nulos futuros.

### Duplicados exactos y casi duplicados

In [11]:
clima = ["temp_mean", "prec_total", "ndvi_mean"]  # Variables climáticas
dup_clima = raw.duplicated(subset=["time_idx"] + clima, keep=False)  # Misma semana y mismo clima en distintos municipios
pd.Series({
    "Filas duplicadas exactas": int(raw.duplicated().sum()),
    "Claves (municipio, semana) repetidas": int(raw.duplicated(["COD_MUN_N", "time_idx"]).sum()),
    "Filas idénticas salvo identificadores": int(raw.drop(columns=["COD_MUN_N", "time_idx", "week_start", "SEMANA"]).duplicated().sum()),
    "Filas con clima idéntico a otro municipio en la misma semana": f"{int(dup_clima.sum()):,} ({dup_clima.mean():.1%})",
}, name="conteo").to_frame()

,conteo
Filas duplicadas exactas,0
"Claves (municipio, semana) repetidas",0
Filas idénticas salvo identificadores,0
Filas con clima idéntico a otro municipio en la misma semana,0 (0.0%)


**Interpretación.** No hay duplicados exactos, claves (municipio, semana) repetidas ni filas que coincidan en todo salvo los identificadores. Tampoco hay casi duplicados climáticos: ningún par de municipios comparte exactamente temperatura, lluvia y NDVI en la misma semana. No se elimina ninguna fila por duplicación.

### Valores atípicos

In [12]:
cont = ["casos_totales", "temp_mean", "prec_total", "ndvi_mean", "oni_anom", "Flujo_in", "poblacion"]  # Variables continuas
q1, q3 = raw[cont].quantile(0.25), raw[cont].quantile(0.75)  # Cuartiles
iqr = q3 - q1  # Rango intercuartílico
fuera = (raw[cont] < q1 - 1.5 * iqr) | (raw[cont] > q3 + 1.5 * iqr)  # Regla de Tukey
pd.DataFrame({"límite inferior": q1 - 1.5 * iqr, "límite superior": q3 + 1.5 * iqr,  # Límites
              "atípicos (%)": fuera.mean() * 100, "máximo": raw[cont].max()})

,límite inferior,límite superior,atípicos (%),máximo
casos_totales,-3.0000,5.0000,11.7997,"1,426.0000"
temp_mean,19.1335,27.3999,18.1375,34.0667
prec_total,-82.2500,148.5500,4.6682,"1,053.0000"
ndvi_mean,0.3435,0.9443,1.6621,0.8772
oni_anom,-2.3550,2.3250,2.3018,2.7500
Flujo_in,-1.9642,4.0137,12.5646,"1,337.5596"
poblacion,"-34,761.5000","90,352.5000",14.0957,"7,149,540.0000"


In [13]:
top = raw.nlargest(8, "casos_totales").merge(m, on="COD_MUN_N")  # Semanas con más casos
top[["municipio", "departamento", "week_start", "casos_totales", "poblacion"]]

,municipio,departamento,week_start,casos_totales,poblacion
0,Santiago De Cali,Valle Del Cauca,2024-05-20,1426,1811385
1,Santiago De Cali,Valle Del Cauca,2024-06-03,1399,1811385
2,Santiago De Cali,Valle Del Cauca,2024-05-27,1378,1811385
3,Santiago De Cali,Valle Del Cauca,2024-06-17,1366,1811385
4,Santiago De Cali,Valle Del Cauca,2024-06-10,1364,1811385
5,Santiago De Cali,Valle Del Cauca,2024-04-01,1284,1811385
6,Santiago De Cali,Valle Del Cauca,2024-04-29,1281,1811385
7,Santiago De Cali,Valle Del Cauca,2024-04-15,1277,1811385


**Interpretación y tratamiento.** Los casos tienen una cola muy larga: casi el 12 % de las semanas supera el límite de Tukey (más de 5 casos), y las ocho semanas con más casos son todas de Cali en la epidemia de 2024, no errores de digitación. **No se eliminan**: son precisamente los eventos que se quieren anticipar. Se tratan con la transformación $\log(1 + x)$, que comprime la cola sin borrar información; lo mismo aplica a la población y al flujo de personas (12–14 % de atípicos por sus colas largas). El 18 % de «atípicos» de temperatura es un artefacto: la imputación concentra el 64 % de los valores cerca de 22,4 °C, estrecha el rango intercuartílico y convierte en atípicas temperaturas normales de tierra caliente.

### Valores imposibles o inconsistentes

In [14]:
edades = ["casos_0_4", "casos_5_14", "casos_15_64", "casos_65_plus"]  # Grupos de edad
chequeos = {  # Reglas de consistencia
    "Conteos negativos": int((raw[["casos_totales"] + edades + ["casos_m", "casos_f"]] < 0).sum().sum()),
    "Suma de edades ≠ total": int((raw[edades].sum(axis=1) != raw["casos_totales"]).sum()),
    "Hombres + mujeres ≠ total": int((raw["casos_m"] + raw["casos_f"] != raw["casos_totales"]).sum()),
    "Temperatura fuera de [0, 40] °C": int((~raw["temp_mean"].between(0, 40)).sum()),
    "Precipitación negativa": int((raw["prec_total"] < 0).sum()),
    "NDVI fuera de [−1, 1]": int((~raw["ndvi_mean"].between(-1, 1)).sum()),
    "Población ≤ 0": int((raw["poblacion"] <= 0).sum()),
    "Semanas que no avanzan 7 días": int((raw.sort_values(["COD_MUN_N", "time_idx"]).groupby("COD_MUN_N")["week_start"].diff().dt.days.dropna() != 7).sum()),
    "Población distinta dentro de un mismo año": int((raw.groupby(["COD_MUN_N", "ANO"])["poblacion"].nunique() > 1).sum()),
    "Casos mayores que la población": int((raw["casos_totales"] > raw["poblacion"]).sum()),
}
pd.Series(chequeos, name="filas con problema").to_frame()

,filas con problema
Conteos negativos,0
Suma de edades ≠ total,0
Hombres + mujeres ≠ total,0
"Temperatura fuera de [0, 40] °C",0
Precipitación negativa,0
"NDVI fuera de [−1, 1]",0
Población ≤ 0,0
Semanas que no avanzan 7 días,0
Población distinta dentro de un mismo año,0
Casos mayores que la población,0


In [15]:
mayor_lluvia = raw.nlargest(3, "prec_total").merge(m, on="COD_MUN_N")[["municipio", "departamento", "week_start", "prec_total"]]  # Lluvias extremas
mayor_lluvia

,municipio,departamento,week_start,prec_total
0,Istmina,Chocó,2024-04-15,"1,053.0000"
1,Istmina,Chocó,2022-07-04,874.0000
2,Guamal,Meta,2011-11-28,796.0000


**Interpretación.** No hay valores imposibles: los conteos son coherentes entre sí (edades y sexo suman el total), los rangos físicos son válidos, la frecuencia es estrictamente semanal y la población es anual. Las lluvias semanales extremas (más de 1 000 mm) corresponden a municipios del Pacífico, una de las regiones más lluviosas del planeta, así que son plausibles. Las categorías son códigos numéricos, sin errores de escritura.

### Sesgos de muestreo y representatividad

In [16]:
todos = pd.read_csv(d.RUTA_DIVIPOLA, dtype=str)  # Los 1 122 municipios del país
todos["region"] = todos["cod_dpto"].astype(int).map(d.REGIONES)  # Región
rep = pd.DataFrame({"municipios del país": todos["region"].value_counts(),  # Universo
                    "municipios del panel": m["region"].value_counts()})  # Muestra
rep["cobertura (%)"] = rep["municipios del panel"] / rep["municipios del país"] * 100  # Cobertura
rep["participación en el panel (%)"] = rep["municipios del panel"] / rep["municipios del panel"].sum() * 100  # Composición
rep["participación en el país (%)"] = rep["municipios del país"] / rep["municipios del país"].sum() * 100  # Composición nacional
pob_panel = raw.loc[raw["ANO"] == 2024].groupby("COD_MUN_N")["poblacion"].first().sum()  # Población cubierta en 2024
print(f"Municipios en el panel: {len(m)} de {len(todos)} ({len(m) / len(todos):.1%}) · población cubierta en 2024: {pob_panel / 1e6:.1f} millones")  # Cobertura
rep.round(1)

Municipios en el panel: 376 de 1122 (33.5%) · población cubierta en 2024: 32.2 millones


,municipios del país,municipios del panel,cobertura (%),participación en el panel (%),participación en el país (%)
region,,,,,
Andina,629,173,27.5000,46.0000,56.1000
Caribe,197,99,50.3000,26.3000,17.6000
Pacífica,179,44,24.6000,11.7000,16.0000
Orinoquía,59,41,69.5000,10.9000,5.3000
Amazonía,58,19,32.8000,5.1000,5.2000


**Interpretación.** El panel incluye un tercio de los municipios del país pero cerca del 60 % de su población (32 de unos 53 millones de habitantes), porque contiene las capitales y los municipios con transmisión habitual de dengue. La composición regional no es la nacional: el Caribe y la Orinoquía están sobrerrepresentados (son regiones de tierra caliente, endémicas) y la región Andina y la Pacífica, subrepresentadas. Hay tres sesgos a declarar: (1) **selección**, los municipios de alta montaña sin transmisión no están, así que las conclusiones aplican a municipios endémicos; (2) **subregistro**, los casos son los notificados al SIVIGILA y su calidad varía entre municipios y años (por ejemplo, durante la pandemia de 2020); y (3) **cobertura climática**, la medición de temperatura falta justamente en los municipios pequeños.

## Consideraciones éticas

* **Datos personales.** El panel solo contiene conteos agregados por municipio y semana; no hay nombres, documentos ni direcciones. No requiere consentimiento informado ni aprobación de un comité de ética.
* **Riesgo de reidentificación.** Es bajo pero no nulo: en municipios pequeños, una semana con un caso de 65 años o más y sexo conocido podría asociarse a una persona por conocimiento local. Por eso el informe solo publica agregados y nunca cruces finos de edad × sexo × semana en municipios pequeños.
* **Coordenadas y tiempo.** Las coordenadas son centroides oficiales del municipio, no ubicaciones de viviendas, y el tiempo está agregado por semana: no permiten ubicar casos individuales.
* **Uso del modelo.** Una alerta falsa puede generar gasto o alarma injustificada; un brote no anticipado cuesta salud. El modelo es una herramienta de apoyo a la vigilancia y no reemplaza el criterio epidemiológico. Además, si la calidad de la notificación es peor en municipios pobres, el modelo podría subestimar su riesgo (sesgo de equidad).

# Partición y variable objetivo

## Orden de trabajo: primero se reserva la prueba

Antes de cualquier decisión basada en los datos se separa el conjunto de prueba. Como hay tiempo, la partición es **cronológica**: entrenamiento 2015–2021, validación 2022 (umbral de decisión) y prueba 2023–2024. Los años 2010–2014 solo sirven para construir el canal endémico (necesita cinco años previos).

**Purga con intervalo de separación.** El objetivo de la fila $t$ es el brote de $t+4$. Una fila de diciembre de 2021 tiene su etiqueta en enero de 2022; si se dejara en el entrenamiento, una etiqueta de la validación se usaría para entrenar. Por eso una fila solo pertenece a un periodo si su semana $t$ **y** su semana objetivo $t+4$ caen dentro de él; las cuatro semanas de cada frontera se descartan.

In [17]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
df = d.panel()  # Panel con objetivo, variables, coordenadas y periodo
resumen = df.groupby("periodo").agg(filas=("COD_MUN_N", "size"), desde=("week_start", "min"), hasta=("week_start", "max"))  # Tamaño por periodo
resumen = resumen.loc[["historia", "entrenamiento", "validación", "prueba", "purgada"]]  # Orden
resumen["desde"], resumen["hasta"] = resumen["desde"].dt.date, resumen["hasta"].dt.date  # Fechas legibles
resumen

,filas,desde,hasta
periodo,,,
historia,96256,2010-01-04,2014-11-24
entrenamiento,136112,2014-12-29,2021-11-29
validación,18048,2022-01-03,2022-11-28
prueba,37600,2023-01-02,2024-11-25
purgada,6016,2014-12-01,2024-12-23


In [18]:
linea = df.groupby(["week_start", "periodo"]).size().reset_index(name="n")  # Filas por semana y periodo
fig = px.bar(linea, x="week_start", y="n", color="periodo", color_discrete_map=PERIODO_COLOR,  # Línea de tiempo
             category_orders={"periodo": ["historia", "entrenamiento", "validación", "prueba", "purgada"]})
fig.update_traces(marker_line_width=0)  # Sin bordes
fig.update_layout(bargap=0, title="Partición cronológica (las franjas claras son las semanas purgadas)", xaxis_title="", yaxis_title="municipios")  # Formato
mostrar(fig, 340)  # Muestra
tr_t = df[df["periodo"] == "entrenamiento"]  # Entrenamiento
va_t = df[df["periodo"] == "validación"]  # Validación
print(f"Última semana objetivo del entrenamiento: {(tr_t['week_start'].max() + pd.Timedelta(weeks=4)).date()}")  # Frontera
print(f"Primera semana t de la validación: {va_t['week_start'].min().date()}")  # Frontera

Última semana objetivo del entrenamiento: 2021-12-27
Primera semana t de la validación: 2022-01-03


**A partir de aquí, todo el EDA usa solo semanas hasta 2021** (historia + entrenamiento). La validación de 2022 solo se mira para medir la deriva (sección 2.6), y la prueba 2023–2024 no se abre hasta evaluar el modelo base.

## Definición del objetivo: el canal endémico

El umbral $u_{m,t}$ es el tercer cuartil de los casos de las mismas semanas (± 2) de los cinco años anteriores: 25 valores. Un brote exige superar el umbral **y** tener al menos 3 casos (en municipios sin historia el umbral es 0 y un solo caso sería un "brote").

In [19]:
eda = d.entrenamiento_eda(df)  # Semanas t de 2010–2021
for mun in [76001, 68001]:  # Cali y Bucaramanga
    s = eda[(eda["COD_MUN_N"] == mun) & (eda["ANO"] >= 2015)]  # Serie del municipio en el entrenamiento
    fig = go.Figure()  # Figura
    fig.add_scatter(x=s["week_start"], y=s["umbral"], name="Umbral (Q3 de 5 años)", line=dict(color=GRIS, dash="dot"))  # Umbral
    fig.add_scatter(x=s["week_start"], y=s["casos_totales"], name="Casos", line=dict(color=AZUL))  # Casos
    b = s[s["brote"] == 1]  # Semanas de brote
    fig.add_scatter(x=b["week_start"], y=b["casos_totales"], mode="markers", name="Brote", marker=dict(color=ROJO, size=5))  # Brotes
    fig.update_layout(title=f"Canal endémico de {s['municipio'].iloc[0]} (2015–2021)", yaxis_title="casos por semana")  # Formato
    mostrar(fig, 340)  # Muestra

## Frecuencia de las clases y desbalance

In [20]:
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Conjuntos ya purgados
clases = pd.DataFrame({"filas": [len(tr), len(va)], "brotes": [int(tr[cf.TARGET].sum()), int(va[cf.TARGET].sum())]},  # Conteos
                      index=["entrenamiento 2015–2021", "validación 2022"])
clases["no brotes"] = clases["filas"] - clases["brotes"]  # Clase mayoritaria
clases["prevalencia (%)"] = clases["brotes"] / clases["filas"] * 100  # Proporción de brotes
clases["razón no brote : brote"] = clases["no brotes"] / clases["brotes"]  # Grado de desbalance
print(f"Prueba 2023–2024: {len(te):,} filas reservadas (su prevalencia no se consulta en el EDA)")  # Prueba sellada
clases.round(2)

Prueba 2023–2024: 37,600 filas reservadas (su prevalencia no se consulta en el EDA)


,filas,brotes,no brotes,prevalencia (%),razón no brote : brote
entrenamiento 2015–2021,132728,13644,119084,10.2800,8.7300
validación 2022,18048,2229,15819,12.3500,7.1000


In [21]:
anual = tr.groupby("ANO")[cf.TARGET].agg(["mean", "sum", "size"]).rename(columns={"mean": "prevalencia", "sum": "brotes", "size": "filas"})  # Por año
fig = px.bar(anual.reset_index(), x="ANO", y="prevalencia", text=anual["brotes"].map("{:,}".format), color_discrete_sequence=[AZUL])  # Barras
fig.update_layout(title="Prevalencia del brote por año de entrenamiento (texto: número de brotes)", yaxis_tickformat=".0%", xaxis_title="")  # Formato
mostrar(fig, 340)  # Muestra
anual

,prevalencia,brotes,filas
ANO,,,
2015,0.1281,2119,16544
2016,0.0906,1772,19552
2017,0.0167,326,19552
2018,0.0724,1416,19552
2019,0.2532,4951,19552
2020,0.0992,1977,19928
2021,0.0600,1083,18048


**Interpretación.** La clase brote es minoritaria (≈ 10 % en el entrenamiento, una razón de casi 9 a 1), pero con más de 13 000 casos de la clase minoritaria hay información suficiente para aprender. El desbalance **cambia mucho entre años**: del 1,7 % en 2017 al 25 % en 2019. Consecuencias:

* **Métrica.** La exactitud es engañosa (no alertar nunca acierta el 90 % en el entrenamiento). Se priorizan el AUC-PR (se concentra en la clase brote y su referencia es la prevalencia), el AUC-ROC, el MCC y la calibración (Brier).
* **Validación.** Como la prevalencia depende del año, los pliegues deben ser años completos (validación hacia adelante por años); un pliegue estratificado al azar ocultaría esa variación.
* **Desbalance.** Se compara `class_weight=None` contra `balanced` dentro de la validación cruzada en vez de decidirlo de antemano; no se usa sobremuestreo sintético porque inventaría semanas que rompen la estructura temporal.

## El objetivo en el tiempo y en el espacio

In [22]:
nac = eda[eda["ANO"] >= 2015].groupby("week_start").agg(brote=("brote", "mean"), casos=("casos_totales", "sum"))  # Serie nacional
fig = make_subplots(specs=[[{"secondary_y": True}]])  # Dos ejes
fig.add_bar(x=nac.index, y=nac["casos"], name="Casos (eje derecho)", marker_color="#cbd5e1", secondary_y=True)  # Casos
fig.add_scatter(x=nac.index, y=nac["brote"], name="Municipios en brote", line=dict(color=ROJO), secondary_y=False)  # Proporción en brote
fig.update_layout(title="Proporción de municipios en brote y casos nacionales (2015–2021)", bargap=0)  # Formato
fig.update_yaxes(tickformat=".0%", secondary_y=False)  # Porcentaje
mostrar(fig, 360)  # Muestra

In [23]:
mes = tr.groupby("MES")[cf.TARGET].mean()  # Prevalencia por mes
reg = tr.groupby("region")[cf.TARGET].mean().sort_values()  # Prevalencia por región
fig = make_subplots(1, 2, subplot_titles=("Por mes de la semana t", "Por región"))  # Dos paneles
fig.add_bar(x=mes.index, y=mes.values, marker_color=AZUL, row=1, col=1, showlegend=False)  # Meses
fig.add_bar(x=reg.values, y=reg.index, orientation="h", marker_color=VERDE, row=1, col=2, showlegend=False)  # Regiones
fig.update_yaxes(tickformat=".0%", row=1, col=1)  # Porcentaje
fig.update_xaxes(tickformat=".0%", row=1, col=2)  # Porcentaje
fig.update_layout(title="Prevalencia del brote en cuatro semanas (entrenamiento)")  # Título
mostrar(fig, 340)  # Muestra

In [24]:
mapa = tr.groupby(["COD_MUN_N", "municipio", "departamento", "latitud", "longitud"])[cf.TARGET].mean().reset_index(name="prevalencia")  # Por municipio
fig = px.scatter_geo(mapa, lat="latitud", lon="longitud", color="prevalencia", hover_name="municipio",  # Mapa de puntos
                     hover_data={"departamento": True, "prevalencia": ":.1%", "latitud": False, "longitud": False},
                     color_continuous_scale="YlOrRd", range_color=(0, 0.3))
fig.update_traces(marker=dict(size=7, line=dict(width=0.3, color="#334155")))  # Tamaño de los puntos
fig.update_layout(title="Proporción de semanas con brote en cuatro semanas, por municipio (2015–2021)", coloraxis_colorbar_tickformat=".0%")  # Formato
mostrar(mapa_base(fig), 560)  # Muestra
print(mapa["prevalencia"].describe())  # Heterogeneidad entre municipios

count   376.0000
mean      0.1028
std       0.0848
min       0.0000
25%       0.0340
50%       0.0822
75%       0.1671
max       0.4108
Name: prevalencia, dtype: float64


**Interpretación.** El objetivo tiene dinámica propia en el tiempo: hay ondas epidémicas nacionales (2015–2016 y 2019) separadas por años casi sin brotes (2017–2018), así que la prevalencia de la prueba puede ser muy distinta a la del entrenamiento. La estacionalidad mensual del objetivo es débil (entre 9 % y 12 % según el mes) comparada con esas ondas: el canal endémico ya descuenta la estacionalidad habitual de cada municipio. En el espacio, la proporción de semanas en brote varía entre municipios, pero el canal endémico la normaliza: cada municipio se compara consigo mismo, así que municipios con muchos casos no tienen necesariamente más brotes. La relación del objetivo con cada predictora se estudia en el análisis bidimensional.

# Análisis unidimensional

Todas las estadísticas usan las semanas 2010–2021 (historia y entrenamiento); la prueba sigue sellada.

In [25]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
from scipy import stats  # Pruebas de normalidad
df = d.panel()  # Panel completo
eda = d.entrenamiento_eda(df)  # Semanas t de 2010–2021
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Entrenamiento (para las variables derivadas)
print(f"Filas del EDA: {len(eda):,}")  # Tamaño

Filas del EDA: 235,376


## Variables numéricas originales: estadísticos, forma y normalidad

Las pruebas de normalidad (D'Agostino–Pearson y Jarque–Bera) se aplican a una muestra de 5 000 filas: con cientos de miles de filas rechazarían cualquier desviación mínima, así que se leen junto con la asimetría y la curtosis.

In [26]:
num = ["casos_totales", "casos_0_4", "casos_5_14", "casos_15_64", "casos_65_plus", "casos_m", "casos_f",  # Conteos
       "temp_mean", "prec_total", "ndvi_mean", "oni_anom", "oni_total", "Flujo_in", "poblacion"]  # Continuas
muestra = eda[num].sample(5000, random_state=cf.SEMILLA)  # Muestra para las pruebas
filas = []  # Resultados
for v in num:  # Cada variable
    x = eda[v]  # Serie
    q1, q3 = x.quantile([0.25, 0.75])  # Cuartiles
    iqr = q3 - q1  # Rango intercuartílico
    filas.append({"variable": v, "media": x.mean(), "mediana": x.median(), "desv. est.": x.std(),  # Centro y dispersión
                  "p1": x.quantile(0.01), "p5": x.quantile(0.05), "p25": q1, "p75": q3, "p95": x.quantile(0.95), "p99": x.quantile(0.99),  # Percentiles
                  "asimetría": x.skew(), "curtosis": x.kurt(),  # Forma
                  "atípicos IQR (%)": ((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).mean() * 100,  # Regla de Tukey
                  "p normalidad (K²)": stats.normaltest(muestra[v])[1], "p Jarque-Bera": stats.jarque_bera(muestra[v])[1]})  # Normalidad
uni = pd.DataFrame(filas).set_index("variable")  # Tabla
uni.round(3)

,media,mediana,desv. est.,p1,p5,p25,p75,p95,p99,asimetría,curtosis,atípicos IQR (%),p normalidad (K²),p Jarque-Bera
variable,,,,,,,,,,,,,,
casos_totales,3.5550,0.0000,20.2740,0.0000,0.0000,0.0000,2.0000,12.0000,62.0000,18.2710,473.3340,10.0980,0.0000,0.0000
casos_0_4,0.3430,0.0000,2.0030,0.0000,0.0000,0.0000,0.0000,1.0000,7.0000,15.2840,359.0420,11.6100,0.0000,0.0000
casos_5_14,1.1510,0.0000,6.5530,0.0000,0.0000,0.0000,0.0000,4.0000,22.0000,16.3320,406.3000,22.6940,0.0000,0.0000
casos_15_64,1.8960,0.0000,12.0440,0.0000,0.0000,0.0000,1.0000,6.0000,32.0000,22.2920,686.6410,11.8360,0.0000,0.0000
casos_65_plus,0.1640,0.0000,1.0680,0.0000,0.0000,0.0000,0.0000,1.0000,3.0000,19.6710,594.3870,7.7290,0.0000,0.0000
casos_m,1.8670,0.0000,10.2470,0.0000,0.0000,0.0000,1.0000,7.0000,33.0000,17.4740,438.2050,12.1590,0.0000,0.0000
casos_f,1.6870,0.0000,10.1520,0.0000,0.0000,0.0000,1.0000,6.0000,30.0000,18.9700,509.5220,10.6310,0.0000,0.0000
temp_mean,23.1830,22.4630,3.0150,13.2500,18.2040,22.2330,24.2800,28.6500,30.2330,-0.2160,2.1400,18.1570,0.0000,0.0000
prec_total,42.5320,25.1000,51.8550,0.0000,0.0000,4.1000,62.0000,145.0000,232.0000,2.2590,8.3740,4.6380,0.0000,0.0000


In [27]:
def histograma(x, bins=60, log=False):  # Histograma precalculado (la figura no embebe todos los datos)
    x = np.log1p(x) if log else x  # Escala logarítmica opcional
    conteo, bordes = np.histogram(x, bins=bins)  # Conteos
    return (bordes[:-1] + bordes[1:]) / 2, conteo / conteo.sum()  # Centros y proporciones

paneles = [("casos_totales", True), ("poblacion", True), ("Flujo_in", True), ("temp_mean", False),  # (variable, ¿log?)
           ("prec_total", False), ("ndvi_mean", False), ("oni_anom", False), ("oni_total", False)]
fig = make_subplots(2, 4, subplot_titles=[f"log(1 + {v})" if lg else v for v, lg in paneles])  # Rejilla
for i, (v, lg) in enumerate(paneles):  # Cada variable
    xc, pr = histograma(eda[v], log=lg)  # Histograma
    fig.add_bar(x=xc, y=pr, marker_color=AZUL, showlegend=False, row=i // 4 + 1, col=i % 4 + 1)  # Barras
fig.update_layout(title="Distribuciones de las variables numéricas originales (2010–2021)", bargap=0)  # Formato
mostrar(fig, 520)  # Muestra

In [28]:
fig = go.Figure()  # Diagramas de caja precalculados
for v in ["temp_mean", "prec_total", "ndvi_mean", "oni_anom"]:  # Variables en escala comparable tras estandarizar
    x = (eda[v] - eda[v].mean()) / eda[v].std()  # Estandariza para un mismo eje
    q1, me, q3 = x.quantile([0.25, 0.5, 0.75])  # Cuartiles
    iqr = q3 - q1  # Rango intercuartílico
    fig.add_box(name=v, q1=[q1], median=[me], q3=[q3], lowerfence=[max(x.min(), q1 - 1.5 * iqr)],  # Caja de Tukey
                upperfence=[min(x.max(), q3 + 1.5 * iqr)], marker_color=AZUL, showlegend=False)
    fig.add_scatter(x=[v, v], y=[x.min(), x.max()], mode="markers", marker=dict(color=ROJO, size=7, symbol="x"), showlegend=False)  # Extremos
fig.update_layout(title="Diagramas de caja estandarizados (x: mínimo y máximo)", yaxis_title="desviaciones estándar")  # Formato
mostrar(fig, 380)  # Muestra

**Interpretación.**

* **Casos.** Distribución de conteo extremadamente asimétrica (asimetría > 15, curtosis en cientos): la mediana es 0 y la media unos pocos casos. Es la firma de un fenómeno epidémico con muchas semanas tranquilas y pocas explosivas. Necesita $\log(1+x)$; los grupos de edad y sexo tienen la misma forma y son colineales con el total (suman exactamente el total).
* **Población y flujo.** Colas largas por las grandes ciudades: también se usan en escala logarítmica.
* **Temperatura.** Bimodal por la imputación (un pico artificial en 22,4 °C y la temperatura real de tierra caliente alrededor de 26–28 °C).
* **Precipitación.** Asimétrica a la derecha, con semanas secas frecuentes y lluvias extremas en el Pacífico.
* **NDVI.** Ligeramente asimétrico a la izquierda (zonas urbanas y secas con menos vegetación).
* **ONI.** Una sola serie nacional repetida en todos los municipios: su variabilidad es solo temporal.
* **Normalidad.** Ninguna variable es normal (todas las pruebas rechazan aun con 5 000 filas). No es un problema para la regresión logística, que no supone normalidad de las predictoras, pero sí justifica las transformaciones y el uso de pruebas no paramétricas en el análisis bidimensional.

## Variables predictoras derivadas (entrenamiento)

Las 34 predictoras se describen en el capítulo de preprocesamiento; aquí se resume su distribución.

In [29]:
der = tr[cf.FEATURES].describe(percentiles=[0.05, 0.5, 0.95]).T  # Estadísticos
der["asimetría"] = tr[cf.FEATURES].skew()  # Forma
der["curtosis"] = tr[cf.FEATURES].kurt()  # Colas
der["únicos"] = tr[cf.FEATURES].nunique()  # Cardinalidad
der.round(3)

,count,mean,std,min,5%,50%,95%,max,asimetría,curtosis,únicos
lc_l0,"132,728.0000",0.5570,0.9130,0.0000,0.0000,0.0000,2.4850,6.7810,2.0780,5.0160,356
lc_l1,"132,728.0000",0.5570,0.9140,0.0000,0.0000,0.0000,2.4850,6.7810,2.0760,4.9990,355
lc_l2,"132,728.0000",0.5580,0.9140,0.0000,0.0000,0.0000,2.4850,6.7810,2.0720,4.9750,354
lc_l3,"132,728.0000",0.5590,0.9150,0.0000,0.0000,0.0000,2.4850,6.7810,2.0700,4.9560,353
lc_m4,"132,728.0000",0.5580,0.8580,0.0000,0.0000,0.1730,2.3840,6.7100,2.3500,6.6100,8006
lc_m8,"132,728.0000",0.5590,0.8430,0.0000,0.0000,0.1730,2.3510,6.6080,2.4080,6.9660,12945
lc_m12,"132,728.0000",0.5600,0.8330,0.0000,0.0000,0.2070,2.3220,6.4860,2.4290,7.1220,16220
brote_l0,"132,728.0000",0.1030,0.3040,0.0000,0.0000,0.0000,1.0000,1.0000,2.6100,4.8110,2
brote_m8,"132,728.0000",0.1040,0.2390,0.0000,0.0000,0.0000,0.7500,1.0000,2.5800,5.7530,9
ratio_umbral,"132,728.0000",-0.2810,0.8150,-5.5650,-1.6090,0.0000,1.0990,3.8290,-0.0040,1.2330,3437


**Interpretación.** Tras el logaritmo, la historia de casos (`lc_*`) sigue siendo asimétrica porque la mayoría de las semanas tienen 0 casos, pero las colas se reducen de cientos a unas pocas unidades de curtosis. La incidencia (`incid`) conserva una cola larga (municipios pequeños con un brote tienen incidencias altísimas); el estandarizado del Pipeline no la elimina, y por eso el modelo también recibe la versión logarítmica de los casos.

## Variables categóricas y de calendario

In [30]:
eda = eda.assign(fase_enso=pd.cut(eda["oni_anom"], [-9, -0.5, 0.5, 9], labels=["La Niña", "Neutral", "El Niño"]))  # Fase ENSO
cat = {"región": eda.groupby("region")["COD_MUN_N"].nunique(),  # Municipios por región
       "departamento": eda.groupby("departamento")["COD_MUN_N"].nunique().sort_values(),  # Municipios por departamento
       "fase ENSO (semanas)": eda.drop_duplicates("time_idx")["fase_enso"].value_counts(),  # Semanas por fase
       "temperatura imputada (filas)": eda["temp_mean_missing"].value_counts()}  # Bandera
card = pd.DataFrame({"cardinalidad": {"MES": eda["MES"].nunique(), "SEMANA": eda["SEMANA"].nunique(), "región": eda["region"].nunique(),  # Cardinalidad
                                      "departamento": eda["departamento"].nunique(), "fase ENSO": 3, "temp_mean_missing": 2}})
card

,cardinalidad
MES,12
SEMANA,53
departamento,32
fase ENSO,3
región,5
temp_mean_missing,2


In [31]:
dep = cat["departamento"]  # Municipios por departamento
fig = make_subplots(1, 3, column_widths=[0.5, 0.25, 0.25], subplot_titles=("Municipios por departamento", "Municipios por región", "Semanas por fase ENSO"))  # Tres paneles
fig.add_bar(x=dep.values, y=dep.index, orientation="h", marker_color=np.where(dep.values < 3, ROJO, AZUL), showlegend=False, row=1, col=1)  # Departamentos (rojo: raros)
fig.add_bar(x=cat["región"].index, y=cat["región"].values, marker_color=VERDE, showlegend=False, row=1, col=2)  # Regiones
fe = cat["fase ENSO (semanas)"]  # Fases
fig.add_bar(x=fe.index.astype(str), y=fe.values, marker_color=[AZUL if f == "La Niña" else GRIS if f == "Neutral" else ROJO for f in fe.index], showlegend=False, row=1, col=3)  # Fases
fig.update_layout(title="Frecuencias de las variables categóricas (rojo: departamentos con menos de 3 municipios)")  # Título
mostrar(fig, 620)  # Muestra
semanas = eda.drop_duplicates("time_idx")["SEMANA"].value_counts().sort_index()  # Semanas por número de semana
print(f"Semana 53: {semanas.get(53, 0)} ocurrencias (años 2015 y 2020) frente a {semanas.drop(53).min()}–{semanas.drop(53).max()} del resto")  # Categoría rara
print("Departamentos con menos de 3 municipios en el panel:", ", ".join(dep[dep < 3].index))  # Raros

Semana 53: 2 ocurrencias (años 2015 y 2020) frente a 12–12 del resto
Departamentos con menos de 3 municipios en el panel: Guainía, Quindío, Vaupés, Bogotá, D.C., Guaviare, Amazonas, Arauca


**Interpretación y tratamiento de categorías raras.**

* **Departamento** (32 categorías) tiene varias categorías raras (departamentos amazónicos y San Andrés con 1–2 municipios). Codificarlo como variable indicadora haría que el modelo memorice lugares y no generalice a zonas nuevas; **no se usa como predictora**. Se usa como bloque de la validación espacial y para agregar la región.
* **Región** (5 categorías) no tiene categorías raras; se usa en el EDA para describir la heterogeneidad espacial.
* **Semana 53** solo ocurre en 2015 y 2020. Se codifica con seno y coseno de $2\pi\,\text{semana}/52$, de modo que la semana 53 queda junto a la semana 1, sin crear una categoría aparte.
* **Fase ENSO**: de 626 semanas, 287 son neutrales, 179 de La Niña y 160 de El Niño; ninguna fase es rara. Se conserva el ONI continuo (no las fases) para no perder información.
* **Temperatura imputada**: 64 % de las filas; es la bandera del mecanismo de faltantes y se conserva.

# Análisis bidimensional

Relaciones entre pares de variables y de cada predictora con el objetivo, con tamaños de efecto y corrección por comparaciones múltiples. Se usa el conjunto de entrenamiento (2015–2021).

In [32]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
from scipy import stats  # Correlaciones y pruebas
from sklearn.feature_selection import mutual_info_classif  # Información mutua
from sklearn.metrics import roc_auc_score  # AUC univariado
df = d.panel()  # Panel completo
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Solo se usa el entrenamiento
tr = tr.assign(fase_enso=pd.cut(tr["oni_anom"], [-9, -0.5, 0.5, 9], labels=["La Niña", "Neutral", "El Niño"]))  # Fase ENSO
obs = tr[tr["temp_missing"] == 0]  # Filas con temperatura medida (para relaciones con la temperatura)
print(f"Entrenamiento: {len(tr):,} filas · con temperatura medida: {len(obs):,}")  # Tamaños

Entrenamiento: 132,728 filas · con temperatura medida: 48,070


## Numéricas contra numéricas

Con más de 100 000 filas un diagrama de dispersión se satura; se usan mapas de densidad (equivalentes a *hexbin*).

In [33]:
def densidad(x, y, nx=60, ny=60):  # Histograma bidimensional precalculado
    H, bx, by = np.histogram2d(x, y, bins=[nx, ny])  # Conteos
    return go.Heatmap(x=(bx[:-1] + bx[1:]) / 2, y=(by[:-1] + by[1:]) / 2, z=np.log1p(H.T), colorscale="Blues", showscale=False)  # log(1 + conteo)

pares = [("temp_mean_m4", "prec_total_m4", obs), ("lc_m8", "lc_l0", tr), ("temp_mean_m8", "lc_m8", obs), ("oni_l8", "lc_m8", tr)]  # Pares
fig = make_subplots(1, 4, subplot_titles=[f"{y} vs {x}" for x, y, _ in pares])  # Cuatro paneles
filas = []  # Correlaciones
for i, (x, y, base) in enumerate(pares):  # Cada par
    fig.add_trace(densidad(base[x], base[y]), row=1, col=i + 1)  # Mapa de densidad
    fig.update_xaxes(title_text=x, row=1, col=i + 1)  # Eje
    filas.append({"par": f"{x} – {y}", "Pearson": stats.pearsonr(base[x], base[y])[0], "Spearman": stats.spearmanr(base[x], base[y])[0], "n": len(base)})  # Correlaciones
fig.update_layout(title="Densidad conjunta (escala log del conteo)")  # Título
mostrar(fig, 360)  # Muestra
pd.DataFrame(filas).set_index("par").round(3)

,Pearson,Spearman,n
par,,,
temp_mean_m4 – prec_total_m4,0.0560,-0.0940,48070
lc_m8 – lc_l0,0.9110,0.7800,132728
temp_mean_m8 – lc_m8,0.0510,0.0350,48070
oni_l8 – lc_m8,0.1580,0.1880,132728


**Interpretación.** La temperatura y la lluvia medidas están poco correlacionadas entre sí. La historia de casos es muy persistente (la semana actual y la media de 8 semanas tienen correlación alta). La relación de la temperatura con los casos es débil y no lineal, y el ONI rezagado se asocia positivamente con los casos: El Niño precede a más transmisión. En el par de casos, Pearson (0,91) supera a Spearman (0,78) porque la correlación lineal está dominada por las semanas epidémicas, mientras que en las semanas con pocos casos el orden cambia mucho de una semana a otra.

## Matriz de correlación y multicolinealidad

In [34]:
corr = tr[cf.FEATURES].corr(method="spearman")  # Correlación de Spearman (robusta a asimetría)
fig = go.Figure(go.Heatmap(z=corr.values, x=corr.columns, y=corr.index, zmin=-1, zmax=1, colorscale="RdBu_r",  # Mapa de calor
                           colorbar=dict(title="ρ")))
fig.update_layout(title="Correlación de Spearman entre las 34 predictoras (entrenamiento)", yaxis_autorange="reversed")  # Formato
mostrar(fig, 760)  # Muestra
altas = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()  # Pares sin repetir
altas = altas[altas.abs() > 0.9].sort_values(ascending=False)  # Correlaciones muy altas
print(f"Pares con |ρ| > 0,9: {len(altas)}")  # Conteo
altas.head(15).round(3).to_frame("ρ")

Pares con |ρ| > 0,9: 13


,,ρ
temp_mean_m8,temp_mean_m12,0.9780
temp_mean_m4,temp_mean_m8,0.9710
lc_m8,lc_m12,0.9700
ndvi_mean_m8,ndvi_mean_m12,0.9650
lc_l0,incid,0.9580
temp_mean_m1,temp_mean_m4,0.9530
ndvi_mean_m4,ndvi_mean_m8,0.9450
prec_total_m8,prec_total_m12,0.9390
lc_m4,lc_m8,0.9380
ndvi_mean_m1,ndvi_mean_m4,0.9300


In [35]:
vif = pd.Series(es.vif(tr[cf.FEATURES].sample(50000, random_state=cf.SEMILLA)), index=cf.FEATURES, name="VIF")  # Factor de inflación
vif.sort_values(ascending=False).round(1).to_frame().T

,temp_mean_m8,temp_mean_m12,temp_mean_m4,lc_m8,temp_mean_m1,lc_m12,ndvi_mean_m8,ndvi_mean_m4,prec_total_m8,ndvi_mean_m12,prec_total_m12,ndvi_mean_m1,prec_total_m4,lc_l3,lc_l2,lc_l1,oni_l8,oni_anom,brote_m8,flujo_m4,lpob,lc_l0,brote_l0,prec_total_m1,l_umbral_obj,incid,prec_anom_m8,ndvi_anom_m8,lc_m4,temp_missing,temp_anom_m8,ratio_umbral,sem_sin,sem_cos
VIF,"1,026.9000",525.3000,357.5000,122.1000,79.5000,61.5000,55.7000,29.8000,27.4000,24.3000,15.2000,9.5000,8.8000,6.5000,6.3000,6.0000,6.0000,5.5000,5.0000,4.6000,4.2000,4.2000,3.5000,2.5000,2.3000,2.0000,1.6000,1.5000,1.5000,1.3000,1.3000,1.2000,1.2000,1.1000


**Interpretación.** Hay multicolinealidad severa (VIF ≫ 10) entre las medias móviles de una misma variable (por ejemplo, la temperatura de 4, 8 y 12 semanas) y entre los rezagos del log de casos. Para predecir no es un problema grave, pero vuelve inestables e ininterpretables los coeficientes individuales de una regresión logística sin regularizar. **Consecuencia:** el modelo base usa penalización L2 (se elige su fuerza $C$ en la validación cruzada) y los coeficientes se interpretan por grupos, con intervalos bootstrap.

## Categóricas contra numéricas

In [36]:
def caja(x):  # Estadísticos de una caja de Tukey
    q1, me, q3 = np.quantile(x, [0.25, 0.5, 0.75])  # Cuartiles
    iqr = q3 - q1  # Rango
    return dict(q1=[q1], median=[me], q3=[q3], lowerfence=[max(x.min(), q1 - 1.5 * iqr)], upperfence=[min(x.max(), q3 + 1.5 * iqr)])  # Caja

regiones = sorted(tr["region"].unique())  # Regiones
fig = make_subplots(1, 2, subplot_titles=("log(1 + casos) por región", "Temperatura medida (media 8 sem.) por región"))  # Dos paneles
for r in regiones:  # Cada región
    fig.add_box(name=r, **caja(tr.loc[tr["region"] == r, "lc_l0"]), marker_color=AZUL, showlegend=False, row=1, col=1)  # Casos
    fig.add_box(name=r, **caja(obs.loc[obs["region"] == r, "temp_mean_m8"]), marker_color=AMBAR, showlegend=False, row=1, col=2)  # Temperatura
mostrar(fig, 380)  # Muestra
pruebas = []  # Kruskal-Wallis
for v, base in [("lc_l0", tr), ("temp_mean_m8", obs), ("prec_total_m8", tr), ("ndvi_mean_m8", tr)]:  # Variables
    h, p, eps2 = es.kruskal([base.loc[base["region"] == r, v] for r in regiones])  # Prueba por región
    pruebas.append({"variable": v, "H": h, "p": p, "ε²": eps2})  # Resultado
kw = pd.DataFrame(pruebas).set_index("variable")  # Tabla
kw["p (Holm)"] = es.holm(kw["p"])  # Corrección
kw.round(4)

,H,p,ε²,p (Holm)
variable,,,,
lc_l0,835.9909,0.0000,0.0063,0.0000
temp_mean_m8,"19,859.6053",0.0000,0.4131,0.0000
prec_total_m8,"15,390.8373",0.0000,0.1160,0.0000
ndvi_mean_m8,"6,353.1595",0.0000,0.0479,0.0000


**Interpretación.** Todas las diferencias entre regiones son significativas (con este tamaño de muestra siempre lo son), pero el tamaño de efecto separa lo importante: la temperatura difiere muchísimo entre regiones (ε² ≈ 0,41, grande), la lluvia moderadamente (ε² ≈ 0,12) y el NDVI poco (ε² ≈ 0,05), mientras que la diferencia en casos es despreciable (ε² < 0,01). La región es un fuerte determinante del clima, lo que anticipa confusión: comparar clima y brotes entre municipios mezcla el efecto del clima con el de la región. Las anomalías climáticas (desviación respecto a la normal del propio municipio) evitan ese problema.

## Predictoras contra el objetivo

Para cada predictora: correlación biserial de rangos (Mann-Whitney, brote frente a no brote), AUC univariado, correlación punto-biserial e información mutua (captura relaciones no lineales). Se corrigen los 34 valores $p$ por Holm y por FDR (Benjamini-Hochberg).

In [37]:
y = tr[cf.TARGET].to_numpy()  # Objetivo
muestra = tr.sample(40000, random_state=cf.SEMILLA)  # Muestra para la información mutua
mi = mutual_info_classif(muestra[cf.FEATURES], muestra[cf.TARGET], random_state=cf.SEMILLA)  # Información mutua (nats)
filas = []  # Resultados
for j, v in enumerate(cf.FEATURES):  # Cada predictora
    x = tr[v].to_numpy()  # Valores
    rrb, p = es.mann_whitney(x[y == 1], x[y == 0])  # Brote frente a no brote
    filas.append({"variable": v, "r biserial de rangos": rrb, "p": p, "AUC univariado": roc_auc_score(y, x),  # Asociación
                  "r punto-biserial": stats.pointbiserialr(y, x)[0], "información mutua": mi[j]})  # Lineal y no lineal
asoc = pd.DataFrame(filas).set_index("variable")  # Tabla
asoc["p (Holm)"], asoc["q (FDR)"] = es.holm(asoc["p"]), es.fdr_bh(asoc["p"])  # Correcciones
asoc.sort_values("información mutua", ascending=False).round(4)

,r biserial de rangos,p,AUC univariado,r punto-biserial,información mutua,p (Holm),q (FDR)
variable,,,,,,,
brote_m8,0.7323,0.0000,0.8662,0.5922,0.1192,0.0000,0.0000
incid,0.6932,0.0000,0.8466,0.3880,0.1076,0.0000,0.0000
ratio_umbral,0.6123,0.0000,0.8061,0.3694,0.1072,0.0000,0.0000
lc_m4,0.7625,0.0000,0.8812,0.4906,0.1040,0.0000,0.0000
lc_m8,0.7539,0.0000,0.8769,0.4721,0.1023,0.0000,0.0000
lc_m12,0.7376,0.0000,0.8688,0.4508,0.0933,0.0000,0.0000
lc_l0,0.7044,0.0000,0.8522,0.4807,0.0924,0.0000,0.0000
brote_l0,0.5402,0.0000,0.7701,0.5394,0.0911,0.0000,0.0000
lc_l1,0.6828,0.0000,0.8414,0.4666,0.0880,0.0000,0.0000


In [38]:
orden = asoc.sort_values("información mutua").index  # Orden
fig = make_subplots(1, 2, shared_yaxes=True, subplot_titles=("Información mutua con el objetivo", "|r biserial de rangos|"))  # Dos paneles
fig.add_bar(x=asoc.loc[orden, "información mutua"], y=orden, orientation="h", marker_color=AZUL, showlegend=False, row=1, col=1)  # MI
fig.add_bar(x=asoc.loc[orden, "r biserial de rangos"].abs(), y=orden, orientation="h", marker_color=VERDE, showlegend=False, row=1, col=2)  # Efecto
fig.update_layout(title="Asociación de cada predictora con el brote en cuatro semanas")  # Título
mostrar(fig, 760)  # Muestra
print(f"Predictoras con q (FDR) < 0,05: {(asoc['q (FDR)'] < 0.05).sum()} de {len(asoc)} · con |r_rb| ≥ 0,1: {(asoc['r biserial de rangos'].abs() >= 0.1).sum()}")  # Resumen

Predictoras con q (FDR) < 0,05: 33 de 34 · con |r_rb| ≥ 0,1: 20


In [39]:
def tasa_por_decil(base, v):  # Tasa de brote por decil de una predictora
    dec = pd.qcut(base[v], 10, duplicates="drop")  # Deciles
    t = base.groupby(dec, observed=True)[cf.TARGET].mean()  # Tasa por decil
    return [iv.mid for iv in t.index], t.values  # Centro del decil y tasa

fig = make_subplots(1, 4, subplot_titles=("Temperatura medida (8 sem.)", "Lluvia (8 sem.)", "ONI rezagado 8 sem.", "NDVI (8 sem.)"))  # Cuatro paneles
for i, (v, base) in enumerate([("temp_mean_m8", obs), ("prec_total_m8", tr), ("oni_l8", tr), ("ndvi_mean_m8", tr)]):  # Variables
    xc, t = tasa_por_decil(base, v)  # Tasa por decil
    fig.add_scatter(x=xc, y=t, mode="lines+markers", line=dict(color=ROJO), showlegend=False, row=1, col=i + 1)  # Curva
fig.update_yaxes(tickformat=".0%")  # Porcentaje
fig.update_layout(title="Tasa de brote en cuatro semanas por decil de cada variable climática (relaciones no lineales)")  # Título
mostrar(fig, 340)  # Muestra

**Interpretación.** La historia de casos domina: las medias móviles del log de casos, la persistencia del brote y la incidencia tienen la mayor información mutua y AUC univariados de 0,77–0,88. El flujo de personas (|r_rb| ≈ 0,49), el ONI y su rezago (≈ 0,30–0,32) y la población aportan una asociación moderada. Las variables climáticas locales tienen efectos pequeños (|r_rb| < 0,15) aunque casi todas resultan significativas tras corregir por Holm y FDR: el tamaño de efecto, no el valor $p$, dice que importan poco por sí solas. Las curvas por decil muestran por qué: la relación con la temperatura es **no lineal** (el riesgo sube hacia los 25–28 °C y no sigue subiendo), algo que una regresión logística lineal no puede representar sin transformaciones.

## Categóricas contra categóricas

In [40]:
tablas = []  # Asociación con el objetivo
for v in ["MES", "region", "departamento", "fase_enso", "temp_missing"]:  # Categóricas
    V, chi2, gl, p = es.v_cramer(tr[v], tr[cf.TARGET])  # V de Cramér y χ²
    tablas.append({"variable": v, "categorías": tr[v].nunique(), "χ²": chi2, "gl": gl, "p": p, "V de Cramér": V})  # Resultado
cc = pd.DataFrame(tablas).set_index("variable")  # Tabla
cc["p (Holm)"] = es.holm(cc["p"])  # Corrección
cc.round(4)

,categorías,χ²,gl,p,V de Cramér,p (Holm)
variable,,,,,,
MES,12,85.3226,11,0.0000,0.0237,0.0000
region,5,152.5075,4,0.0000,0.0334,0.0000
departamento,32,876.4571,31,0.0000,0.0798,0.0000
fase_enso,3,"3,700.7494",2,0.0000,0.1669,0.0000
temp_missing,2,359.0001,1,0.0000,0.0519,0.0000


In [41]:
pd.crosstab(tr["fase_enso"], tr[cf.TARGET], normalize="index").rename(columns={0: "no brote", 1: "brote"}).round(3)  # Tabla de contingencia (proporción por fila)

y,no brote,brote
fase_enso,,
La Niña,0.9550,0.0450
Neutral,0.9170,0.0830
El Niño,0.8320,0.1680


**Interpretación.** Todas las asociaciones son significativas, pero débiles salvo la fase ENSO: durante El Niño la proporción de brotes en cuatro semanas (16,8 %) es casi cuatro veces la de La Niña (4,5 %), con la V de Cramér más alta (0,17). El mes y la región explican poco del brote, porque el canal endémico ya compara cada municipio con su propia estacionalidad. El departamento tiene una asociación mayor que la región, pero usarlo como predictora equivaldría a memorizar lugares.

## Selección preliminar de variables

* **Se conservan:** la historia de casos (señal dominante), el ONI y su rezago, el flujo de personas, la población y las variables climáticas con sus anomalías (efecto pequeño pero plausible y no lineal), la estacionalidad (seno y coseno) y la bandera de temperatura imputada.
* **No se usan:** los conteos por edad y sexo (suman exactamente el total: redundantes), `oni_total` (casi lineal con `oni_anom` dentro de cada mes), el departamento y los identificadores.
* **A vigilar:** la multicolinealidad entre medias móviles (se maneja con regularización L2).

# Análisis multivariado

Reducción de dimensionalidad (PCA y t-SNE como apoyo visual), atípicos multivariados (Mahalanobis e Isolation Forest) y estructura de subpoblaciones (agrupamiento de municipios). Todo se ajusta con el entrenamiento.

In [42]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
from scipy import stats  # Distribución χ²
from sklearn.cluster import KMeans  # Agrupamiento
from sklearn.decomposition import PCA  # Componentes principales
from sklearn.ensemble import IsolationForest  # Atípicos por aislamiento
from sklearn.manifold import TSNE  # Proyección no lineal (solo visual)
from sklearn.metrics import silhouette_score  # Calidad del agrupamiento
from sklearn.preprocessing import StandardScaler  # Estandarización
df = d.panel()  # Panel completo
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Solo entrenamiento
esc = StandardScaler().fit(tr[cf.FEATURES])  # Estandarización ajustada con el entrenamiento
Z = esc.transform(tr[cf.FEATURES])  # Matriz estandarizada (132 728 × 34)

## Componentes principales (PCA)

In [43]:
pca = PCA(random_state=cf.SEMILLA).fit(Z)  # PCA completo
var = pd.DataFrame({"varianza explicada": pca.explained_variance_ratio_, "acumulada": np.cumsum(pca.explained_variance_ratio_)},  # Varianza
                   index=[f"CP{i + 1}" for i in range(len(cf.FEATURES))])
fig = make_subplots(specs=[[{"secondary_y": True}]])  # Dos ejes
fig.add_bar(x=var.index, y=var["varianza explicada"], name="Por componente", marker_color=AZUL)  # Varianza
fig.add_scatter(x=var.index, y=var["acumulada"], name="Acumulada", line=dict(color=ROJO), secondary_y=True)  # Acumulada
fig.update_yaxes(tickformat=".0%")  # Porcentaje
fig.update_layout(title="Gráfico de sedimentación (scree plot)")  # Título
mostrar(fig, 360)  # Muestra
k80, k90 = int(np.searchsorted(var["acumulada"], 0.8) + 1), int(np.searchsorted(var["acumulada"], 0.9) + 1)  # Componentes necesarios
kaiser = int((pca.explained_variance_ > 1).sum())  # Criterio de Kaiser (autovalor > 1)
print(f"Componentes para 80 % de la varianza: {k80} · para 90 %: {k90} · con autovalor > 1: {kaiser} (de 34 variables)")  # Dimensionalidad efectiva
var.head(8)

Componentes para 80 % de la varianza: 8 · para 90 %: 12 · con autovalor > 1: 7 (de 34 variables)


,varianza explicada,acumulada
CP1,0.2699,0.2699
CP2,0.1321,0.4020
CP3,0.1142,0.5162
CP4,0.1073,0.6235
CP5,0.0769,0.7004
CP6,0.0514,0.7517
CP7,0.0307,0.7824
CP8,0.0291,0.8114


In [44]:
cargas = pd.DataFrame(pca.components_[:4].T, index=cf.FEATURES, columns=["CP1", "CP2", "CP3", "CP4"])  # Cargas
top = pd.concat({c: cargas[c].abs().sort_values(ascending=False).head(6).index.to_series().reset_index(drop=True) for c in cargas}, axis=1)  # Variables dominantes
top.index = [f"#{i + 1}" for i in range(6)]  # Orden
top

,CP1,CP2,CP3,CP4
#1,lc_m4,temp_mean_m12,temp_mean_m4,prec_total_m4
#2,lc_m8,temp_mean_m8,temp_mean_m1,prec_total_m8
#3,lc_m12,temp_mean_m4,temp_mean_m8,prec_total_m12
#4,lc_l0,temp_mean_m1,temp_mean_m12,prec_total_m1
#5,lc_l2,ndvi_mean_m4,ndvi_mean_m4,ndvi_mean_m8
#6,lc_l1,ndvi_mean_m8,ndvi_mean_m8,ndvi_mean_m12


In [45]:
rng = np.random.default_rng(cf.SEMILLA)  # Generador
idx = rng.choice(len(Z), 6000, replace=False)  # Muestra para la figura
P = pca.transform(Z[idx])[:, :2]  # Dos primeras componentes
yb = tr[cf.TARGET].to_numpy()[idx]  # Objetivo
fig = go.Figure()  # Dispersión
for val, col, nom in [(0, "#94a3b8", "No brote"), (1, ROJO, "Brote en 4 semanas")]:  # Por clase
    s = yb == val  # Filas de la clase
    fig.add_scatter(x=P[s, 0], y=P[s, 1], mode="markers", name=nom, marker=dict(color=col, size=4, opacity=0.5))  # Puntos
fig.update_layout(title="Proyección en CP1–CP2 (6 000 filas al azar)", xaxis_title="CP1", yaxis_title="CP2")  # Formato
mostrar(fig, 460)  # Muestra

**Interpretación.** La dimensionalidad efectiva es mucho menor que 34: 8 componentes explican el 80 % de la varianza, 12 el 90 %, y solo 7 tienen autovalor mayor que 1, porque las predictoras forman **bloques redundantes** (rezagos y medias móviles de una misma variable). La CP1 (27 %) es el eje de la "historia de casos"; la CP2 y la CP3 son ejes de temperatura (nivel y contraste entre ventanas cortas y largas) junto con el NDVI; la CP4 es el eje de lluvia y vegetación. Los brotes se concentran hacia un extremo del eje de historia de casos, pero se superponen con los no brotes: ninguna proyección lineal de dos dimensiones separa las clases.

## t-SNE (solo como herramienta visual)

t-SNE preserva vecindades locales pero no distancias globales ni densidades: sirve para mirar, no para inferir.

In [46]:
idx2 = rng.choice(len(Z), 3000, replace=False)  # Muestra (t-SNE es costoso)
emb = TSNE(n_components=2, perplexity=40, init="pca", random_state=cf.SEMILLA).fit_transform(Z[idx2])  # Proyección
yb2 = tr[cf.TARGET].to_numpy()[idx2]  # Objetivo
reg = tr["region"].to_numpy()[idx2]  # Región
fig = make_subplots(1, 2, subplot_titles=("Coloreado por objetivo", "Coloreado por región"))  # Dos paneles
for val, col, nom in [(0, "#94a3b8", "No brote"), (1, ROJO, "Brote")]:  # Objetivo
    s = yb2 == val  # Filas
    fig.add_scatter(x=emb[s, 0], y=emb[s, 1], mode="markers", name=nom, marker=dict(color=col, size=4, opacity=0.6), row=1, col=1)  # Puntos
for r, col in zip(sorted(set(reg)), [VERDE, AZUL, AMBAR, MORADO, "#0891b2"]):  # Regiones
    s = reg == r  # Filas
    fig.add_scatter(x=emb[s, 0], y=emb[s, 1], mode="markers", name=r, marker=dict(color=col, size=4, opacity=0.6), row=1, col=2)  # Puntos
fig.update_layout(title="t-SNE de las 34 predictoras estandarizadas (3 000 filas)")  # Título
mostrar(fig, 460)  # Muestra
from sklearn.neighbors import NearestNeighbors  # Vecinos en la proyección
vec = NearestNeighbors(n_neighbors=11).fit(emb).kneighbors(emb, return_distance=False)[:, 1:]  # 10 vecinos de cada punto
for nombre, etiqueta in [("región", reg), ("objetivo", yb2)]:  # Pureza de vecindario frente al azar
    pureza = (etiqueta[vec] == etiqueta[:, None]).mean()  # Proporción de vecinos con la misma etiqueta
    azar = sum((etiqueta == e).mean() ** 2 for e in np.unique(etiqueta))  # Esperado si la etiqueta no tuviera estructura
    print(f"Vecinos t-SNE con igual {nombre}: {pureza:.1%} (esperado al azar {azar:.1%})")  # Resultado

Vecinos t-SNE con igual región: 45.1% (esperado al azar 30.9%)
Vecinos t-SNE con igual objetivo: 87.5% (esperado al azar 81.2%)


**Interpretación.** t-SNE muestra grupos locales que coinciden más con la región (es decir, con el clima del lugar) que con el objetivo: el 45 % de los 10 vecinos de cada punto son de la misma región (31 % esperado al azar), mientras que la coincidencia en el objetivo apenas supera el azar (88 % frente a 81 %); los brotes aparecen como zonas densas dentro de varios grupos y no como un grupo aparte. Confirma que el brote no es una "subpoblación" de municipios sino un estado transitorio que depende de la historia reciente de casos.

## Atípicos multivariados

In [47]:
sub = Z[rng.choice(len(Z), 30000, replace=False)]  # Muestra para estimar la covarianza
media, prec = sub.mean(axis=0), np.linalg.pinv(np.cov(sub, rowvar=False))  # Media y precisión (pseudo-inversa por la colinealidad)
dif = Z - media  # Desviaciones
d2 = np.einsum("ij,jk,ik->i", dif, prec, dif)  # Distancia de Mahalanobis al cuadrado de cada fila
corte = stats.chi2.ppf(0.999, df=len(cf.FEATURES))  # Percentil 99,9 de χ²(34)
atip_m = d2 > corte  # Atípicos de Mahalanobis
iso = IsolationForest(n_estimators=200, contamination="auto", random_state=cf.SEMILLA, n_jobs=-1).fit(Z)  # Isolation Forest
atip_i = iso.predict(Z) == -1  # Atípicos por aislamiento
y = tr[cf.TARGET].to_numpy()  # Objetivo
pd.DataFrame({"atípicos (%)": [atip_m.mean() * 100, atip_i.mean() * 100, (atip_m & atip_i).mean() * 100],  # Proporción
              "tasa de brote en atípicos": [y[atip_m].mean(), y[atip_i].mean(), y[atip_m & atip_i].mean()],  # Brote entre atípicos
              "tasa de brote en el resto": [y[~atip_m].mean(), y[~atip_i].mean(), y[~(atip_m & atip_i)].mean()]},  # Brote en el resto
             index=["Mahalanobis (χ² 99,9 %)", "Isolation Forest", "Ambos métodos"])

,atípicos (%),tasa de brote en atípicos,tasa de brote en el resto
"Mahalanobis (χ² 99,9 %)",6.3393,0.3176,0.0883
Isolation Forest,14.7135,0.3886,0.0535
Ambos métodos,4.7428,0.3909,0.0885


In [48]:
cuales = tr.loc[atip_m & atip_i, ["municipio", "region", "ANO"]]  # Filas atípicas por ambos métodos
print("Regiones de los atípicos (ambos métodos):")  # Título
print(cuales["region"].value_counts(normalize=True).round(3).to_string())  # Composición
print("\nMunicipios más frecuentes:")  # Título
print(cuales["municipio"].value_counts().head(8).to_string())  # Municipios

Regiones de los atípicos (ambos métodos):
region
Andina      0.4160
Pacífica    0.1910
Orinoquía   0.1640
Caribe      0.1320
Amazonía    0.0970

Municipios más frecuentes:
municipio
Bogotá, D.C.        297
Santiago De Cali    291
Guamal              222
Florencia           202
Medellín            191
Buenaventura        178
Valledupar          145
Villavicencio       139


**Interpretación.** Mahalanobis marca el 6 % de las filas e Isolation Forest el 15 %; las marcadas por ambos (≈ 5 %) tienen una tasa de brote de ≈ 39 %, más de cuatro veces la del resto. Son semanas epidémicas, grandes ciudades (Bogotá, Cali, Medellín, con poblaciones y flujos extremos) o municipios de clima extremo (Guamal, Buenaventura), no errores. Eliminarlos quitaría justamente los eventos de interés, así que **se conservan**. El dato confirma la decisión del análisis univariado (transformar, no eliminar).

## Subpoblaciones: agrupamiento exploratorio de municipios

Se resume cada municipio con su perfil de 2015–2021 (incidencia, proporción de brotes, clima, vegetación, población y flujo) y se agrupan con k-medias.

In [49]:
perfil = tr.groupby("COD_MUN_N").agg(log_incidencia=("incid", lambda s: np.log1p(s.mean())), prevalencia=(cf.TARGET, "mean"),  # Epidemiología
                                     temp=("temp_mean_m4", "mean"), lluvia=("prec_total_m4", "mean"), ndvi=("ndvi_mean_m4", "mean"),  # Clima
                                     lpob=("lpob", "mean"), flujo=("flujo_m4", "mean"))  # Tamaño y movilidad
Zp = StandardScaler().fit_transform(perfil)  # Estandariza
sil = {k: silhouette_score(Zp, KMeans(k, n_init=20, random_state=cf.SEMILLA).fit_predict(Zp)) for k in range(2, 9)}  # Silueta por k
k = max(sil, key=sil.get)  # Mejor k
perfil["grupo"] = KMeans(k, n_init=20, random_state=cf.SEMILLA).fit_predict(Zp)  # Agrupamiento final
print("Silueta por número de grupos:", {kk: round(v, 3) for kk, v in sil.items()}, "→ k =", k)  # Elección
perfil.groupby("grupo").agg(municipios=("prevalencia", "size"), **{c: (c, "mean") for c in perfil.columns[:-1]}).round(3)

Silueta por número de grupos: {2: 0.327, 3: 0.221, 4: 0.225, 5: 0.226, 6: 0.197, 7: 0.196, 8: 0.188} → k = 2


,municipios,log_incidencia,prevalencia,temp,lluvia,ndvi,lpob,flujo
grupo,,,,,,,,
0,304,1.5250,0.0820,23.3200,43.1400,0.6430,9.7560,0.4190
1,72,1.4570,0.1920,22.9540,35.8230,0.5880,12.0190,1.7440


In [50]:
mm = perfil.join(d.municipios().set_index("COD_MUN_N")).reset_index()  # Une coordenadas
mm["grupo"] = "Grupo " + mm["grupo"].astype(str)  # Etiqueta
fig = px.scatter_geo(mm, lat="latitud", lon="longitud", color="grupo", hover_name="municipio",  # Mapa
                     hover_data={"prevalencia": ":.1%", "temp": ":.1f", "latitud": False, "longitud": False})
fig.update_traces(marker=dict(size=7))  # Tamaño
fig.update_layout(title=f"Agrupamiento de municipios por perfil (k = {k})")  # Título
mostrar(mapa_base(fig), 560)  # Muestra

**Interpretación.** La mejor silueta (0,33, con k = 2) indica una estructura de grupos débil a moderada. Los dos perfiles son: 72 municipios **grandes y conectados** (población y flujo de personas muy superiores) con una proporción de semanas en brote de ≈ 19 %, y 304 municipios **pequeños** con ≈ 8 %. El clima medio casi no separa los grupos: la diferencia está en el tamaño y la movilidad, coherente con que el virus circula más donde llegan más personas. Esto respalda dos decisiones: incluir población, flujo y clima como predictoras, y validar también por **bloques espaciales** para comprobar que el modelo funciona en grupos de municipios que no vio.

# Auditoría de fuga de datos

Una variable tiene fuga si contiene información que no estaría disponible en la semana $t$, cuando se emite la alerta. Se revisa la disponibilidad de cada familia de variables, el desempeño univariado de cada una, los identificadores y el solapamiento entre particiones, y se demuestra cuánto se inflaría el desempeño con fuga.

In [51]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
from sklearn.metrics import roc_auc_score  # AUC univariado
from sklearn.model_selection import KFold, cross_val_score  # Validación aleatoria (para la demostración)
df = d.panel()  # Panel completo
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Particiones purgadas
print(f"Entrenamiento: {len(tr):,} filas")  # Tamaño

Entrenamiento: 132,728 filas


## Disponibilidad de cada variable en la semana de la predicción

In [52]:
familias = pd.DataFrame([  # Familia, variables y justificación de disponibilidad
    ("Historia de casos", "lc_l0–lc_l3, lc_m4–lc_m12, incid", "Sí", "Casos notificados en las semanas t, t−1, …; se conocen al cierre de la semana t."),
    ("Brote actual", "brote_l0, brote_m8", "Sí", "Se calcula con los casos hasta t y el umbral de t, que usa años anteriores."),
    ("Umbral objetivo", "l_umbral_obj, ratio_umbral", "Sí", "u(t+4) es el Q3 de semanas t+4−52k±2 (k ≥ 1): la más reciente es t−46."),
    ("Clima", "temp/prec/ndvi_mean_m1–m12", "Sí", "Medias móviles de semanas ≤ t (rolling hacia atrás)."),
    ("Anomalías", "temp/prec/ndvi_anom_m8", "Sí", "La normal climática se estima solo con 2010–2021 (entrenamiento)."),
    ("El Niño", "oni_anom, oni_l8", "Sí*", "El ONI mensual se publica con unos días de rezago; oni_l8 (8 semanas antes) no tiene ningún riesgo."),
    ("Calendario", "sem_sin, sem_cos", "Sí", "La semana del año se conoce de antemano."),
    ("Contexto", "lpob, flujo_m4, temp_missing", "Sí", "Población anual proyectada; flujo de semanas ≤ t; bandera de imputación."),
    ("Objetivo y futuro", "y, casos_obj (casos de t+4)", "No", "Son el objetivo o información de t+4: nunca se usan como predictoras."),
    ("Identificadores", "COD_MUN_N, ANO, time_idx, departamento", "Sí, pero se excluyen", "Permitirían memorizar lugares o épocas y no generalizan."),
], columns=["familia", "variables", "¿disponible en t?", "justificación"])
familias

,familia,variables,¿disponible en t?,justificación
0,Historia de casos,"lc_l0–lc_l3, lc_m4–lc_m12, incid",Sí,"Casos notificados en las semanas t, t−1, …; se..."
1,Brote actual,"brote_l0, brote_m8",Sí,Se calcula con los casos hasta t y el umbral d...
2,Umbral objetivo,"l_umbral_obj, ratio_umbral",Sí,u(t+4) es el Q3 de semanas t+4−52k±2 (k ≥ 1): ...
3,Clima,temp/prec/ndvi_mean_m1–m12,Sí,Medias móviles de semanas ≤ t (rolling hacia a...
4,Anomalías,temp/prec/ndvi_anom_m8,Sí,La normal climática se estima solo con 2010–20...
5,El Niño,"oni_anom, oni_l8",Sí*,El ONI mensual se publica con unos días de rez...
6,Calendario,"sem_sin, sem_cos",Sí,La semana del año se conoce de antemano.
7,Contexto,"lpob, flujo_m4, temp_missing",Sí,Población anual proyectada; flujo de semanas ≤...
8,Objetivo y futuro,"y, casos_obj (casos de t+4)",No,Son el objetivo o información de t+4: nunca se...
9,Identificadores,"COD_MUN_N, ANO, time_idx, departamento","Sí, pero se excluyen",Permitirían memorizar lugares o épocas y no ge...


In [53]:
# Verificación numérica del umbral objetivo: semanas históricas que usa u(t+4)
desfases = [cf.HORIZONTE - 52 * k + j for k in range(1, cf.ANIOS_HISTORIA + 1) for j in range(-cf.VENTANA_SEMANAS, cf.VENTANA_SEMANAS + 1)]  # Desfases respecto a t
print(f"El umbral objetivo usa las semanas t{min(desfases):+d} a t{max(desfases):+d}: todas anteriores a t → sin fuga")  # Rango

El umbral objetivo usa las semanas t-258 a t-46: todas anteriores a t → sin fuga


:::{note}
El ONI de un mes se publica a inicios del mes siguiente, así que en las primeras semanas de cada mes el valor contemporáneo `oni_anom` podría no estar publicado todavía. Por eso se conserva también `oni_l8`, que no tiene ningún riesgo, y la variable contemporánea queda **en observación**: en la auditoría de desempeño no se comporta como un proxy del objetivo (AUC univariado ≈ 0,65).
:::

## Desempeño univariado: señal de alerta si se acerca a 1

In [54]:
filas = []  # AUC univariado de cada candidata
candidatas = {**{v: tr[v] for v in cf.FEATURES},  # Las 34 predictoras
              "casos_obj (casos de t+4)": np.log1p(tr["casos_obj"]),  # Información del futuro (para contraste)
              "COD_MUN_N (identificador)": tr["COD_MUN_N"], "time_idx (identificador)": tr["time_idx"], "ANO (identificador)": tr["ANO"]}  # Identificadores
for nombre, x in candidatas.items():  # Cada candidata
    a = roc_auc_score(tr[cf.TARGET], x)  # AUC univariado
    filas.append({"variable": nombre, "AUC univariado": max(a, 1 - a)})  # Simétrico (la dirección no importa)
auc_uni = pd.DataFrame(filas).set_index("variable").sort_values("AUC univariado", ascending=False)  # Tabla
auc_uni["alerta (> 0,9)"] = np.where(auc_uni["AUC univariado"] > 0.9, "⚠ fuga probable", "")  # Bandera
auc_uni.round(3).head(15)

,AUC univariado,"alerta (> 0,9)"
variable,,
casos_obj (casos de t+4),0.9630,⚠ fuga probable
lc_m4,0.8810,
lc_m8,0.8770,
lc_m12,0.8690,
brote_m8,0.8660,
lc_l0,0.8520,
incid,0.8470,
lc_l1,0.8410,
lc_l2,0.8300,


In [55]:
orden = auc_uni.sort_values("AUC univariado").index  # Orden
colores = [ROJO if "obj" in v else GRIS if "identificador" in v else AZUL for v in orden]  # Rojo: futuro; gris: identificadores
fig = go.Figure(go.Bar(x=auc_uni.loc[orden, "AUC univariado"], y=orden, orientation="h", marker_color=colores))  # Barras
fig.add_vline(x=0.9, line_dash="dash", line_color=ROJO, annotation_text="alerta 0,9")  # Umbral de alerta
fig.update_layout(title="AUC univariado de cada variable (rojo: información del futuro; gris: identificadores)", xaxis_range=[0.45, 1])  # Formato
mostrar(fig, 820)  # Muestra

**Interpretación.** Ninguna de las 34 predictoras supera 0,9: la más fuerte es la media de 4 semanas del log de casos (≈ 0,88), coherente con la persistencia de las epidemias y no con una fuga. Los casos de la semana objetivo, en cambio, alcanzan ≈ 0,96: es exactamente el patrón que delata una fuga, y por eso esa columna se descarta. Los identificadores tienen AUC moderados (los brotes se concentran en ciertos años y municipios); no se usan porque el modelo aprendería *cuándo* y *dónde* hubo brotes en el entrenamiento en lugar de *por qué*.

## Duplicados y entidades repetidas entre particiones

In [56]:
clave = lambda x: set(zip(x["COD_MUN_N"], x["time_idx"]))  # Clave única (municipio, semana)
ultima_obj_tr = (tr["time_idx"] + cf.HORIZONTE).max()  # Última semana objetivo del entrenamiento
pd.Series({
    "Filas compartidas entrenamiento ∩ validación": len(clave(tr) & clave(va)),
    "Filas compartidas entrenamiento ∩ prueba": len(clave(tr) & clave(te)),
    "Filas compartidas validación ∩ prueba": len(clave(va) & clave(te)),
    "Última semana objetivo del entrenamiento (time_idx)": int(ultima_obj_tr),
    "Primera semana t de la validación (time_idx)": int(va["time_idx"].min()),
    "Última semana objetivo de la validación (time_idx)": int((va["time_idx"] + cf.HORIZONTE).max()),
    "Primera semana t de la prueba (time_idx)": int(te["time_idx"].min()),
    "Filas purgadas en las fronteras": int((df["periodo"] == "purgada").sum()),
    "Municipios presentes en las tres particiones": len(set(tr["COD_MUN_N"]) & set(va["COD_MUN_N"]) & set(te["COD_MUN_N"])),
}, name="valor").to_frame()

,valor
Filas compartidas entrenamiento ∩ validación,0
Filas compartidas entrenamiento ∩ prueba,0
Filas compartidas validación ∩ prueba,0
Última semana objetivo del entrenamiento (time_idx),625
Primera semana t de la validación (time_idx),626
Última semana objetivo de la validación (time_idx),677
Primera semana t de la prueba (time_idx),678
Filas purgadas en las fronteras,6016
Municipios presentes en las tres particiones,376


**Interpretación.** No hay filas compartidas y ninguna etiqueta de un periodo se usa en el anterior: la última semana objetivo del entrenamiento es anterior a la primera semana de la validación, y lo mismo entre validación y prueba (la purga quita 4 semanas en cada frontera). Los 376 municipios aparecen en las tres particiones **por diseño**: el objetivo principal es predecir el futuro de municipios conocidos. Como las semanas de un municipio están correlacionadas, esto exige (1) que la partición sea temporal y (2) evaluar aparte la generalización a **zonas nuevas** con bloques espaciales (capítulo del modelo base).

## Demostración: cuánto infla la fuga el desempeño

Se entrena la misma regresión logística del modelo base en dos escenarios incorrectos y en el correcto.

In [57]:
pipe = mb.pipeline_base().set_params(clf__C=0.01)  # Mismo Pipeline del modelo base
temporal = cross_val_score(pipe, tr[cf.FEATURES], tr[cf.TARGET], groups=mb.grupos(tr), cv=mb.ValidacionAnualPurgada(), scoring="roc_auc").mean()  # Correcto
aleatorio = cross_val_score(pipe, tr[cf.FEATURES], tr[cf.TARGET], cv=KFold(5, shuffle=True, random_state=cf.SEMILLA), scoring="roc_auc").mean()  # Semanas mezcladas
con_futuro = tr.assign(lc_obj=np.log1p(tr["casos_obj"]))  # Agrega los casos de t+4
futuro = cross_val_score(pipe, con_futuro[cf.FEATURES + ["lc_obj"]], con_futuro[cf.TARGET], groups=mb.grupos(tr),  # Con información del futuro
                         cv=mb.ValidacionAnualPurgada(), scoring="roc_auc").mean()
pd.DataFrame({"AUC-ROC de validación cruzada": [temporal, aleatorio, futuro]},  # Comparación
             index=["Correcto: años hacia adelante con purga", "Fuga temporal: semanas mezcladas al azar",
                    "Fuga directa: casos de la semana objetivo"]).round(4)

,AUC-ROC de validación cruzada
Correcto: años hacia adelante con purga,0.8985
Fuga temporal: semanas mezcladas al azar,0.9064
Fuga directa: casos de la semana objetivo,0.9941


**Interpretación.** Con la validación correcta el AUC ronda 0,90. Mezclar semanas al azar cambia poco en este modelo lineal (la regresión logística no puede memorizar interacciones finas entre semanas vecinas), pero la fuga directa lo lleva casi a 1: un modelo «perfecto» e inútil. La auditoría no solo revisa columnas; también comprueba que el esquema de validación respete el tiempo, porque con modelos más flexibles (los de los siguientes entregables) la mezcla aleatoria sí infla el desempeño.

## Resultado de la auditoría

| Decisión | Variables | Motivo |
|---|---|---|
| **Descartadas** | `casos_obj` y cualquier dato de $t+1$ a $t+4$ | Información del futuro (AUC ≈ 0,96) |
| **Descartadas** | `COD_MUN_N`, `ANO`, `time_idx`, departamento | Identificadores: memorizan lugar y época |
| **Descartadas** | Casos por edad y sexo | Redundantes (suman el total), no son fuga |
| **En observación** | `oni_anom` | Posible rezago de publicación; se conserva `oni_l8` como respaldo |
| **En observación** | `brote_m8`, `lc_m4`, `ratio_umbral` | Señal fuerte (AUC 0,81–0,88) pero legítima: semanas ≤ t |
| **En observación** | `temp_missing` | Proxy del tamaño del municipio, no del objetivo |

# Componente temporal

Validación de la variable de tiempo, estacionalidad, descomposición, estacionariedad, dependencia temporal, cambios de régimen, deriva y heterogeneidad del panel. Se usan las semanas 2010–2021; la validación 2022 solo entra en la medición de deriva.

In [58]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
from statsmodels.tsa.seasonal import STL  # Descomposición estacional
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf  # Autocorrelación y estacionariedad
from scipy import stats  # Kolmogórov-Smirnov
from sklearn.metrics import roc_auc_score  # AUC por año
warnings.simplefilter("ignore")  # statsmodels reactiva sus avisos de interpolación al importarse
df = d.panel()  # Panel completo
eda = d.entrenamiento_eda(df)  # Semanas t de 2010–2021
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Particiones purgadas

## Validación de la variable temporal

In [59]:
fechas = eda["week_start"]  # Fechas de inicio de semana
pasos = eda.sort_values(["COD_MUN_N", "time_idx"]).groupby("COD_MUN_N")["week_start"].diff().dt.days.dropna()  # Paso entre semanas consecutivas
esperadas = (fechas.max() - fechas.min()).days // 7 + 1  # Semanas esperadas en el periodo
anio_iso = fechas.dt.isocalendar().year  # Año ISO de la fecha
pd.Series({
    "Tipo de dato": str(fechas.dtype),
    "Zona horaria": "sin zona (fecha civil de inicio de semana; no hay horas)",
    "Día de la semana de week_start": fechas.dt.day_name().unique()[0],
    "Frecuencia": f"semanal ({pasos.value_counts().index[0]:.0f} días en el {pasos.value_counts(normalize=True).iloc[0]:.0%} de los pasos)",
    "Marcas de tiempo duplicadas (municipio, semana)": int(eda.duplicated(["COD_MUN_N", "week_start"]).sum()),
    "Semanas esperadas / presentes por municipio": f"{esperadas} / {eda.groupby('COD_MUN_N')['week_start'].nunique().min()}",
    "Huecos (semanas faltantes)": int(esperadas - eda.groupby("COD_MUN_N")["week_start"].nunique().min()),
    "Cobertura": f"{fechas.min().date()} a {fechas.max().date()}",
    "Años con semana 53": ", ".join(map(str, sorted(eda.loc[eda["SEMANA"] == 53, "ANO"].unique()))),
    "Filas donde ANO ≠ año ISO de la fecha": int((eda["ANO"] != anio_iso).sum()),
}, name="valor").to_frame()

,valor
Tipo de dato,datetime64[ns]
Zona horaria,sin zona (fecha civil de inicio de semana; no ...
Día de la semana de week_start,Monday
Frecuencia,semanal (7 días en el 100% de los pasos)
"Marcas de tiempo duplicadas (municipio, semana)",0
Semanas esperadas / presentes por municipio,626 / 626
Huecos (semanas faltantes),0
Cobertura,2010-01-04 a 2021-12-27
Años con semana 53,"2015, 2020"
Filas donde ANO ≠ año ISO de la fecha,0


**Interpretación.** La serie es regular: una fecha por lunes, pasos exactos de 7 días, sin duplicados ni huecos. El calendario es el de semanas epidemiológicas (coincide con el año ISO), con semana 53 en 2015 y 2020. No hay zona horaria porque el dato es semanal.

## Serie completa y agregaciones

In [60]:
nac = eda.groupby("week_start")["casos_totales"].sum()  # Casos nacionales por semana
mensual = nac.resample("MS").sum()  # Por mes
anual = eda.groupby("ANO")["casos_totales"].sum()  # Por año
fig = make_subplots(3, 1, subplot_titles=("Casos por semana", "Casos por mes", "Casos por año"), vertical_spacing=0.09)  # Tres paneles
fig.add_scatter(x=nac.index, y=nac.values, line=dict(color=AZUL, width=1), showlegend=False, row=1, col=1)  # Semanal
fig.add_scatter(x=mensual.index, y=mensual.values, line=dict(color=VERDE), showlegend=False, row=2, col=1)  # Mensual
fig.add_bar(x=anual.index, y=anual.values, marker_color=AMBAR, showlegend=False, row=3, col=1)  # Anual
fig.update_layout(title="Casos de dengue en los 376 municipios (2010–2021)")  # Título
mostrar(fig, 720)  # Muestra

In [61]:
ln = np.log1p(nac)  # Escala logarítmica
tabla = pd.DataFrame({"log_casos": ln.values, "mes": ln.index.month, "semana": ln.index.isocalendar().week.values}, index=ln.index)  # Calendario
fig = make_subplots(1, 2, column_widths=[0.35, 0.65], subplot_titles=("Por mes", "Por semana epidemiológica"))  # Dos paneles
fig.add_box(x=tabla["mes"], y=tabla["log_casos"], marker_color=AZUL, showlegend=False, row=1, col=1)  # Meses
fig.add_box(x=tabla["semana"], y=tabla["log_casos"], marker_color=VERDE, showlegend=False, row=1, col=2)  # Semanas
fig.update_layout(title="Estacionalidad: log(1 + casos nacionales) por ciclo")  # Título
mostrar(fig, 380)  # Muestra
print(tabla.groupby("mes")["log_casos"].median().round(2).to_dict())  # Mediana por mes

{1: 7.28, 2: 7.37, 3: 7.33, 4: 7.27, 5: 7.13, 6: 7.12, 7: 6.77, 8: 6.89, 9: 6.91, 10: 6.92, 11: 7.05, 12: 7.23}


## Descomposición STL (tendencia, estacionalidad y residuo)

In [62]:
stl = STL(ln, period=52, robust=True).fit()  # Descomposición robusta con periodo anual
fig = make_subplots(4, 1, shared_xaxes=True, subplot_titles=("Serie (log)", "Tendencia", "Estacionalidad", "Residuo"), vertical_spacing=0.05)  # Cuatro paneles
for i, (s, col) in enumerate([(ln, AZUL), (stl.trend, ROJO), (stl.seasonal, VERDE), (stl.resid, GRIS)]):  # Componentes
    fig.add_scatter(x=s.index, y=s.values, line=dict(color=col, width=1.2), showlegend=False, row=i + 1, col=1)  # Línea
fig.update_layout(title="Descomposición STL de log(1 + casos nacionales)")  # Título
mostrar(fig, 760)  # Muestra
f_tend = max(0, 1 - np.var(stl.resid) / np.var(stl.resid + stl.trend))  # Fuerza de la tendencia (Wang et al., 2006)
f_est = max(0, 1 - np.var(stl.resid) / np.var(stl.resid + stl.seasonal))  # Fuerza de la estacionalidad
print(f"Fuerza de la tendencia: {f_tend:.2f} · fuerza de la estacionalidad: {f_est:.2f} (0 = nula, 1 = total)")  # Resultado

Fuerza de la tendencia: 0.66 · fuerza de la estacionalidad: 0.02 (0 = nula, 1 = total)


**Interpretación.** La serie nacional está dominada por la **tendencia de varios años** (ondas epidémicas de 2010, 2013 y 2019 con valles en 2011–2012 y 2017–2018), no por un ciclo anual. La estacionalidad existe (algo más de casos en el primer semestre, tras la primera temporada de lluvias) pero su fuerza es muy baja frente a la tendencia. Para el modelo esto significa que el calendario (seno y coseno de la semana) aporta poco y que la información relevante está en el **nivel reciente** de la serie: rezagos y medias móviles de casos.

## Estacionariedad: medias y varianzas móviles, ADF y KPSS

In [63]:
fig = make_subplots(specs=[[{"secondary_y": True}]])  # Dos ejes
fig.add_scatter(x=ln.index, y=ln.rolling(52).mean(), name="Media móvil 52 sem.", line=dict(color=AZUL))  # Media móvil
fig.add_scatter(x=ln.index, y=ln.rolling(52).std(), name="Desviación móvil 52 sem. (eje derecho)", line=dict(color=ROJO), secondary_y=True)  # Desviación móvil
fig.update_layout(title="Media y desviación móviles de log(1 + casos nacionales)")  # Título
mostrar(fig, 340)  # Muestra
filas = []  # Pruebas
for nombre, s in [("Nivel: log casos", ln), ("Primera diferencia", ln.diff().dropna())]:  # Serie y su diferencia
    adf = adfuller(s, autolag="AIC")  # H0: raíz unitaria (no estacionaria)
    kp = kpss(s, regression="c", nlags="auto")  # H0: estacionaria
    filas.append({"serie": nombre, "ADF estadístico": adf[0], "ADF p": adf[1], "KPSS estadístico": kp[0], "KPSS p": kp[1]})  # Resultado
pd.DataFrame(filas).set_index("serie").round(4)

,ADF estadístico,ADF p,KPSS estadístico,KPSS p
serie,,,,
Nivel: log casos,-2.5110,0.1128,0.2516,0.1000
Primera diferencia,-14.9251,0.0000,0.0506,0.1000


In [64]:
p_adf, p_kpss = [], []  # Pruebas por municipio
for _, g in eda[eda["ANO"] >= 2015].groupby("COD_MUN_N"):  # Cada municipio (2015–2021)
    x = np.log1p(g["casos_totales"].to_numpy())  # Serie del municipio
    if x.std() > 0:  # Si la serie varía
        p_adf.append(adfuller(x, maxlag=8, autolag=None)[1])  # ADF con 8 rezagos
        p_kpss.append(kpss(x, regression="c", nlags=8)[1])  # KPSS con 8 rezagos
p_adf, p_kpss = np.array(p_adf), np.array(p_kpss)  # Vectores
print(f"Municipios: {len(p_adf)} · ADF rechaza raíz unitaria (p < 0,05): {np.mean(p_adf < 0.05):.0%} · "
      f"KPSS rechaza estacionariedad (p < 0,05): {np.mean(p_kpss < 0.05):.0%}")  # Resumen

Municipios: 376 · ADF rechaza raíz unitaria (p < 0,05): 62% · KPSS rechaza estacionariedad (p < 0,05): 49%


**Interpretación.** En la serie nacional las dos pruebas no se contradicen de forma concluyente: ADF no rechaza la raíz unitaria y KPSS tampoco rechaza la estacionariedad, el caso típico de una serie **muy persistente** con ondas largas, donde las pruebas tienen poca potencia. La primera diferencia es claramente estacionaria. A nivel municipal la mayoría de las series rechaza la raíz unitaria, pero muchas también rechazan la estacionariedad. Consecuencias: (1) las medias y varianzas cambian con el tiempo, así que el desempeño de un año no garantiza el de otro; (2) se usan variables relativas al propio municipio (log de casos frente al umbral, anomalías climáticas) en lugar de niveles absolutos; y (3) la validación debe recorrer varios años distintos.

## Dependencia temporal: ACF, PACF y correlación cruzada

In [65]:
rez = 104  # Dos años de rezagos
a, p_ = acf(ln, nlags=rez), pacf(ln, nlags=52)  # ACF y PACF
banda = 1.96 / np.sqrt(len(ln))  # Banda de confianza aproximada
fig = make_subplots(1, 2, subplot_titles=("ACF de log casos nacionales", "PACF"))  # Dos paneles
fig.add_bar(x=np.arange(rez + 1), y=a, marker_color=AZUL, showlegend=False, row=1, col=1)  # ACF
fig.add_bar(x=np.arange(53), y=p_, marker_color=VERDE, showlegend=False, row=1, col=2)  # PACF
for col in (1, 2):  # Bandas
    fig.add_hline(y=banda, line_dash="dot", line_color=GRIS, row=1, col=col)  # Superior
    fig.add_hline(y=-banda, line_dash="dot", line_color=GRIS, row=1, col=col)  # Inferior
fig.update_layout(title="Autocorrelación de la serie nacional (bandas: ±1,96/√n)")  # Título
mostrar(fig, 360)  # Muestra
print(f"ACF en el rezago 4: {a[4]:.2f} · rezago 52: {a[52]:.2f} · PACF rezago 1: {p_[1]:.2f}, rezago 2: {p_[2]:.2f}")  # Valores clave

ACF en el rezago 4: 0.78 · rezago 52: -0.05 · PACF rezago 1: 0.81, rezago 2: 0.43


In [66]:
base = eda[eda["ANO"] >= 2011].copy()  # Desde 2011 (la normal y los rezagos ya existen)
base["lc_c"] = base["lc"] - base.groupby("COD_MUN_N")["lc"].transform("mean")  # Casos centrados por municipio
g = base.groupby("COD_MUN_N")  # Agrupación
ccf = {}  # Correlación cruzada media dentro de los municipios
for v in ["temp_anom", "prec_anom", "ndvi_anom", "oni_anom"]:  # Variables climáticas
    xc = base[v] - g[v].transform("mean")  # Centrada por municipio
    ccf[v] = [np.corrcoef(xc.groupby(base["COD_MUN_N"]).shift(k).fillna(0), base["lc_c"])[0, 1] for k in range(0, 21)]  # corr(x_{t−k}, casos_t)
ccf = pd.DataFrame(ccf, index=pd.Index(range(21), name="rezago (semanas)"))  # Tabla
fig = px.line(ccf, markers=True)  # Una línea por variable
fig.update_layout(title="Correlación cruzada dentro de los municipios: clima en t−k frente a casos en t", yaxis_title="correlación", legend_title="")  # Formato
mostrar(fig, 360)  # Muestra
ccf.abs().idxmax().to_frame("rezago de máxima |correlación|").join(ccf.abs().max().to_frame("máxima |correlación|")).round(3)

,rezago de máxima |correlación|,máxima |correlación|
temp_anom,0,0.0560
prec_anom,15,0.0360
ndvi_anom,12,0.0330
oni_anom,5,0.1720


**Interpretación.** La ACF decae muy lentamente (la semana actual explica buena parte de la semana $t+4$) y la PACF se concentra en los primeros rezagos: es un proceso autorregresivo persistente, lo que justifica rezagos de 1 a 3 semanas y medias móviles de 4, 8 y 12. En el rezago 52 la ACF es prácticamente 0: no hay un ciclo anual fuerte. La correlación cruzada del clima local con los casos es pequeña (|r| < 0,06): la lluvia y el NDVI alcanzan su máximo con 12–15 semanas de rezago y la temperatura de forma casi contemporánea. El ONI es la señal climática más fuerte (r ≈ 0,17) y llega con unas 5 semanas de rezago, coherente con el ciclo mosquito–virus–síntomas. Por eso se usan medias móviles largas del clima y el ONI rezagado 8 semanas.

## Cambios de régimen, eventos y calendario

In [67]:
sigma2 = np.var(stl.resid)  # Varianza del ruido (residuo de STL)
cortes = es.segmentacion_binaria(ln.to_numpy(), penalizacion=3 * 2 * sigma2 * np.log(len(ln)), min_tam=52)  # Regímenes de al menos un año
fechas_corte = [ln.index[i] for i in cortes]  # Fechas
segmentos = np.split(ln.to_numpy(), cortes)  # Segmentos
inicios = [ln.index[0]] + fechas_corte  # Inicio de cada segmento
oni = eda.drop_duplicates("week_start").set_index("week_start")["oni_anom"]  # ONI semanal
fig = go.Figure()  # Figura
for ini, fin in zip(oni.index[(oni > 0.5) & (oni.shift() <= 0.5)], oni.index[(oni <= 0.5) & (oni.shift() > 0.5)]):  # Episodios de El Niño
    fig.add_vrect(x0=ini, x1=fin, fillcolor=ROJO, opacity=0.08, line_width=0)  # Sombreado
fig.add_scatter(x=ln.index, y=ln.values, name="log casos", line=dict(color=AZUL, width=1))  # Serie
for ini, seg in zip(inicios, segmentos):  # Media de cada régimen
    fin = ln.index[min(ln.index.get_loc(ini) + len(seg) - 1, len(ln) - 1)]  # Fin del segmento
    fig.add_scatter(x=[ini, fin], y=[seg.mean()] * 2, mode="lines", line=dict(color=AMBAR, width=3), showlegend=False)  # Nivel
fig.update_layout(title="Regímenes detectados (ámbar) y episodios de El Niño (franjas rojas)")  # Título
mostrar(fig, 380)  # Muestra
print("Puntos de cambio:", ", ".join(str(f.date()) for f in fechas_corte))  # Fechas

Puntos de cambio: 2011-01-03, 2012-12-31, 2016-09-05, 2018-05-28, 2019-05-27, 2020-06-08


In [68]:
sem = eda.groupby(["ANO", "SEMANA"])["casos_totales"].sum().reset_index()  # Casos por año y semana
sem["vecinas"] = sem.groupby("ANO")["casos_totales"].transform(lambda s: s.rolling(7, center=True, min_periods=3).median())  # Mediana de semanas vecinas
fin_anio = sem[sem["SEMANA"].isin([52, 53, 1])]  # Fin y comienzo de año (festividades)
print(f"Casos en las semanas 52–53 y 1 frente a la mediana de sus vecinas: {(fin_anio['casos_totales'] / fin_anio['vecinas']).median():.2f}")  # Efecto de calendario
covid = sem[sem["ANO"] == 2020].set_index("SEMANA")["casos_totales"]  # 2020
print(f"2020, semanas 1–11 frente a 14–26 (inicio de la cuarentena): {covid.loc[1:11].mean():,.0f} → {covid.loc[14:26].mean():,.0f} casos por semana")  # Evento

Casos en las semanas 52–53 y 1 frente a la mediana de sus vecinas: 0.97
2020, semanas 1–11 frente a 14–26 (inicio de la cuarentena): 2,949 → 1,172 casos por semana


**Interpretación.** Los seis puntos de cambio siguen las ondas epidémicas: el fin de la epidemia de 2010 (enero de 2011), el inicio de la epidemia de 2013 (diciembre de 2012), el valle de 2017–2018 (septiembre de 2016), la subida previa a la epidemia de 2019 (mayo de 2018 y mayo de 2019) y la caída de junio de 2020, que coincide con la pandemia: entre las semanas 1–11 y 14–26 de 2020 los casos semanales cayeron de ≈ 2 950 a ≈ 1 170 (menos consultas y notificación). Los episodios de El Niño anteceden a los regímenes altos. El efecto de las festividades de fin de año es pequeño (≈ 3 % menos casos que en las semanas vecinas).

## Deriva temporal: *covariate shift* y *concept drift*

In [69]:
temprano, tardio = tr[tr["ANO"] <= 2018], tr[tr["ANO"] >= 2019]  # Dos mitades del entrenamiento
vars_deriva = ["lc_m4", "brote_m8", "ratio_umbral", "temp_mean_m4", "temp_anom_m8", "prec_total_m4", "ndvi_mean_m4", "oni_anom", "oni_l8", "flujo_m4", "lpob"]  # Variables
filas = []  # Deriva por variable
for v in vars_deriva:  # Cada variable
    filas.append({"variable": v, "PSI 2015–18 → 2019–21": es.psi(temprano[v], tardio[v]), "KS 2015–18 vs 2019–21": stats.ks_2samp(temprano[v], tardio[v])[0],  # Dentro del entrenamiento
                  "PSI entrenamiento → validación 2022": es.psi(tr[v], va[v]), "KS entrenamiento vs 2022": stats.ks_2samp(tr[v], va[v])[0]})  # Hacia la validación
deriva = pd.DataFrame(filas).set_index("variable")  # Tabla
print(f"Prevalencia del objetivo: 2015–18 {temprano[cf.TARGET].mean():.1%} · 2019–21 {tardio[cf.TARGET].mean():.1%} · 2022 {va[cf.TARGET].mean():.1%}")  # Deriva del objetivo
deriva.round(3)

Prevalencia del objetivo: 2015–18 7.5% · 2019–21 13.9% · 2022 12.4%


,PSI 2015–18 → 2019–21,KS 2015–18 vs 2019–21,PSI entrenamiento → validación 2022,KS entrenamiento vs 2022
variable,,,,
lc_m4,0.0400,0.1000,0.0070,0.0300
brote_m8,0.0590,0.1000,0.0030,0.0230
ratio_umbral,0.0780,0.1080,0.0330,0.0760
temp_mean_m4,0.0030,0.0200,0.0310,0.0330
temp_anom_m8,0.0270,0.0320,0.2010,0.1490
prec_total_m4,0.0020,0.0220,0.1050,0.1440
ndvi_mean_m4,0.0020,0.0110,0.0200,0.0560
oni_anom,1.9550,0.3000,7.4470,0.8640
oni_l8,1.8520,0.2600,7.5870,0.8870


In [70]:
anios = sorted(tr["ANO"].unique()) + [cf.ANIO_VALIDACION]  # Años
datos_anio = pd.concat([tr, va])  # Entrenamiento y validación
concepto = pd.DataFrame({v: [roc_auc_score(datos_anio.loc[datos_anio["ANO"] == a, cf.TARGET], datos_anio.loc[datos_anio["ANO"] == a, v]) for a in anios]  # AUC por año
                         for v in ["lc_m4", "brote_m8", "ratio_umbral", "oni_l8", "temp_anom_m8"]}, index=anios)
fig = px.line(concepto, markers=True)  # Una línea por variable
fig.add_hline(y=0.5, line_dash="dot", line_color=GRIS)  # Azar
fig.update_layout(title="Concept drift: AUC univariado de cada variable, año por año", yaxis_title="AUC", legend_title="")  # Formato
mostrar(fig, 360)  # Muestra
concepto.round(3)

,lc_m4,brote_m8,ratio_umbral,oni_l8,temp_anom_m8
2015,0.8310,0.8200,0.7170,0.5960,0.5530
2016,0.8640,0.8640,0.8110,0.6780,0.6440
2017,0.8300,0.7660,0.6760,0.5020,0.5020
2018,0.8860,0.8480,0.7900,0.6530,0.5510
2019,0.8830,0.8620,0.7900,0.4630,0.5090
2020,0.8710,0.8680,0.8260,0.6680,0.5570
2021,0.8910,0.8550,0.7820,0.5740,0.5170
2022,0.8940,0.8670,0.7860,0.5040,0.4590


**Interpretación.** La deriva de las covariables es baja para la historia de casos y el clima local dentro del entrenamiento (PSI < 0,1), moderada hacia 2022 para las anomalías de temperatura y la lluvia (PSI 0,10–0,20, un año de La Niña) y **fuerte para el ONI** (PSI ≫ 0,25): cada periodo tiene una fase distinta de El Niño, de modo que su distribución nunca es estable. La prevalencia del objetivo también cambia (*prior shift*: 7,5 % en 2015–2018, 13,9 % en 2019–2021, 12,4 % en 2022). En cambio, el **concepto** es estable para la historia de casos: el AUC univariado de `lc_m4` se mantiene entre 0,83 y 0,89 todos los años, mientras que el del ONI rezagado (0,46–0,68) y el de las anomalías de temperatura (0,46–0,64) varía mucho y llega a quedar por debajo de 0,5. Consecuencias: la relación casos → brote es la base confiable; las señales climáticas deben interpretarse con cautela y el umbral de decisión debe revisarse periódicamente porque la prevalencia cambia.

## Panel: series por entidad y heterogeneidad

In [71]:
orden = d.municipios().sort_values(["region", "latitud"])  # Orden por región y latitud
mat = tr.pivot_table(index="COD_MUN_N", columns="week_start", values=cf.TARGET).loc[orden["COD_MUN_N"]]  # Municipio × semana
fig = go.Figure(go.Heatmap(z=mat.values, x=mat.columns, y=np.arange(len(mat)), colorscale=[[0, "#f8fafc"], [1, ROJO]], showscale=False))  # Mapa de calor
cambios = orden["region"].ne(orden["region"].shift()).to_numpy().nonzero()[0]  # Inicio de cada región
fig.update_yaxes(tickvals=cambios, ticktext=orden["region"].iloc[cambios].tolist(), autorange="reversed")  # Etiquetas de región
fig.update_layout(title="Brote en cuatro semanas por municipio (filas) y semana (columnas), 2015–2021")  # Título
mostrar(fig, 640)  # Muestra

In [72]:
ejemplo = {76001: "Cali", 8001: "Barranquilla", 54001: "Cúcuta", 50001: "Villavicencio"}  # Municipios de ejemplo
serie = eda[eda["COD_MUN_N"].isin(ejemplo) & (eda["ANO"] >= 2015)]  # Series
fig = px.line(serie, x="week_start", y="lc", color=serie["COD_MUN_N"].map(ejemplo), labels={"lc": "log(1 + casos)", "color": ""})  # Líneas
fig.update_layout(title="Series de municipios de distintas regiones", xaxis_title="")  # Título
mostrar(fig, 360)  # Muestra
prev = tr.groupby("COD_MUN_N")[cf.TARGET].mean()  # Prevalencia por municipio
print(f"Prevalencia por municipio: mediana {prev.median():.1%}, percentil 10 {prev.quantile(0.1):.1%}, percentil 90 {prev.quantile(0.9):.1%}")  # Heterogeneidad

Prevalencia por municipio: mediana 8.2%, percentil 10 1.1%, percentil 90 22.4%


**Interpretación.** El mapa de calor muestra que los brotes llegan en **ondas sincronizadas** (franjas verticales que atraviesan muchas regiones a la vez, como en 2019) y también como brotes locales aislados. Las series municipales son heterogéneas en nivel y en tiempo: una epidemia no golpea a todos los municipios en la misma semana. Esto justifica normalizar por municipio (canal endémico, anomalías) y evaluar el error por conglomerados.

## Consecuencias para el modelado

* **Partición cronológica**, nunca aleatoria (2015–2021 / 2022 / 2023–2024) con purga de 4 semanas en cada frontera.
* **Validación cruzada de ventana creciente** por años (2018, 2019, 2020 y 2021), con la misma purga: es el equivalente a `TimeSeriesSplit` con `gap` igual al horizonte.
* **Rezagos y ventanas móviles solo con el pasado**: rezagos de 0–3 semanas y medias de 4, 8 y 12 semanas.
* **Componentes cíclicos** (seno y coseno de la semana) en lugar de variables indicadoras de mes.
* **Riesgo de deriva**: alto para el ONI y la prevalencia; el umbral debe recalibrarse y la evaluación debe hacerse en varios años.

# Componente espacial

Cada municipio se representa por su centroide oficial (DIVIPOLA, DANE). Se validan las coordenadas, se mapean el objetivo y las variables, se estudia el patrón de puntos, la autocorrelación espacial (Moran, LISA y Getis-Ord), la heterogeneidad entre regiones, el efecto de la escala (MAUP) y las consecuencias para la validación. Se usa el periodo de entrenamiento 2015–2021.

In [73]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
from scipy import stats  # Pruebas
from sklearn.cluster import DBSCAN  # Agrupamiento por densidad
df = d.panel()  # Panel completo
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Solo entrenamiento
m = d.municipios()  # Centroides de los 376 municipios
lat, lon = m["latitud"].to_numpy(), m["longitud"].to_numpy()  # Coordenadas (grados)
D = sp.haversine(lat, lon)  # Distancias de gran círculo (km)
xy = sp.proyectar(lat, lon)  # Coordenadas planas locales (km)

## Validación de coordenadas

In [74]:
pd.Series({
    "Sistema de referencia": "EPSG:4326 (WGS84, grados decimales), el de la DIVIPOLA",
    "Latitud en [−90°, 90°]": bool(np.all(np.abs(lat) <= 90)),
    "Longitud en [−180°, 180°]": bool(np.all(np.abs(lon) <= 180)),
    "Rango de latitud": f"{lat.min():.2f}° a {lat.max():.2f}°",
    "Rango de longitud": f"{lon.min():.2f}° a {lon.max():.2f}°",
    "Dentro de Colombia (lat −4,3 a 13,5; lon −82 a −66,8)": bool(np.all((lat > -4.3) & (lat < 13.5) & (lon > -82) & (lon < -66.8))),
    "Posible inversión lat/lon (|lat| > |lon|)": int(np.sum(np.abs(lat) > np.abs(lon))),
    "Coordenadas nulas o (0, 0)": int(np.sum(np.isnan(lat) | np.isnan(lon) | ((lat == 0) & (lon == 0)))),
    "Decimales de precisión": "6 (≈ 0,1 m; es un centroide, no una ubicación exacta)",
    "Coordenadas duplicadas": int(m.duplicated(["latitud", "longitud"]).sum()),
    "Municipios sin coordenadas": int(m["latitud"].isna().sum()),
}, name="valor").to_frame()

,valor
Sistema de referencia,"EPSG:4326 (WGS84, grados decimales), el de la ..."
"Latitud en [−90°, 90°]",True
"Longitud en [−180°, 180°]",True
Rango de latitud,-4.20° a 11.71°
Rango de longitud,-78.11° a -67.49°
"Dentro de Colombia (lat −4,3 a 13,5; lon −82 a −66,8)",True
Posible inversión lat/lon (|lat| > |lon|),0
"Coordenadas nulas o (0, 0)",0
Decimales de precisión,"6 (≈ 0,1 m; es un centroide, no una ubicación ..."
Coordenadas duplicadas,0


**Interpretación.** Las 376 coordenadas son válidas: dentro de los rangos y de Colombia (de Leticia, a 4,2° S, hasta La Guajira, a 11,7° N), sin inversiones, sin ceros y sin duplicados. Son **centroides**: representan al municipio entero, de modo que las distancias entre municipios son aproximadas y no hay información dentro de cada municipio.

## Mapas: puntos, densidad, objetivo y variables relevantes

In [75]:
todos = pd.read_csv(d.RUTA_DIVIPOLA, dtype=str)  # Los 1 122 municipios del país
for col in ("latitud", "longitud"):  # Coma decimal
    todos[col] = todos[col].str.replace(",", ".").astype(float)  # A número
todos["en_panel"] = todos["cod_mpio"].astype(int).isin(m["COD_MUN_N"])  # ¿Está en el panel?
fig = go.Figure()  # Mapa
fig.add_scattergeo(lat=todos.loc[~todos["en_panel"], "latitud"], lon=todos.loc[~todos["en_panel"], "longitud"], mode="markers",  # Fuera del panel
                   marker=dict(size=4, color="#cbd5e1"), name="Municipios fuera del panel")
pob = tr.groupby("COD_MUN_N")["lpob"].mean().loc[m["COD_MUN_N"]].to_numpy()  # log población
fig.add_scattergeo(lat=lat, lon=lon, mode="markers", text=m["municipio"], name="Municipios del panel",  # Panel
                   marker=dict(size=3 + 1.6 * (pob - pob.min()), color=AZUL, opacity=0.7, line=dict(width=0.3, color="white")))
fig.update_layout(title="Cobertura espacial: municipios del panel (tamaño ∝ log población) frente al resto del país")  # Título
mostrar(mapa_base(fig), 620)  # Muestra

In [76]:
agg = tr.groupby("COD_MUN_N").agg(casos=("casos_totales", "sum"), pob=("poblacion", "mean"), prevalencia=(cf.TARGET, "mean"),  # Perfil espacial
                                  semanas=("time_idx", "size")).loc[m["COD_MUN_N"]]
agg["incidencia"] = agg["casos"] / agg["pob"] / (agg["semanas"] / 52) * 1e5  # Casos por 100 000 habitantes por año
agg["log_incidencia"] = np.log1p(agg["incidencia"])  # Escala log
mm = m.join(agg.reset_index(drop=True))  # Une al mapa
fig = make_subplots(1, 2, specs=[[{"type": "scattergeo"}, {"type": "scattergeo"}]],  # Dos mapas
                    subplot_titles=("Incidencia anual (log, casos por 100 000)", "Proporción de semanas con brote en 4 semanas"))
fig.add_scattergeo(lat=lat, lon=lon, text=mm["municipio"], mode="markers", row=1, col=1,  # Incidencia
                   marker=dict(size=7, color=mm["log_incidencia"], colorscale="YlOrRd", colorbar=dict(x=0.45, title="log"), line=dict(width=0.3)))
fig.add_scattergeo(lat=lat, lon=lon, text=mm["municipio"], mode="markers", row=1, col=2,  # Prevalencia del brote
                   marker=dict(size=7, color=mm["prevalencia"], colorscale="YlOrRd", colorbar=dict(x=1.0, title="%", tickformat=".0%"), line=dict(width=0.3)))
fig.update_geos(fitbounds="locations", showcountries=True, landcolor="#f8fafc", showland=True)  # Encuadre
fig.update_layout(title="La variable objetivo y la carga de enfermedad en el espacio (2015–2021)", showlegend=False)  # Título
mostrar(fig, 520)  # Muestra

In [77]:
clima = tr[tr["temp_missing"] == 0].groupby("COD_MUN_N")["temp_mean_m4"].mean()  # Temperatura medida por municipio
otras = tr.groupby("COD_MUN_N")[["prec_total_m4", "flujo_m4"]].mean()  # Lluvia y flujo
mv = m.set_index("COD_MUN_N").join(clima).join(otras).reset_index()  # Une
fig = make_subplots(1, 3, specs=[[{"type": "scattergeo"}] * 3], subplot_titles=("Temperatura medida (°C)", "Lluvia (mm/semana)", "Flujo de personas (log)"))  # Tres mapas
for i, (v, esc) in enumerate([("temp_mean_m4", "RdBu_r"), ("prec_total_m4", "Blues"), ("flujo_m4", "Purples")]):  # Variables
    s = mv[v].notna()  # Municipios con dato
    fig.add_scattergeo(lat=mv.loc[s, "latitud"], lon=mv.loc[s, "longitud"], text=mv.loc[s, "municipio"], mode="markers", row=1, col=i + 1,
                       marker=dict(size=6, color=mv.loc[s, v], colorscale=esc, colorbar=dict(x=0.3 + 0.35 * i, len=0.8), line=dict(width=0.3)))
fig.update_geos(fitbounds="locations", showcountries=True, landcolor="#f8fafc", showland=True)  # Encuadre
fig.update_layout(title="Variables relevantes en el espacio (temperatura solo donde se mide)", showlegend=False)  # Título
mostrar(fig, 460)  # Muestra

:::{note}
El dataset no trae polígonos municipales, así que no se dibujan coropletas por municipio: se usan **mapas de símbolos** sobre los centroides, que cumplen la misma función. La densidad de puntos se analiza a continuación.
:::

## Patrón de puntos y cobertura

In [78]:
ce = sp.clark_evans(xy)  # Índice del vecino más cercano
distancias = np.array([10, 25, 50, 75, 100, 150, 200, 300])  # Escalas (km)
L, lo, hi = sp.ripley_l(xy, distancias, simulaciones=99)  # Función L con envolvente
print(f"Clark-Evans: R = {ce['R']:.3f} (distancia media al vecino más cercano {ce['r_obs']:.1f} km frente a {ce['r_esp']:.1f} km esperados), z = {ce['z']:.2f}, p = {ce['p']:.1e}")  # Resultado
fig = go.Figure()  # Figura
fig.add_scatter(x=distancias, y=hi, line=dict(color=GRIS, width=0), showlegend=False)  # Envolvente superior
fig.add_scatter(x=distancias, y=lo, fill="tonexty", fillcolor="rgba(100,116,139,0.2)", line=dict(color=GRIS, width=0), name="Envolvente CSR (99 sim.)")  # Envolvente
fig.add_scatter(x=distancias, y=L, mode="lines+markers", line=dict(color=ROJO), name="Observada")  # L observada
fig.add_hline(y=0, line_dash="dot", line_color=GRIS)  # Aleatoriedad
fig.update_layout(title="Función L de Ripley: L(d) − d", xaxis_title="distancia d (km)", yaxis_title="L(d) − d (km)")  # Formato
mostrar(fig, 360)  # Muestra

Clark-Evans: R = 0.772 (distancia media al vecino más cercano 23.2 km frente a 30.1 km esperados), z = -8.47, p = 2.5e-17


In [79]:
db = DBSCAN(eps=30 / sp.RADIO_TIERRA, min_samples=5, metric="haversine", algorithm="ball_tree").fit(np.radians(m[["latitud", "longitud"]]))  # DBSCAN con distancia haversine (30 km)
m_db = m.assign(cluster=["ruido" if l == -1 else f"C{l}" for l in db.labels_])  # Etiquetas
print(f"Clústeres DBSCAN (eps = 30 km, mínimo 5 municipios): {db.labels_.max() + 1} · municipios aislados (ruido): {(db.labels_ == -1).sum()}")  # Resultado
fig = px.scatter_geo(m_db, lat="latitud", lon="longitud", color="cluster", hover_name="municipio")  # Mapa
fig.update_traces(marker=dict(size=6))  # Tamaño
fig.update_layout(title="Agrupamientos de municipios por densidad (DBSCAN, haversine)")  # Título
mostrar(mapa_base(fig), 560)  # Muestra
m_db.groupby("cluster").agg(municipios=("municipio", "size"), ejemplo=("municipio", "first"), departamento=("departamento", "first")).sort_values("municipios", ascending=False).head(14)

Clústeres DBSCAN (eps = 30 km, mínimo 5 municipios): 13 · municipios aislados (ruido): 193


,municipios,ejemplo,departamento
cluster,,,
ruido,193,Abejorral,Antioquia
C3,37,Manizales,Caldas
C6,32,Anapoima,Cundinamarca
C7,23,Páez,Cauca
C11,18,Moniquirá,Boyacá
C0,13,Medellín,Antioquia
C1,10,Barranquilla,Atlántico
C10,10,Villavicencio,Meta
C5,10,Chinú,Córdoba


**Interpretación.** Los municipios del panel están **agrupados**, no distribuidos al azar: el índice de Clark-Evans es menor que 1 (R ≈ 0,77: la distancia media al vecino más cercano es 23 km frente a 30 km esperados) y la función L está por encima de la envolvente de aleatoriedad en todas las escalas analizadas. DBSCAN encuentra 13 núcleos densos, sobre todo en la región Andina (Eje Cafetero, Cundinamarca, Boyacá, Antioquia) y el Caribe, y deja aislados a 193 municipios, entre ellos casi todos los de la Orinoquía y la Amazonía, donde la cobertura es escasa. Es un **sesgo de muestreo espacial**: el modelo tendrá menos información para esas regiones, y cualquier métrica global estará dominada por las zonas densas.

## Matriz de pesos espaciales

Como el panel solo tiene un tercio de los municipios, la contigüidad (compartir frontera) no es aplicable: muchos vecinos reales no están. Se usan **k vecinos más cercanos** (distancia haversine), que garantizan que todo municipio tenga vecinos, con $k = 6$ y estandarización por filas (el rezago espacial es el promedio de los vecinos). Se comprueba la sensibilidad a esa elección.

In [80]:
k6 = np.sort(D, axis=1)[:, 6]  # Distancia al sexto vecino
print(f"Distancia al 6.º vecino: mediana {np.median(k6):.0f} km, percentil 90 {np.percentile(k6, 90):.0f} km, máximo {k6.max():.0f} km ({m.loc[np.argmax(k6), 'municipio']})")  # Escala
sens = []  # Sensibilidad del I de Moran a la matriz de pesos
for nombre, W in [("k = 4", sp.pesos_knn(D, 4)), ("k = 6", sp.pesos_knn(D, 6)), ("k = 8", sp.pesos_knn(D, 8)), ("k = 12", sp.pesos_knn(D, 12)),  # k vecinos
                  ("banda 50 km", sp.pesos_banda(D, 50)), ("banda 100 km", sp.pesos_banda(D, 100)), ("banda 200 km", sp.pesos_banda(D, 200))]:  # Bandas
    r = sp.moran(mm["log_incidencia"].to_numpy(), W, permutaciones=499)  # Moran
    sens.append({"matriz": nombre, "municipios sin vecinos": int((W.sum(axis=1) == 0).sum()), "I de Moran": r["I"], "z": r["z"], "p": r["p"]})  # Resultado
pd.DataFrame(sens).set_index("matriz").round(4)

Distancia al 6.º vecino: mediana 46 km, percentil 90 86 km, máximo 822 km (Leticia)


,municipios sin vecinos,I de Moran,z,p
matriz,,,,
k = 4,0,0.3319,9.7818,0.0020
k = 6,0,0.3081,11.0876,0.0020
k = 8,0,0.2754,11.7778,0.0020
k = 12,0,0.2546,14.0761,0.0020
banda 50 km,27,0.3274,9.0605,0.0020
banda 100 km,6,0.2060,10.7724,0.0020
banda 200 km,3,0.1430,13.6026,0.0020


## Autocorrelación espacial global y local

In [81]:
W = sp.pesos_knn(D, 6)  # Matriz elegida
glob = pd.DataFrame({v: sp.moran(mm[v].to_numpy(), W, permutaciones=999) for v in ["log_incidencia", "prevalencia"]}).T  # Moran global
glob.index = ["log incidencia anual", "proporción de semanas con brote"]  # Nombres
glob.round(4)

,I,E_I,z,p
log incidencia anual,0.3081,-0.0027,11.4422,0.0010
proporción de semanas con brote,0.0453,-0.0027,1.7035,0.0550


In [82]:
zx = mm["log_incidencia"] - mm["log_incidencia"].mean()  # Desviaciones
rez = W @ zx.to_numpy()  # Rezago espacial
pend = np.polyfit(zx, rez, 1)[0]  # Pendiente = I de Moran (con W estandarizada)
fig = px.scatter(x=zx, y=rez, hover_name=mm["municipio"], labels={"x": "log incidencia (centrada)", "y": "promedio de los 6 vecinos"})  # Diagrama de Moran
fig.add_scatter(x=[zx.min(), zx.max()], y=[pend * zx.min(), pend * zx.max()], mode="lines", line=dict(color=ROJO), name=f"pendiente = {pend:.3f}")  # Recta
fig.add_hline(y=0, line_color=GRIS)  # Ejes
fig.add_vline(x=0, line_color=GRIS)  # Ejes
fig.update_layout(title="Diagrama de dispersión de Moran (cuadrantes: AA arriba a la derecha, BB abajo a la izquierda)")  # Título
mostrar(fig, 420)  # Muestra

In [83]:
Ii, p_loc, cuad = sp.lisa(mm["log_incidencia"].to_numpy(), W, permutaciones=999)  # LISA
q_fdr = es.fdr_bh(p_loc)  # Corrección por FDR (376 pruebas)
mm["LISA"] = np.where(p_loc < 0.05, cuad, "No significativo")  # Clasificación con p < 0,05
mm["Gi*"] = sp.getis_ord(mm["log_incidencia"].to_numpy(), D, k=6)  # Getis-Ord
print("LISA (p < 0,05):", mm["LISA"].value_counts().to_dict())  # Conteos
print(f"LISA significativos tras FDR (q < 0,05): {(q_fdr < 0.05).sum()}")  # Con corrección
print(f"Gi*: focos calientes (z > 1,96): {(mm['Gi*'] > 1.96).sum()} · focos fríos (z < −1,96): {(mm['Gi*'] < -1.96).sum()}")  # Hotspots
fig = make_subplots(1, 2, specs=[[{"type": "scattergeo"}] * 2], subplot_titles=("LISA (p < 0,05)", "Getis-Ord Gi* (puntaje z)"))  # Dos mapas
for cat_, col in CUADRANTE_COLOR.items():  # Cuadrantes LISA
    s = mm["LISA"] == cat_  # Municipios
    fig.add_scattergeo(lat=mm.loc[s, "latitud"], lon=mm.loc[s, "longitud"], text=mm.loc[s, "municipio"], mode="markers", name=cat_,
                       marker=dict(size=7 if cat_ != "No significativo" else 4, color=col, line=dict(width=0.3)), row=1, col=1)
fig.add_scattergeo(lat=mm["latitud"], lon=mm["longitud"], text=mm["municipio"], mode="markers", showlegend=False, row=1, col=2,  # Gi*
                   marker=dict(size=6, color=mm["Gi*"], colorscale="RdBu_r", cmid=0, colorbar=dict(title="z", x=1.0), line=dict(width=0.3)))
fig.update_geos(fitbounds="locations", showcountries=True, landcolor="#f8fafc", showland=True)  # Encuadre
fig.update_layout(title="Focos espaciales de la incidencia de dengue (2015–2021)")  # Título
mostrar(fig, 520)  # Muestra
mm.loc[mm["LISA"] == "Alto-Alto", ["municipio", "departamento", "region"]].groupby("departamento").size().sort_values(ascending=False).to_frame("municipios Alto-Alto")

LISA (p < 0,05): {'No significativo': 252, 'Alto-Alto': 76, 'Bajo-Bajo': 30, 'Alto-Bajo': 11, 'Bajo-Alto': 7}
LISA significativos tras FDR (q < 0,05): 53
Gi*: focos calientes (z > 1,96): 66 · focos fríos (z < −1,96): 36


,municipios Alto-Alto
departamento,
Meta,23
Tolima,22
Huila,12
Putumayo,6
Cundinamarca,5
Santander,4
Casanare,3
Boyacá,1


**Interpretación.** La incidencia tiene **autocorrelación espacial positiva clara** (I de Moran ≈ 0,31 con k = 6, muy por encima de su esperanza −0,003), y el resultado es robusto: con cualquier k o banda de distancia el I es positivo y significativo. Los municipios vecinos tienen cargas de dengue parecidas. Los indicadores locales señalan 76 municipios Alto-Alto (focos calientes), concentrados en el Meta, el Tolima, el Huila y el Putumayo, y 30 Bajo-Bajo; Getis-Ord coincide (66 focos calientes y 36 fríos). Tras corregir las 376 pruebas por FDR siguen siendo significativos 53 municipios, así que los focos no son un artefacto de las comparaciones múltiples.

En cambio, la **proporción de semanas en brote** tiene una autocorrelación global débil y no significativa (I ≈ 0,05, p ≈ 0,06): el canal endémico compara a cada municipio consigo mismo y elimina buena parte de la diferencia de nivel entre zonas. El componente espacial del brote es **sincrónico** (vecinos que entran en brote al mismo tiempo), lo que se estudia en el capítulo espacio-temporal.

## Semivariograma empírico

Los datos son de área (municipios) y no puntuales y continuos, así que el semivariograma se presenta como descripción complementaria de la escala de dependencia, no para krigeado.

In [84]:
iu = np.triu_indices(len(m), 1)  # Pares sin repetir
dist, dif2 = D[iu], 0.5 * (mm["log_incidencia"].to_numpy()[:, None] - mm["log_incidencia"].to_numpy()[None, :])[iu] ** 2  # Distancia y semivarianza
bordes = np.arange(0, 820, 40)  # Intervalos de 40 km
clase = np.digitize(dist, bordes)  # Intervalo de cada par
vario = pd.DataFrame({"d": dist, "g": dif2, "c": clase}).groupby("c").agg(distancia=("d", "mean"), semivarianza=("g", "mean"), pares=("g", "size"))  # Semivariograma
fig = px.scatter(vario, x="distancia", y="semivarianza", size="pares", labels={"distancia": "distancia (km)"})  # Puntos
fig.add_hline(y=mm["log_incidencia"].var(), line_dash="dot", line_color=ROJO, annotation_text="varianza total (meseta)")  # Meseta
fig.update_layout(title="Semivariograma empírico de la log incidencia")  # Título
mostrar(fig, 360)  # Muestra

**Interpretación.** La semivarianza crece con la distancia y alcanza la meseta (la varianza total) hacia los 100 km: esa es la escala aproximada de la dependencia espacial (el **alcance**). Más allá, conocer a un municipio dice poco de otro. El efecto pepita es alto (≈ 70 % de la meseta ya en los primeros 40 km): buena parte de la variación es propia de cada municipio, y la dependencia espacial, aunque clara, es parcial.

## Heterogeneidad espacial

In [85]:
regiones = sorted(tr["region"].unique())  # Regiones
het = tr.groupby("region").agg(municipios=("COD_MUN_N", "nunique"), prevalencia=(cf.TARGET, "mean"),  # Por región
                               temperatura_medida=("temp_mean_m4", lambda s: s[tr.loc[s.index, "temp_missing"] == 0].mean()))
from sklearn.metrics import roc_auc_score  # AUC por región
for v in ["lc_m4", "oni_l8", "temp_anom_m8", "prec_total_m8"]:  # Relación de cada variable con el objetivo, por región
    het[f"AUC {v}"] = [roc_auc_score(tr.loc[tr["region"] == r, cf.TARGET], tr.loc[tr["region"] == r, v]) for r in het.index]  # AUC univariado
h, p, eps2 = es.kruskal([mm.loc[mm["region"] == r, "log_incidencia"] for r in regiones])  # Incidencia entre regiones
print(f"Kruskal-Wallis de la log incidencia entre regiones: H = {h:.1f}, p = {p:.2e}, ε² = {eps2:.3f}")  # Resultado
het.round(3)

Kruskal-Wallis de la log incidencia entre regiones: H = 64.6, p = 3.06e-13, ε² = 0.172


,municipios,prevalencia,temperatura_medida,AUC lc_m4,AUC oni_l8,AUC temp_anom_m8,AUC prec_total_m8
region,,,,,,,
Amazonía,19,0.1420,25.7140,0.8600,0.6260,0.5160,0.4730
Andina,173,0.0960,22.8160,0.8760,0.6930,0.5730,0.4690
Caribe,99,0.1020,28.1060,0.8990,0.6150,0.5280,0.5250
Orinoquía,41,0.1080,26.9060,0.8590,0.6600,0.5280,0.4870
Pacífica,44,0.1080,21.9690,0.8910,0.6570,0.6330,0.4630


**Interpretación.** Hay heterogeneidad espacial en el **nivel** (la incidencia difiere entre regiones, ε² ≈ 0,17) y en las **relaciones**: la historia de casos predice bien en todas las regiones (AUC 0,86–0,90), pero la asociación del ONI (0,62–0,69) y de las anomalías de temperatura (0,52–0,63, más fuerte en el Pacífico) cambia de intensidad, y la de la lluvia cambia incluso de dirección (AUC por encima de 0,5 en el Caribe y por debajo en las demás regiones). Un único coeficiente climático para todo el país es una simplificación; modelos con interacciones por región o no lineales (siguientes entregables) pueden capturarlo.

## Distancias y escala de agregación (MAUP)

In [86]:
grados = np.sqrt((lat[:, None] - lat[None, :]) ** 2 + (lon[:, None] - lon[None, :]) ** 2) * 111.32  # Distancia euclidiana sobre grados (incorrecta), en km
err = np.abs(grados[iu] - D[iu]) / D[iu]  # Error relativo frente a haversine
print(f"Error de usar distancia euclidiana sobre grados: mediana {np.median(err):.1%}, máximo {err.max():.1%} → se usa haversine")  # Justificación
dep = tr.groupby("depto").agg(casos=("casos_totales", "sum"), pob=("poblacion", "mean"), n=("COD_MUN_N", "nunique"), prevalencia=(cf.TARGET, "mean"))  # Nivel departamento
dep["pob"] = tr.groupby(["depto", "COD_MUN_N"])["poblacion"].mean().groupby("depto").sum()  # Población del departamento (municipios del panel)
dep["log_incidencia"] = np.log1p(dep["casos"] / dep["pob"] / 7 * 1e5)  # Incidencia anual
t_mun = tr[tr["temp_missing"] == 0].groupby("COD_MUN_N")["temp_mean_m4"].mean()  # Temperatura medida por municipio
t_dep = tr[tr["temp_missing"] == 0].groupby("depto")["temp_mean_m4"].mean()  # Por departamento
r_mun = stats.spearmanr(t_mun, mm.set_index("COD_MUN_N").loc[t_mun.index, "log_incidencia"])[0]  # Correlación municipal
r_dep = stats.spearmanr(t_dep, dep.loc[t_dep.index, "log_incidencia"])[0]  # Correlación departamental
print(f"Correlación de Spearman temperatura – incidencia: municipios {r_mun:.2f} (n = {len(t_mun)}) · departamentos {r_dep:.2f} (n = {len(t_dep)})")  # MAUP

Error de usar distancia euclidiana sobre grados: mediana 0.2%, máximo 2.1% → se usa haversine
Correlación de Spearman temperatura – incidencia: municipios 0.10 (n = 150) · departamentos 0.22 (n = 31)


**Interpretación.** Cerca del ecuador la distancia euclidiana sobre grados se equivoca poco (mediana 0,2 %, máximo 2 %), pero el error no es nulo y crece con la latitud (un grado de longitud no mide lo mismo en todas partes), así que se usa siempre haversine. El **efecto de la escala** (MAUP) es visible: la correlación entre temperatura e incidencia es 0,10 entre municipios y se duplica (0,22) entre departamentos, porque la agregación suaviza la variación local y exagera la relación. Por eso el modelo trabaja a escala municipal y las variables espaciales se construyen con vecinos, no con promedios departamentales.

## Ingeniería de características espaciales y consecuencias

* **Latitud y longitud crudas no se usan como predictoras**: el objetivo incluye generalizar a zonas nuevas, y las coordenadas permitirían memorizar *dónde* hubo brotes. La información geográfica relevante entra por el clima (que ya depende de la ubicación) y por variables de vecindad.
* **Variables de vecindad (rezago espacial)**: el promedio de la log incidencia y de los brotes de los k vecinos en la semana $t$ (disponibles en $t$). Su utilidad se confirma en el modelo base con el I de Moran de los residuos; se incorporan en los modelos de los siguientes entregables.
* **Validación por bloques espaciales**: como municipios cercanos se parecen (alcance ≈ 100 km), una validación aleatoria por municipios sería optimista. Se usa `GroupKFold` por **departamento** con una **zona de separación de 50 km**: se eliminan del entrenamiento los municipios a menos de 50 km de un municipio de prueba.

# Componente espacio-temporal

¿Cambian los patrones espaciales con el tiempo? ¿Se desplazan los focos? ¿Los brotes se agrupan a la vez en el espacio y en el tiempo? Se usan las semanas 2015–2021.

In [87]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
df = d.panel()  # Panel completo
eda = d.entrenamiento_eda(df)  # Semanas t de 2010–2021
e = eda[eda["ANO"] >= 2015].copy()  # Periodo con canal endémico
m = d.municipios()  # Centroides
D = sp.haversine(m["latitud"].to_numpy(), m["longitud"].to_numpy())  # Distancias (km)
W = sp.pesos_knn(D, 6)  # k = 6 vecinos, estandarizada por filas
print(f"Filas: {len(e):,} · municipios: {e['COD_MUN_N'].nunique()} · semanas: {e['time_idx'].nunique()}")  # Tamaño

Filas: 137,616 · municipios: 376 · semanas: 366


## Mapas por periodo

In [88]:
anual = e.groupby(["ANO", "COD_MUN_N"]).agg(casos=("casos_totales", "sum"), pob=("poblacion", "mean"), brote=("brote", "mean")).reset_index()  # Año × municipio
anual["log_incidencia"] = np.log1p(anual["casos"] / anual["pob"] * 1e5)  # Incidencia anual (log)
anual = anual.merge(m, on="COD_MUN_N")  # Coordenadas
fig = px.scatter_geo(anual, lat="latitud", lon="longitud", color="log_incidencia", animation_frame="ANO", hover_name="municipio",  # Animación por año
                     color_continuous_scale="YlOrRd", range_color=(0, anual["log_incidencia"].quantile(0.98)))
fig.update_traces(marker=dict(size=7, line=dict(width=0.3)))  # Tamaño
fig.update_layout(title="Log incidencia anual por municipio (use el control para recorrer los años)")  # Título
mostrar(mapa_base(fig), 600)  # Muestra

In [89]:
fig = px.scatter_geo(anual, lat="latitud", lon="longitud", color="brote", facet_col="ANO", facet_col_wrap=4,  # Paneles por año
                     color_continuous_scale="YlOrRd", range_color=(0, 0.5), hover_name="municipio")
fig.update_traces(marker=dict(size=3))  # Puntos pequeños
fig.update_geos(fitbounds="locations", showcountries=True, landcolor="#f8fafc", showland=True)  # Encuadre
fig.update_layout(title="Proporción de semanas en brote por municipio y año", coloraxis_colorbar_tickformat=".0%")  # Título
mostrar(fig, 620)  # Muestra

## Series por zona y comparación entre zonas

In [90]:
reg = e.groupby(["week_start", "region"]).agg(brote=("brote", "mean")).reset_index()  # Proporción en brote por región y semana
fig = px.line(reg, x="week_start", y="brote", color="region", labels={"brote": "municipios en brote", "week_start": ""})  # Líneas
fig.update_layout(title="Proporción de municipios en brote por región", yaxis_tickformat=".0%")  # Formato
mostrar(fig, 380)  # Muestra
piv = reg.pivot(index="week_start", columns="region", values="brote")  # Semana × región
piv.corr().round(2)  # Sincronía entre regiones

region,Amazonía,Andina,Caribe,Orinoquía,Pacífica
region,,,,,
Amazonía,1.0000,0.7200,0.5900,0.6000,0.3800
Andina,0.7200,1.0000,0.6400,0.6800,0.4700
Caribe,0.5900,0.6400,1.0000,0.6000,0.0100
Orinoquía,0.6000,0.6800,0.6000,1.0000,0.0300
Pacífica,0.3800,0.4700,0.0100,0.0300,1.0000


**Interpretación.** Las regiones comparten las grandes ondas (2015–2016 y 2019), pero con distinta intensidad y desfase. La región Andina, la Amazonía, la Orinoquía y el Caribe están bastante sincronizadas (correlaciones de 0,59 a 0,72), mientras que el Pacífico sigue una dinámica casi propia (0,01 con el Caribe y 0,03 con la Orinoquía). El patrón espacial **cambia año a año**: en 2017 casi no hay brotes en ninguna parte, en 2019 se extienden por gran parte del país, y en los demás años aparecen focos localizados que cambian de lugar.

## ¿Se desplazan los focos? Getis-Ord por año y focos emergentes

Se calcula Gi* de la log incidencia de cada año y se aplica la prueba de Mann-Kendall a la serie de puntajes z de cada municipio: un foco **persistente** es caliente casi todos los años; uno **emergente** se vuelve caliente con tendencia creciente; uno **decreciente** pierde intensidad; uno **esporádico** aparece y desaparece.

In [91]:
anios = sorted(anual["ANO"].unique())  # Años
gi = pd.DataFrame({a: sp.getis_ord(anual[anual["ANO"] == a].set_index("COD_MUN_N").loc[m["COD_MUN_N"], "log_incidencia"].to_numpy(), D, k=6)  # Gi* por año
                   for a in anios}, index=m["COD_MUN_N"])
caliente = gi > 1.96  # Foco caliente en el año
filas = []  # Clasificación
for mun, z in gi.iterrows():  # Cada municipio
    s, zmk, pmk = es.mann_kendall(z.to_numpy())  # Tendencia de Gi*
    n_cal = int(caliente.loc[mun].sum())  # Años como foco caliente
    if n_cal >= 0.8 * len(anios):  # Casi siempre caliente
        clase = "Persistente"
    elif caliente.loc[mun].iloc[-1] and pmk < 0.1 and zmk > 0:  # Caliente al final y en aumento
        clase = "Emergente / intensificándose"
    elif n_cal > 0 and pmk < 0.1 and zmk < 0:  # Fue caliente y disminuye
        clase = "Decreciente"
    elif n_cal > 0:  # Caliente algunos años sin tendencia
        clase = "Esporádico"
    else:  # Nunca caliente
        clase = "Nunca foco"
    filas.append({"COD_MUN_N": mun, "categoría": clase, "años como foco": n_cal, "tendencia z (MK)": zmk})  # Resultado
focos = pd.DataFrame(filas).merge(m, on="COD_MUN_N")  # Une coordenadas
print(focos["categoría"].value_counts().to_string())  # Conteo
colores = {"Persistente": ROJO, "Emergente / intensificándose": AMBAR, "Decreciente": AZUL, "Esporádico": "#f9a8d4", "Nunca foco": "#e2e8f0"}  # Colores
fig = px.scatter_geo(focos, lat="latitud", lon="longitud", color="categoría", color_discrete_map=colores, hover_name="municipio",  # Mapa
                     hover_data={"años como foco": True, "latitud": False, "longitud": False})
fig.update_traces(marker=dict(size=6))  # Tamaño
fig.update_layout(title="Clasificación de focos calientes 2015–2021 (Gi* anual + Mann-Kendall)")  # Título
mostrar(mapa_base(fig), 560)  # Muestra

categoría
Nunca foco                      193
Esporádico                      159
Decreciente                       8
Emergente / intensificándose      8
Persistente                       8


**Interpretación.** De 376 municipios, 193 nunca fueron foco caliente, 159 lo fueron de forma esporádica y solo 8 son persistentes; otros 8 son emergentes y 8 decrecientes. Es decir, los focos **se desplazan** entre años. Esa movilidad es la razón por la que la posición geográfica fija (latitud y longitud) no basta para predecir: importa qué está pasando ahora en el municipio y en sus vecinos.

## Interacción espacio-tiempo: prueba de Knox

Eventos: el **inicio** de cada episodio de brote (primera semana en brote tras una semana sin brote). Se cuentan los pares de inicios cercanos en el espacio y en el tiempo y se comparan con 999 permutaciones de los tiempos.

In [92]:
b = e[["COD_MUN_N", "time_idx", "brote"]].sort_values(["COD_MUN_N", "time_idx"])  # Brotes por municipio
b["inicio"] = (b["brote"] == 1) & (b.groupby("COD_MUN_N")["brote"].shift(1) == 0)  # Primera semana de cada episodio
ev = b[b["inicio"]].merge(m, on="COD_MUN_N")  # Eventos con coordenadas
xy_ev = sp.proyectar(ev["latitud"].to_numpy(), ev["longitud"].to_numpy())  # Coordenadas planas (km)
filas = []  # Resultados
for ds, dt in [(25, 2), (50, 4), (100, 4), (100, 8), (200, 8)]:  # Umbrales de cercanía (km, semanas)
    r = sp.knox(xy_ev, ev["time_idx"].to_numpy(), ds, dt, permutaciones=999)  # Prueba de Knox
    filas.append({"distancia (km)": ds, "tiempo (semanas)": dt, "pares observados": r["observado"], "pares esperados": round(r["esperado"]),  # Conteos
                  "razón obs/esp": r["razon"], "p": r["p"]})
print(f"Inicios de brote 2015–2021: {len(ev):,}")  # Número de eventos
pd.DataFrame(filas).round(4)

Inicios de brote 2015–2021: 4,972


,distancia (km),tiempo (semanas),pares observados,pares esperados,razón obs/esp,p
0,25,2,2278,2017,1.1291,0.0010
1,50,4,11127,8721,1.2758,0.0010
2,100,4,29314,24233,1.2097,0.0010
3,100,8,54888,45297,1.2117,0.0010
4,200,8,161579,144888,1.1152,0.0010


**Interpretación.** En todas las escalas hay **más pares de inicios cercanos a la vez en espacio y tiempo de lo esperado** si fueran independientes (razón > 1, p ≈ 0,001): los brotes se contagian o comparten causas entre municipios vecinos y en semanas cercanas. Es la evidencia más directa de que un municipio cuyo vecino acaba de entrar en brote tiene más riesgo, y justifica variables de vecindad rezagadas en el tiempo.

## Autocorrelación espacio-temporal

In [93]:
piv = e.pivot_table(index="time_idx", columns="COD_MUN_N", values="lc").loc[:, m["COD_MUN_N"]]  # Semana × municipio (log casos)
anom = piv - piv.mean()  # Centra cada municipio (quita el nivel propio)
I_sem = []  # I de Moran de cada semana
for t, fila in anom.iterrows():  # Cada semana
    z = fila.to_numpy() - fila.mean()  # Desviaciones de la semana
    I_sem.append((z @ W @ z) / (z @ z) if z @ z > 0 else np.nan)  # I de Moran (W estandarizada: S0 = n)
I_sem = pd.Series(I_sem, index=e.drop_duplicates("time_idx").set_index("time_idx").loc[anom.index, "week_start"])  # Serie semanal
fig = px.line(x=I_sem.index, y=I_sem.values, labels={"x": "", "y": "I de Moran"})  # Línea
fig.add_hline(y=-1 / (len(m) - 1), line_dash="dot", line_color=GRIS, annotation_text="E[I]")  # Esperanza
fig.update_layout(title="I de Moran semanal del log de casos centrado por municipio")  # Título
mostrar(fig, 340)  # Muestra
print(f"I de Moran semanal: mediana {I_sem.median():.3f}, percentil 10 {I_sem.quantile(0.1):.3f}, percentil 90 {I_sem.quantile(0.9):.3f}")  # Resumen

I de Moran semanal: mediana 0.145, percentil 10 0.047, percentil 90 0.261


In [94]:
Wd = pd.DataFrame(W, index=m["COD_MUN_N"], columns=m["COD_MUN_N"])  # Pesos con índices
rezago = anom @ Wd.T  # Rezago espacial de cada municipio en cada semana (promedio de vecinos)
filas = []  # Correlación cruzada espacio-temporal
for k in range(0, 13):  # Rezagos temporales
    propio = anom.shift(-k)  # Municipio en t + k
    filas.append({"k (semanas)": k, "corr(vecinos en t, municipio en t+k)": np.corrcoef(rezago.iloc[: len(anom) - k].to_numpy().ravel(), propio.iloc[: len(anom) - k].to_numpy().ravel())[0, 1],
                  "corr(municipio en t, municipio en t+k)": np.corrcoef(anom.iloc[: len(anom) - k].to_numpy().ravel(), propio.iloc[: len(anom) - k].to_numpy().ravel())[0, 1]})
cruz = pd.DataFrame(filas).set_index("k (semanas)")  # Tabla
fig = px.line(cruz, markers=True)  # Líneas
fig.update_layout(title="¿Anticipa el vecindario lo que pasará en el municipio?", yaxis_title="correlación", legend_title="")  # Formato
mostrar(fig, 340)  # Muestra
cruz.round(3)

,"corr(vecinos en t, municipio en t+k)","corr(municipio en t, municipio en t+k)"
k (semanas),,
0,0.4340,1.0000
1,0.4240,0.6880
2,0.4190,0.6620
3,0.4120,0.6430
4,0.4050,0.6180
5,0.3950,0.5940
6,0.3860,0.5710
7,0.3750,0.5480
8,0.3640,0.5260


**Interpretación.** La autocorrelación espacial de los casos (ya centrados por municipio) es positiva casi todas las semanas (I de Moran semanal con mediana 0,15) y sube durante las epidemias: los vecinos se desvían de su normalidad al mismo tiempo. Además, el promedio de los vecinos en la semana $t$ está correlacionado con lo que pasará en el municipio en $t+4$ (r ≈ 0,41), aunque menos que la propia historia del municipio (r ≈ 0,62): el vecindario **añade** información, pero no reemplaza la historia propia.

## Consecuencias para el modelado

* **Objetivo de predicción.** El uso principal es predecir el **futuro** de los municipios vigilados. Como la dinámica espacio-temporal también importa para municipios sin historia, se reportan tres esquemas por separado:
  1. **Futuro, mismas zonas**: entrenar 2015–2021 y evaluar 2023–2024 en los 376 municipios (esquema principal).
  2. **Zonas nuevas, mismo periodo**: bloques por departamento dentro de 2015–2021, con 50 km de separación.
  3. **Futuro y zonas nuevas**: entrenar 2015–2021 sin un bloque de departamentos y evaluar ese bloque en 2023–2024.
* **Riesgo de fuga por vecindad.** Una variable de vecindad calculada con datos de la semana objetivo de los vecinos sería fuga; las variables de vecindad deben usar solo semanas ≤ $t$. En la validación por zonas nuevas, los vecinos del entrenamiento a menos de 50 km de la prueba se excluyen para no filtrar información a través de la autocorrelación.

# Preprocesamiento

Cada decisión se rastrea a un hallazgo del EDA. Hay dos niveles:

1. **Ingeniería de características** (`src/datos.py`): rezagos, medias móviles, anomalías y componentes cíclicos. Ninguna se *ajusta* con datos de otras filas, salvo la normal climática, que se estima solo con 2010–2021. Todas usan semanas ≤ $t$.
2. **Pipeline de scikit-learn**: imputación y estandarización, ajustadas **solo con el entrenamiento** (dentro de cada pliegue de la validación cruzada, el Pipeline se reajusta con la parte de entrenamiento del pliegue).

In [95]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
df = d.panel()  # Panel completo
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Particiones purgadas
print(f"Entrenamiento {len(tr):,} · validación {len(va):,} · prueba {len(te):,} filas · {len(cf.FEATURES)} predictoras")  # Tamaños

Entrenamiento 132,728 · validación 18,048 · prueba 37,600 filas · 34 predictoras


## Decisiones y hallazgos que las motivan

In [96]:
decisiones = pd.DataFrame([  # (paso, decisión, hallazgo del EDA, capítulo)
    ("Partición", "Cronológica 2015–21 / 2022 / 2023–24 con purga de 4 semanas", "Autocorrelación temporal; objetivo en t+4", "Objetivo, Temporal"),
    ("Validación", "Ventana creciente por años (2018–2021) con purga", "Prevalencia y régimen cambian por año", "Objetivo, Temporal"),
    ("Imputación", "Se conserva temp_missing (método del indicador); SimpleImputer(mediana) como protección", "Faltantes MAR estructurales por municipio", "Base de datos"),
    ("Transformación", "log(1 + x) de casos, flujo y población", "Asimetría > 14 y curtosis en cientos", "Univariado"),
    ("Escalamiento", "StandardScaler dentro del Pipeline", "Escalas muy distintas; penalización L2", "Univariado, Bivariado"),
    ("Codificación categórica", "Semana → seno y coseno; departamento y región no se codifican", "Estacionalidad débil y cíclica; departamentos raros", "Univariado, Temporal"),
    ("Atípicos", "No se eliminan; se comprimen con el logaritmo", "Atípicos = epidemias reales con mucho más brote", "Base de datos, Multivariado"),
    ("Rezagos", "log casos en t, t−1, t−2, t−3", "PACF concentrada en los primeros rezagos", "Temporal"),
    ("Ventanas móviles", "Medias de 4, 8 y 12 semanas de casos y clima", "ACF persistente; clima actúa con 8–12 semanas", "Temporal"),
    ("Anomalías climáticas", "Desviación respecto a la normal mensual del municipio (2010–2021)", "El clima difiere mucho entre regiones (ε² ≈ 0,41)", "Bivariado, Espacial"),
    ("Canal endémico", "Casos frente al umbral objetivo (ratio_umbral) y umbral objetivo", "Brote se define relativo a la historia propia", "Objetivo"),
    ("El Niño", "ONI actual y rezagado 8 semanas", "Deriva fuerte del ONI; efecto rezagado", "Bivariado, Temporal"),
    ("Regularización", "L2 con C elegido por validación cruzada", "VIF ≫ 10 entre medias móviles", "Bivariado"),
    ("Desbalance", "class_weight None vs balanced en la malla; sin SMOTE", "≈ 10 % de brotes; prevalencia cambia por año", "Objetivo"),
    ("Variables espaciales", "No en el modelo base; propuestas para los siguientes entregables", "Moran ≈ 0,31; Knox significativo", "Espacial, Espacio-temporal"),
    ("Coordenadas", "Latitud y longitud no se usan como predictoras", "Focos que se desplazan; generalizar a zonas nuevas", "Espacio-temporal"),
], columns=["paso", "decisión", "hallazgo del EDA", "capítulo"])
decisiones

,paso,decisión,hallazgo del EDA,capítulo
0,Partición,Cronológica 2015–21 / 2022 / 2023–24 con purga...,Autocorrelación temporal; objetivo en t+4,"Objetivo, Temporal"
1,Validación,Ventana creciente por años (2018–2021) con purga,Prevalencia y régimen cambian por año,"Objetivo, Temporal"
2,Imputación,Se conserva temp_missing (método del indicador...,Faltantes MAR estructurales por municipio,Base de datos
3,Transformación,"log(1 + x) de casos, flujo y población",Asimetría > 14 y curtosis en cientos,Univariado
4,Escalamiento,StandardScaler dentro del Pipeline,Escalas muy distintas; penalización L2,"Univariado, Bivariado"
5,Codificación categórica,Semana → seno y coseno; departamento y región ...,Estacionalidad débil y cíclica; departamentos ...,"Univariado, Temporal"
6,Atípicos,No se eliminan; se comprimen con el logaritmo,Atípicos = epidemias reales con mucho más brote,"Base de datos, Multivariado"
7,Rezagos,"log casos en t, t−1, t−2, t−3",PACF concentrada en los primeros rezagos,Temporal
8,Ventanas móviles,"Medias de 4, 8 y 12 semanas de casos y clima",ACF persistente; clima actúa con 8–12 semanas,Temporal
9,Anomalías climáticas,Desviación respecto a la normal mensual del mu...,"El clima difiere mucho entre regiones (ε² ≈ 0,41)","Bivariado, Espacial"


## Las 34 variables predictoras

In [97]:
familia = {**{v: "Historia de casos" for v in cf.CASOS}, **{v: "Clima y ambiente" for v in cf.CLIMA}, **{v: "Contexto" for v in cf.CONTEXTO}}  # Familia
formula = {  # Definición de cada variable (todas con semanas ≤ t)
    "lc_l0": "log(1 + casos_t)", "lc_l1": "log(1 + casos_{t−1})", "lc_l2": "log(1 + casos_{t−2})", "lc_l3": "log(1 + casos_{t−3})",
    "lc_m4": "media de log casos en t−3…t", "lc_m8": "media en t−7…t", "lc_m12": "media en t−11…t",
    "brote_l0": "brote en t (canal endémico)", "brote_m8": "proporción de semanas en brote en t−7…t",
    "ratio_umbral": "log(1 + casos_t) − log(1 + u_{t+4})", "l_umbral_obj": "log(1 + u_{t+4}) (años anteriores)", "incid": "casos_t / población × 10⁵",
    "temp_anom_m8": "media 8 sem. de (temp − normal del mes)", "prec_anom_m8": "media 8 sem. de (lluvia − normal)", "ndvi_anom_m8": "media 8 sem. de (NDVI − normal)",
    "oni_anom": "ONI de la semana", "oni_l8": "ONI de t−8", "temp_missing": "1 si la temperatura de t es imputada",
    "sem_sin": "sen(2π·semana/52)", "sem_cos": "cos(2π·semana/52)", "lpob": "log(población)", "flujo_m4": "log(1 + media 4 sem. del flujo)",
}
for v in ("temp_mean", "prec_total", "ndvi_mean"):  # Medias móviles del clima
    for w in (1, 4, 8, 12):  # Ventanas
        formula[f"{v}_m{w}"] = f"media de {v} en t−{w - 1}…t" if w > 1 else f"{v} de la semana t"
pd.DataFrame({"familia": familia, "definición": formula}).loc[cf.FEATURES]

,familia,definición
lc_l0,Historia de casos,log(1 + casos_t)
lc_l1,Historia de casos,log(1 + casos_{t−1})
lc_l2,Historia de casos,log(1 + casos_{t−2})
lc_l3,Historia de casos,log(1 + casos_{t−3})
lc_m4,Historia de casos,media de log casos en t−3…t
lc_m8,Historia de casos,media en t−7…t
lc_m12,Historia de casos,media en t−11…t
brote_l0,Historia de casos,brote en t (canal endémico)
brote_m8,Historia de casos,proporción de semanas en brote en t−7…t
ratio_umbral,Historia de casos,log(1 + casos_t) − log(1 + u_{t+4})


## El Pipeline

In [98]:
from sklearn import set_config  # Diagrama del Pipeline
set_config(display="diagram")  # Muestra el Pipeline como diagrama
pipe = mb.pipeline_base()  # SimpleImputer → StandardScaler → LogisticRegression
pipe

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(max_iter=5000))])

In [99]:
prep = mb.pipeline_base()[:-1].fit(tr[cf.FEATURES])  # Solo el preprocesamiento, ajustado con el entrenamiento
Xtr = prep.transform(tr[cf.FEATURES])  # Entrenamiento transformado
Xva = prep.transform(va[cf.FEATURES])  # Validación transformada con los parámetros del entrenamiento
chequeo = pd.DataFrame({"media entrenamiento": Xtr.mean(axis=0), "desv. entrenamiento": Xtr.std(axis=0),  # ≈ 0 y 1
                        "media validación": Xva.mean(axis=0), "desv. validación": Xva.std(axis=0)}, index=cf.FEATURES)  # No exactamente 0 y 1
print(f"Nulos tras el preprocesamiento: {int(np.isnan(Xtr).sum())} · medianas de imputación aprendidas solo con {len(tr):,} filas de entrenamiento")  # Verificación
chequeo.round(3).head(12)

Nulos tras el preprocesamiento: 0 · medianas de imputación aprendidas solo con 132,728 filas de entrenamiento


,media entrenamiento,desv. entrenamiento,media validación,desv. validación
lc_l0,-0.0000,1.0000,0.0120,0.9970
lc_l1,0.0000,1.0000,0.0090,0.9940
lc_l2,-0.0000,1.0000,0.0060,0.9920
lc_l3,0.0000,1.0000,0.0030,0.9900
lc_m4,-0.0000,1.0000,0.0080,0.9940
lc_m8,-0.0000,1.0000,0.0020,0.9910
lc_m12,0.0000,1.0000,-0.0010,0.9910
brote_l0,0.0000,1.0000,0.0540,1.0670
brote_m8,0.0000,1.0000,0.0570,1.0620
ratio_umbral,0.0000,1.0000,0.1460,0.8970


**Interpretación.** En el entrenamiento, cada variable queda con media 0 y desviación 1; en la validación no exactamente, porque se transforma con la media y la desviación **del entrenamiento**. Esa pequeña diferencia es la prueba de que el preprocesamiento no vio la validación. No hay nulos: el `SimpleImputer` solo actuaría si en producción llegara una semana sin dato. La normal climática de las anomalías se estimó con 2010–2021, así que la prueba tampoco influye en ella.

# Modelo base: regresión logística

* **Objetivo:** $y_{m,t} = 1$ si el municipio $m$ está en brote en la semana $t+4$ (canal endémico).
* **Modelo base:** regresión logística con penalización L2, dentro de un `Pipeline` (imputación → estandarización → modelo), ajustada con `GridSearchCV` y **validación cruzada hacia adelante por años con purga** (el mismo tipo de partición que entrenamiento/prueba).
* **Líneas base triviales:** `DummyClassifier` (proporción de entrenamiento), **persistencia** (¿hay brote ahora?), **persistencia de 8 semanas** (proporción reciente de semanas en brote) e **ingenuo estacional** (¿hubo brote en la misma semana hace un año?).
* **Selección:** AUC-PR en la validación cruzada (clase minoritaria); umbral de decisión que maximiza F1 en la validación de 2022. La prueba 2023–2024 se usa **una sola vez**, en este capítulo.

In [100]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve  # Métricas y curvas
df = d.panel()  # Panel completo
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Particiones purgadas
print(f"Entrenamiento {len(tr):,} · validación {len(va):,} · prueba {len(te):,} filas")  # Tamaños

Entrenamiento 132,728 · validación 18,048 · prueba 37,600 filas


## Entrenamiento: Pipeline + GridSearchCV con validación purgada

In [101]:
grid = mb.ajustar_base(tr)  # Prueba 10 combinaciones × 4 pliegues
res = pd.DataFrame(grid.cv_results_)  # Resultados de la validación cruzada
tabla = pd.DataFrame({"C": res["param_clf__C"], "class_weight": res["param_clf__class_weight"].astype(str),  # Hiperparámetros
                      "AUC-PR (media)": res["mean_test_auc_pr"], "AUC-PR (desv.)": res["std_test_auc_pr"],  # Criterio
                      "AUC-ROC (media)": res["mean_test_auc_roc"], "AUC-ROC entrenamiento": res["mean_train_auc_roc"],  # Diagnóstico de sobreajuste
                      "Brier (media)": -res["mean_test_neg_brier"]}).sort_values("AUC-PR (media)", ascending=False)  # Calibración
print("Mejores hiperparámetros:", grid.best_params_)  # Elección
tabla.round(4)

Mejores hiperparámetros: {'clf__C': 0.01, 'clf__class_weight': None}


,C,class_weight,AUC-PR (media),AUC-PR (desv.),AUC-ROC (media),AUC-ROC entrenamiento,Brier (media)
2,0.0100,None,0.6433,0.0821,0.8985,0.9011,0.0631
4,0.1000,None,0.6429,0.0819,0.8980,0.9009,0.0630
6,1.0000,None,0.6422,0.0820,0.8973,0.9007,0.0630
8,10.0000,None,0.6419,0.0821,0.8970,0.9007,0.0630
0,0.0010,None,0.6414,0.0826,0.8998,0.9016,0.0635
3,0.0100,balanced,0.6373,0.0828,0.9015,0.9040,0.0969
5,0.1000,balanced,0.6370,0.0824,0.9010,0.9040,0.0969
7,1.0000,balanced,0.6366,0.0825,0.9005,0.9040,0.0970
9,10.0000,balanced,0.6365,0.0826,0.9004,0.9040,0.0971
1,0.0010,balanced,0.6364,0.0840,0.9020,0.9036,0.0979


In [102]:
mejor = grid.best_index_  # Fila de la mejor combinación
pliegues = pd.DataFrame({"año validado": [2018, 2019, 2020, 2021],  # Desempeño por pliegue
                         "AUC-PR": [res.loc[mejor, f"split{i}_test_auc_pr"] for i in range(4)],
                         "AUC-ROC": [res.loc[mejor, f"split{i}_test_auc_roc"] for i in range(4)]}).set_index("año validado")
pliegues.round(4)

,AUC-PR,AUC-ROC
año validado,,
2018,0.6089,0.8969
2019,0.7812,0.8956
2020,0.6184,0.8993
2021,0.5648,0.9022


**Interpretación.** El AUC-PR cambia muy poco entre valores de $C$: con 130 000 filas la regularización importa poco para el desempeño, pero la más fuerte razonable ($C = 0{,}01$) estabiliza los coeficientes ante la multicolinealidad. `class_weight="balanced"` sube levemente el AUC-ROC pero baja el AUC-PR y empeora la calibración (infla las probabilidades), así que no se pondera. El desempeño por pliegue varía con el año (2019, año epidémico, es el más fácil en AUC-PR), lo que confirma que una sola partición no basta para estimar la incertidumbre.

## Umbral de decisión (validación 2022) y evaluación en la prueba 2023–2024

In [103]:
modelo = grid.best_estimator_  # Pipeline reentrenado con todo el entrenamiento
p_va = modelo.predict_proba(va[cf.FEATURES])[:, 1]  # Probabilidades en validación
p_te = modelo.predict_proba(te[cf.FEATURES])[:, 1]  # Probabilidades en prueba (única vez)
umbral = mb.umbral_f1(va[cf.TARGET], p_va)  # Umbral que maximiza F1 en 2022
lb_va, lb_te = mb.lineas_base(tr, va), mb.lineas_base(tr, te)  # Líneas base
umbrales = {"Regresión logística": umbral, "Dummy (prior)": 0.5, "Persistencia": 1.0,  # Reglas binarias: alerta si el indicador es 1
            "Persistencia (8 sem.)": mb.umbral_f1(va[cf.TARGET], lb_va["Persistencia (8 sem.)"]), "Ingenuo estacional": 1.0}  # Persistencia continua: F1 en 2022
puntajes = {"Regresión logística": p_te, **lb_te}  # Puntajes en prueba
print(f"Umbral de la regresión logística (máximo F1 en 2022): {umbral:.3f}")  # Umbral
print(f"Prevalencia en la prueba: {te[cf.TARGET].mean():.1%} (entrenamiento {tr[cf.TARGET].mean():.1%})")  # Cambio de prevalencia

Umbral de la regresión logística (máximo F1 en 2022): 0.244
Prevalencia en la prueba: 30.3% (entrenamiento 10.3%)


In [104]:
resultados = {}  # Métricas con IC 95 % (bootstrap de 1 000 réplicas por municipio)
for nombre, p in puntajes.items():  # Cada modelo
    resultados[nombre] = mb.bootstrap_metricas(te[cf.TARGET], p, umbrales[nombre], te["COD_MUN_N"], B=1000)  # Métricas e IC
tab = pd.concat({n: r.apply(lambda f: f"{f['valor']:.3f} [{f['IC 2,5 %']:.3f}; {f['IC 97,5 %']:.3f}]", axis=1) for n, r in resultados.items()}, axis=1).T  # Formato valor [IC]
tab

,AUC-ROC,AUC-PR,Accuracy,Precision,Recall,F1,MCC,Brier
Regresión logística,0.885 [0.875; 0.895],0.809 [0.782; 0.833],0.825 [0.814; 0.836],0.700 [0.677; 0.720],0.742 [0.718; 0.765],0.720 [0.698; 0.741],0.594 [0.568; 0.616],0.122 [0.115; 0.129]
Dummy (prior),0.500 [0.500; 0.500],0.303 [0.281; 0.327],0.697 [0.673; 0.719],0.000 [0.000; 0.000],0.000 [0.000; 0.000],0.000 [0.000; 0.000],0.000 [0.000; 0.000],0.251 [0.234; 0.270]
Persistencia,0.787 [0.774; 0.800],0.588 [0.557; 0.617],0.823 [0.812; 0.834],0.712 [0.687; 0.734],0.697 [0.673; 0.719],0.705 [0.680; 0.727],0.578 [0.551; 0.603],0.177 [0.166; 0.188]
Persistencia (8 sem.),0.859 [0.849; 0.869],0.728 [0.699; 0.754],0.805 [0.794; 0.816],0.655 [0.629; 0.678],0.756 [0.735; 0.775],0.701 [0.678; 0.723],0.561 [0.536; 0.584],0.135 [0.127; 0.142]
Ingenuo estacional,0.540 [0.527; 0.552],0.325 [0.300; 0.352],0.662 [0.642; 0.680],0.399 [0.359; 0.439],0.230 [0.203; 0.258],0.292 [0.260; 0.323],0.096 [0.065; 0.126],0.338 [0.320; 0.358]


In [105]:
comparaciones = []  # Diferencias pareadas con la mejor línea base
for base in ["Persistencia (8 sem.)", "Persistencia"]:  # Líneas base más fuertes
    for nombre_m, met in [("AUC-ROC", roc_auc_score), ("AUC-PR", average_precision_score)]:  # Métricas
        dif, lo, hi, p = mb.bootstrap_diferencia(te[cf.TARGET], p_te, puntajes[base], te["COD_MUN_N"], metrica=met, B=1000)  # Bootstrap pareado
        comparaciones.append({"comparación": f"Regresión logística − {base}", "métrica": nombre_m, "diferencia": dif,  # Resultado
                              "IC 2,5 %": lo, "IC 97,5 %": hi, "p (bootstrap)": p})
pd.DataFrame(comparaciones).round(4)

,comparación,métrica,diferencia,"IC 2,5 %","IC 97,5 %",p (bootstrap)
0,Regresión logística − Persistencia (8 sem.),AUC-ROC,0.0262,0.0225,0.0296,0.0010
1,Regresión logística − Persistencia (8 sem.),AUC-PR,0.0813,0.0745,0.0883,0.0010
2,Regresión logística − Persistencia,AUC-ROC,0.0980,0.0931,0.1036,0.0010
3,Regresión logística − Persistencia,AUC-PR,0.2206,0.2109,0.2290,0.0010


**Interpretación.** La regresión logística supera con claridad a todas las líneas base en las métricas que importan: AUC-ROC ≈ 0,885 y AUC-PR ≈ 0,81, frente a 0,5 / 0,30 del Dummy y 0,86 / 0,73 de la mejor persistencia; las diferencias pareadas tienen intervalos que no contienen el 0. Su F1 y su MCC también son los más altos. Los intervalos (por municipio) son estrechos, de ± 0,01 en AUC-ROC, porque hay 376 municipios y dos años de prueba.

Ojo con la **exactitud**: la persistencia simple acierta casi lo mismo que el modelo (≈ 0,82) y el Dummy, que nunca alerta, ya llega a 0,70. La exactitud no distingue un buen sistema de alerta de uno trivial; por eso no se usa para decidir. El ingenuo estacional es casi inútil (AUC ≈ 0,54): los brotes no se repiten en la misma semana del año, lo que confirma la estacionalidad débil encontrada en el EDA.

## Matriz de confusión, curvas ROC y precisión-recall

In [106]:
pred = (p_te >= umbral).astype(int)  # Alertas
cm = confusion_matrix(te[cf.TARGET], pred)  # [[VN, FP], [FN, VP]]
etiquetas = [[f"VN<br>{cm[0, 0]:,}", f"FP<br>{cm[0, 1]:,}"], [f"FN<br>{cm[1, 0]:,}", f"VP<br>{cm[1, 1]:,}"]]  # Textos
fig = go.Figure(go.Heatmap(z=cm, x=["Predicho: sin brote", "Predicho: brote"], y=["Real: sin brote", "Real: brote"], text=etiquetas,  # Mapa de calor
                           texttemplate="%{text}", colorscale="Blues", showscale=False, textfont=dict(size=16)))
fig.update_layout(title=f"Matriz de confusión en la prueba (umbral {umbral:.3f})", yaxis_autorange="reversed")  # Formato
mostrar(fig, 380)  # Muestra
print(f"Anticipa {cm[1, 1]:,} de {cm[1].sum():,} brotes (recall {cm[1, 1] / cm[1].sum():.1%}); {cm[0, 1]:,} falsas alarmas; "
      f"{cm[:, 1].sum() / te['time_idx'].nunique():.0f} alertas por semana en promedio")  # Lectura operativa

Anticipa 8,453 de 11,395 brotes (recall 74.2%); 3,624 falsas alarmas; 121 alertas por semana en promedio


In [107]:
fig = make_subplots(1, 2, subplot_titles=("Curva ROC", "Curva precisión-recall"))  # Dos paneles
colores = {"Regresión logística": AZUL, "Persistencia (8 sem.)": VERDE, "Persistencia": AMBAR, "Ingenuo estacional": MORADO, "Dummy (prior)": GRIS}  # Colores
for nombre, col in colores.items():  # Cada modelo
    p = puntajes[nombre]  # Puntajes
    fpr, tpr, _ = roc_curve(te[cf.TARGET], p)  # ROC
    pr, rc, _ = precision_recall_curve(te[cf.TARGET], p)  # PR
    fig.add_scatter(x=fpr, y=tpr, name=f"{nombre} ({roc_auc_score(te[cf.TARGET], p):.3f})", line=dict(color=col), row=1, col=1)  # ROC
    fig.add_scatter(x=rc, y=pr, showlegend=False, line=dict(color=col), row=1, col=2)  # PR
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=GRIS, dash="dot"), showlegend=False, row=1, col=1)  # Azar ROC
fig.add_hline(y=te[cf.TARGET].mean(), line_dash="dot", line_color=GRIS, row=1, col=2)  # Azar PR = prevalencia
fig.update_xaxes(title_text="tasa de falsos positivos", row=1, col=1)  # Ejes
fig.update_yaxes(title_text="recall", row=1, col=1)  # Ejes
fig.update_xaxes(title_text="recall", row=1, col=2)  # Ejes
fig.update_yaxes(title_text="precisión", row=1, col=2)  # Ejes
fig.update_layout(title="Curvas en la prueba 2023–2024 (leyenda: AUC-ROC)")  # Título
mostrar(fig, 440)  # Muestra

## Calibración

In [108]:
frac_te, media_te, ece_te = mb.calibracion(te[cf.TARGET].to_numpy(), p_te)  # Calibración en prueba
frac_va, media_va, ece_va = mb.calibracion(va[cf.TARGET].to_numpy(), p_va)  # Calibración en validación
fig = go.Figure()  # Curva de confiabilidad
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=GRIS, dash="dot"), name="Calibración perfecta")  # Diagonal
fig.add_scatter(x=media_va, y=frac_va, mode="lines+markers", line=dict(color=VERDE), name=f"Validación 2022 (ECE {ece_va:.3f})")  # Validación
fig.add_scatter(x=media_te, y=frac_te, mode="lines+markers", line=dict(color=AZUL), name=f"Prueba 2023–24 (ECE {ece_te:.3f})")  # Prueba
fig.update_layout(title="Curva de calibración por deciles de probabilidad", xaxis_title="probabilidad predicha", yaxis_title="proporción observada de brotes")  # Formato
mostrar(fig, 420)  # Muestra
print(f"Brier: validación {np.mean((p_va - va[cf.TARGET]) ** 2):.4f} · prueba {np.mean((p_te - te[cf.TARGET]) ** 2):.4f} · "
      f"probabilidad media en prueba {p_te.mean():.3f} frente a prevalencia {te[cf.TARGET].mean():.3f}")  # Resumen

Brier: validación 0.0673 · prueba 0.1217 · probabilidad media en prueba 0.257 frente a prevalencia 0.303


**Interpretación.** En la validación de 2022 (prevalencia parecida a la del entrenamiento) el error de calibración es pequeño (ECE ≈ 0,03), aunque el modelo ya subestima un poco (probabilidad media 9,7 % frente a 12,4 % observado). En la prueba la **subestimación** crece (ECE ≈ 0,05; 25,7 % frente a 30,3 %), sobre todo en las probabilidades intermedias: la prevalencia de 2023–2024 (≈ 30 %) triplica la del entrenamiento (≈ 10 %), y la regresión logística aprendió una intersección acorde con la prevalencia antigua. Es la consecuencia directa del *prior shift* documentado en el EDA. El ordenamiento (AUC) se mantiene; lo que se desajusta es la escala, y se corrige recalibrando la intersección o el umbral con datos recientes.

## Diagnóstico de residuos: autocorrelación temporal y espacial

Residuo de respuesta: $r = y - \hat p$. Si el modelo capturara toda la estructura, los residuos no tendrían dependencia temporal (ACF ≈ 0) ni espacial (I de Moran ≈ 0).

In [109]:
resid = (te[cf.TARGET] - p_te).to_numpy()  # Residuos en la prueba
acf_r, n_mun = mb.acf_residuos(te, resid, rezagos=12)  # ACF media dentro de cada municipio
banda = 1.96 / np.sqrt(te.groupby("COD_MUN_N").size().mean())  # Banda aproximada por serie
fig = go.Figure(go.Bar(x=np.arange(1, 13), y=acf_r, marker_color=AZUL))  # ACF
fig.add_hline(y=banda, line_dash="dot", line_color=GRIS)  # Banda
fig.add_hline(y=-banda, line_dash="dot", line_color=GRIS)  # Banda
fig.update_layout(title=f"ACF media de los residuos dentro de los municipios ({n_mun} series)", xaxis_title="rezago (semanas)", yaxis_title="autocorrelación")  # Formato
mostrar(fig, 340)  # Muestra
m = d.municipios()  # Centroides
D = sp.haversine(m["latitud"].to_numpy(), m["longitud"].to_numpy())  # Distancias
r_mun = te.assign(r=resid).groupby("COD_MUN_N")["r"].mean().loc[m["COD_MUN_N"]]  # Residuo medio por municipio
mor = sp.moran(r_mun.to_numpy(), sp.pesos_knn(D, 6), permutaciones=999)  # Moran de los residuos
print(f"ACF de los residuos: rezago 1 = {acf_r[0]:.3f}, rezago 4 = {acf_r[3]:.3f} · I de Moran del residuo medio por municipio: {mor['I']:.3f} (p = {mor['p']:.3f})")  # Resumen
fig = px.scatter_geo(m.assign(residuo=r_mun.to_numpy()), lat="latitud", lon="longitud", color="residuo", hover_name="municipio",  # Mapa de residuos
                     color_continuous_scale="RdBu_r", color_continuous_midpoint=0)
fig.update_traces(marker=dict(size=7, line=dict(width=0.3)))  # Tamaño
fig.update_layout(title="Residuo medio por municipio en la prueba (rojo: brotes no anticipados)")  # Título
mostrar(mapa_base(fig), 540)  # Muestra

ACF de los residuos: rezago 1 = 0.339, rezago 4 = 0.010 · I de Moran del residuo medio por municipio: 0.158 (p = 0.001)


**Interpretación.** Los residuos **sí tienen dependencia**: autocorrelación temporal clara en los primeros 3 rezagos (los episodios de brote duran varias semanas y el modelo se equivoca en rachas) y autocorrelación espacial significativa (municipios vecinos comparten el signo del error, con zonas enteras subestimadas en 2023–2024). Según la guía, esto indica **información estructural sin capturar**: la dinámica de vecindad (Knox y Moran del EDA) y la no linealidad del clima. Es la motivación para los modelos de los siguientes entregables: variables de rezago espacial y modelos no lineales.

## Curva de aprendizaje y sobreajuste

In [110]:
curva = mb.curva_aprendizaje(tr, va, C=grid.best_params_["clf__C"], cw=grid.best_params_["clf__class_weight"])  # Fracciones de municipios
fig = make_subplots(1, 2, subplot_titles=("AUC-ROC", "AUC-PR"))  # Dos paneles
for i, met in enumerate(["AUC-ROC", "AUC-PR"]):  # Métricas
    fig.add_scatter(x=curva["filas"], y=curva[f"{met} entrenamiento"], name="Entrenamiento", line=dict(color=AZUL), showlegend=i == 0, row=1, col=i + 1)  # Entrenamiento
    fig.add_scatter(x=curva["filas"], y=curva[f"{met} validación"], name="Validación 2022", line=dict(color=VERDE), showlegend=i == 0, row=1, col=i + 1)  # Validación
fig.update_xaxes(title_text="filas de entrenamiento")  # Eje
fig.update_layout(title="Curva de aprendizaje (fracciones crecientes de municipios)")  # Título
mostrar(fig, 360)  # Muestra
curva.round(4)

,fraccion,filas,AUC-PR entrenamiento,AUC-PR validación,AUC-ROC entrenamiento,AUC-ROC validación
0,0.1000,13414,0.7160,0.6722,0.9202,0.9028
1,0.2000,26475,0.6758,0.6730,0.9163,0.8977
2,0.3500,46596,0.6513,0.6721,0.9088,0.8982
3,0.5000,66364,0.6356,0.6708,0.9075,0.8998
4,0.7500,99546,0.6417,0.6714,0.9070,0.9005
5,1.0000,132728,0.6532,0.6696,0.9067,0.8978


In [111]:
p_tr = modelo.predict_proba(tr[cf.FEATURES])[:, 1]  # Probabilidades en entrenamiento
brecha = pd.DataFrame({conj: mb.metricas(yy, pp, umbral) for conj, yy, pp in [("Entrenamiento 2015–21", tr[cf.TARGET], p_tr),  # Mismo umbral
                                                                            ("Validación 2022", va[cf.TARGET], p_va), ("Prueba 2023–24", te[cf.TARGET], p_te)]}).T
brecha.round(4)

,AUC-ROC,AUC-PR,Accuracy,Precision,Recall,F1,MCC,Brier
Entrenamiento 2015–21,0.9067,0.6532,0.9178,0.5950,0.6266,0.6104,0.5647,0.0558
Validación 2022,0.8978,0.6696,0.9115,0.6632,0.5751,0.6160,0.5681,0.0673
Prueba 2023–24,0.8855,0.8090,0.8254,0.6999,0.7418,0.7203,0.5940,0.1217


**Interpretación.** La curva de aprendizaje es plana: con el 10 % de los municipios el modelo ya alcanza casi el mismo AUC de validación que con todos, y las curvas de entrenamiento y validación están cerca. El tamaño de muestra es más que suficiente para un modelo lineal y **no hay sobreajuste** (la brecha de AUC-ROC entre entrenamiento y prueba es de unas dos centésimas). El límite es de **sesgo**: el modelo es demasiado simple para la estructura no lineal y espacial; más datos no lo mejorarán, pero un modelo más flexible sí podría. Las diferencias de AUC-PR, precisión y Brier entre conjuntos se deben sobre todo al cambio de prevalencia, no al sobreajuste.

## Interpretación de los coeficientes

Los coeficientes están en unidades de desviación estándar (las variables se estandarizan): $e^{\beta}$ es el cambio en las *odds* de brote cuando la variable sube una desviación estándar, manteniendo las demás fijas. Los intervalos se obtienen reajustando el Pipeline en 200 remuestreos de municipios.

In [112]:
coef = mb.coeficientes_bootstrap(tr, grid.best_params_["clf__C"], grid.best_params_["clf__class_weight"], B=200)  # Coeficientes e IC
coef["odds ratio por 1 DE"] = np.exp(coef["coef"])  # Razón de odds
coef["IC excluye 0"] = (coef["IC 2,5 %"] > 0) | (coef["IC 97,5 %"] < 0)  # Significativo
orden = coef.sort_values("coef")  # Orden
fig = go.Figure(go.Bar(x=orden["coef"], y=orden["variable"], orientation="h", marker_color=np.where(orden["coef"] > 0, ROJO, AZUL),  # Coeficientes
                       error_x=dict(type="data", symmetric=False, array=orden["IC 97,5 %"] - orden["coef"], arrayminus=orden["coef"] - orden["IC 2,5 %"])))
fig.update_layout(title="Coeficientes estandarizados con IC 95 % (bootstrap de municipios)", xaxis_title="coeficiente (log-odds por 1 DE)")  # Formato
mostrar(fig, 760)  # Muestra
coef.set_index("variable").sort_values("coef", key=np.abs, ascending=False).round(3)

,coef,"IC 2,5 %","IC 97,5 %",odds ratio por 1 DE,IC excluye 0
variable,,,,,
brote_m8,0.6020,0.4810,0.6580,1.8260,True
lc_l1,0.4340,0.3720,0.5000,1.5430,True
ratio_umbral,0.4290,0.3810,0.4850,1.5360,True
lc_m12,-0.4020,-0.5330,-0.2310,0.6690,True
lc_m4,0.2580,0.2300,0.2850,1.2940,True
lc_l0,0.2440,0.1810,0.2970,1.2770,True
oni_anom,0.2300,0.1710,0.2870,1.2590,True
ndvi_mean_m12,-0.2040,-0.3070,-0.1160,0.8150,True
lc_l2,0.1970,0.1510,0.2520,1.2180,True


**Interpretación.**

* **Historia de casos.** Los coeficientes más grandes y estables: la proporción reciente de semanas en brote (`brote_m8`), los casos de las últimas semanas (`lc_l0`, `lc_l1`, `lc_m4`) y los casos frente al umbral objetivo (`ratio_umbral`) aumentan el riesgo. Las medias largas (`lc_m8`, `lc_m12`) tienen signo **negativo** condicional a las cortas: con los casos recientes fijos, una historia larga alta significa que el municipio ya venía en epidemia (y el brote tiende a agotarse); lo que predice un brote es la **aceleración** reciente. Es un efecto de colinealidad que debe leerse en conjunto, no variable por variable.
* **El Niño.** El ONI actual aumenta el riesgo (una desviación estándar multiplica las *odds* por ≈ 1,26).
* **Clima local.** Efectos pequeños, de signos mezclados entre ventanas (lluvia de 4–8 semanas positiva, de 12 negativa), coherentes con las relaciones no lineales vistas en el EDA. Los de temperatura no son distinguibles de 0: con el 64 % de la temperatura imputada, su señal está muy atenuada.
* **Contexto.** Más flujo de personas, más riesgo. La estacionalidad y la bandera de temperatura imputada no son distinguibles de 0.

## Generalización a zonas nuevas: esquemas espaciales y espacio-temporales

Bloques por departamento (`GroupKFold`, 5 bloques) con 50 km de separación: se quitan del entrenamiento los municipios a menos de 50 km de cualquier municipio del bloque evaluado.

In [113]:
from sklearn.model_selection import GroupKFold  # Bloques espaciales
mun_xy = m.set_index("COD_MUN_N")[["latitud", "longitud"]]  # Coordenadas por código
filas = []  # Resultados
for k, (i_tr, i_ev) in enumerate(GroupKFold(5).split(tr, groups=tr["depto"])):  # Bloques de departamentos
    deptos = set(tr.iloc[i_ev]["depto"])  # Departamentos evaluados
    m_ev = tr.iloc[i_ev]["COD_MUN_N"].unique()  # Municipios evaluados
    m_tr = tr.iloc[i_tr]["COD_MUN_N"].unique()  # Municipios candidatos al entrenamiento
    Dx = sp.haversine(np.r_[mun_xy.loc[m_tr, "latitud"], mun_xy.loc[m_ev, "latitud"]], np.r_[mun_xy.loc[m_tr, "longitud"], mun_xy.loc[m_ev, "longitud"]])[: len(m_tr), len(m_tr):]  # Distancias cruzadas
    lejos = m_tr[Dx.min(axis=1) > 50]  # Municipios a más de 50 km del bloque evaluado
    ent = tr[tr["COD_MUN_N"].isin(lejos)]  # Entrenamiento con zona de separación
    mdl = mb.pipeline_base().set_params(clf__C=grid.best_params_["clf__C"], clf__class_weight=grid.best_params_["clf__class_weight"]).fit(ent[cf.FEATURES], ent[cf.TARGET])  # Ajuste
    ev_mismo = tr.iloc[i_ev]  # Zonas nuevas, mismo periodo
    ev_futuro = te[te["depto"].isin(deptos)]  # Zonas nuevas y futuro
    filas.append({"bloque": k + 1, "departamentos": len(deptos), "municipios evaluados": len(m_ev), "municipios excluidos por la zona de 50 km": len(m_tr) - len(lejos),
                  "AUC-ROC zonas nuevas (2015–21)": roc_auc_score(ev_mismo[cf.TARGET], mdl.predict_proba(ev_mismo[cf.FEATURES])[:, 1]),
                  "AUC-ROC zonas nuevas + futuro (2023–24)": roc_auc_score(ev_futuro[cf.TARGET], mdl.predict_proba(ev_futuro[cf.FEATURES])[:, 1]),
                  "AUC-PR zonas nuevas + futuro (2023–24)": average_precision_score(ev_futuro[cf.TARGET], mdl.predict_proba(ev_futuro[cf.FEATURES])[:, 1])})
bloques = pd.DataFrame(filas).set_index("bloque")  # Tabla
bloques.round(4)

,departamentos,municipios evaluados,municipios excluidos por la zona de 50 km,AUC-ROC zonas nuevas (2015–21),AUC-ROC zonas nuevas + futuro (2023–24),AUC-PR zonas nuevas + futuro (2023–24)
bloque,,,,,,
1,7,76,32,0.9056,0.8826,0.7735
2,6,75,44,0.9008,0.8639,0.7813
3,6,75,39,0.8987,0.9044,0.8627
4,7,75,52,0.9276,0.8952,0.8125
5,6,75,27,0.8834,0.8695,0.7851


In [114]:
esquemas = pd.DataFrame({  # Resumen de los tres esquemas
    "AUC-ROC": [roc_auc_score(te[cf.TARGET], p_te), bloques["AUC-ROC zonas nuevas (2015–21)"].mean(), bloques["AUC-ROC zonas nuevas + futuro (2023–24)"].mean()],
    "dispersión entre bloques": [np.nan, bloques["AUC-ROC zonas nuevas (2015–21)"].std(), bloques["AUC-ROC zonas nuevas + futuro (2023–24)"].std()],
}, index=["1. Futuro, mismas zonas (prueba 2023–24)", "2. Zonas nuevas, mismo periodo", "3. Zonas nuevas y futuro"])
esquemas.round(4)

,AUC-ROC,dispersión entre bloques
"1. Futuro, mismas zonas (prueba 2023–24)",0.8855,NaN
"2. Zonas nuevas, mismo periodo",0.9032,0.0160
3. Zonas nuevas y futuro,0.8831,0.0170


**Interpretación.** El modelo **generaliza a departamentos que no vio**, incluso con la zona de separación de 50 km: el AUC por bloques es comparable al del esquema principal, con variación entre bloques (algunas regiones son más difíciles). Esto es esperable: las predictoras son relativas a cada municipio (casos frente a su canal endémico, anomalías climáticas) y no identifican lugares. El esquema 3 (zonas nuevas y futuro) es el más exigente y el más variable.

## Nota crítica: ¿un desempeño demasiado alto?

La exactitud en la prueba es ≈ 0,83, dentro del rango 80–90 % que la guía pide revisar. La revisión:

1. **Fuga de datos.** Auditada en el capítulo 6: ninguna predictora supera un AUC univariado de 0,9, la columna con información del futuro se descartó, la partición es temporal con purga y las particiones no comparten filas.
2. **Comparación con la línea base trivial.** El 83 % no es mérito del modelo: el Dummy que nunca alerta acierta el 70 % y la persistencia simple, el 82 %. Por eso el modelo se juzga por AUC-ROC, AUC-PR, F1 y MCC, donde sí supera a las líneas base con intervalos que excluyen el 0.
3. **Estructura temporal y espacial.** La validación respeta el tiempo (años hacia adelante con purga) y el espacio (bloques de departamentos con 50 km de separación); el desempeño se mantiene en ambos esquemas.
4. **Complejidad del dataset.** El problema no es trivial: la mejor regla simple llega a AUC-PR 0,73 y el modelo a 0,81, los residuos conservan estructura espacial y temporal sin explicar y el clima tiene efectos no lineales. Hay margen claro para los modelos de los siguientes entregables.

**Conclusión:** el desempeño alto en exactitud es consecuencia del desbalance y de la persistencia natural de las epidemias, no de una fuga; el dataset es adecuado para el curso.

# Conclusiones, limitaciones y reproducibilidad

## Hallazgos del EDA que guían el modelado

| Hallazgo | Evidencia | Decisión |
|---|---|---|
| La historia de casos domina la predicción | AUC univariados de 0,77–0,88; información mutua máxima; concepto estable año a año | Rezagos y medias móviles del log de casos; persistencia como línea base exigente |
| El clima actúa con rezago y de forma no lineal | Correlación cruzada débil con máximo a 12–15 semanas (lluvia, NDVI); riesgo máximo hacia 25–28 °C | Medias móviles largas y anomalías; modelos no lineales en los siguientes entregables |
| El Niño eleva el riesgo | Brote en 4 semanas: 16,8 % en El Niño frente a 4,5 % en La Niña (V de Cramér 0,17) | ONI actual y rezagado 8 semanas |
| Desbalance que cambia por año | Prevalencia del 1,7 % (2017) al 25 % (2019); 30 % en la prueba | AUC-PR, MCC y Brier; validación por años; umbral en validación |
| Estacionalidad débil, ondas de varios años | Fuerza estacional 0,02 frente a 0,66 de la tendencia (STL) | Seno y coseno de la semana; variables relativas al municipio |
| Autocorrelación espacial | Moran ≈ 0,31 (incidencia), 76 municipios Alto-Alto, alcance ≈ 100 km | Validación por bloques de departamentos con 50 km de separación |
| Interacción espacio-tiempo | Knox: 13–28 % más pares cercanos en espacio y tiempo de lo esperado (p = 0,001) | Variables de vecindad rezagadas para los siguientes modelos |
| Temperatura imputada en el 64 % | MAR estructural: municipios pequeños sin estación (d ≈ −0,7 en población) | Indicador `temp_missing`; no se reimputa con vecinos |
| Atípicos = epidemias | Atípicos multivariados con tasa de brote de ≈ 39 % frente a ≈ 9 % | No se eliminan; logaritmo |
| Multicolinealidad | VIF > 10 entre medias móviles | Regularización L2 |

## Modelo base

La regresión logística con penalización L2 ($C = 0{,}01$, sin ponderar clases) alcanza en la prueba 2023–2024 un
**AUC-ROC de 0,885** y un **AUC-PR de 0,81**, con intervalos de confianza por municipio de unas ± 0,01, y supera de
forma significativa a la mejor línea base trivial (persistencia de 8 semanas: 0,86 y 0,73). La exactitud (≈ 0,83) no
es informativa: el Dummy llega a 0,70 y la persistencia simple a 0,82.

El diagnóstico marca el camino de los siguientes entregables:

* **Sin sobreajuste y con muestra suficiente**: curva de aprendizaje plana y brecha pequeña entre entrenamiento y prueba.
* **Sesgo del modelo**: los residuos conservan autocorrelación temporal (rachas de error) y espacial (I de Moran
  significativo), y el clima tiene efectos no lineales que un modelo lineal no captura.
* **Cambio de prevalencia**: el modelo ordena bien el riesgo en 2023–2024, pero sus probabilidades quedan
  subestimadas; hay que recalibrar el umbral o la intersección con datos recientes.
* **Generaliza a zonas nuevas**: con bloques de departamentos y 50 km de separación, el desempeño se mantiene.

## Limitaciones

* **Representatividad.** 376 de 1 122 municipios (≈ 60 % de la población), con sobrerrepresentación del Caribe y la
  Orinoquía; los resultados aplican a municipios con transmisión habitual.
* **Calidad de la notificación.** Los casos son los notificados al SIVIGILA; hay subregistro variable y caídas como la de 2020
  por la pandemia.
* **Clima imputado.** La temperatura falta en los municipios pequeños; el efecto real de la temperatura puede estar
  atenuado.
* **Escala espacial.** Las coordenadas son centroides; no hay información dentro de cada municipio ni polígonos para
  vecindades por contigüidad.
* **Pocos años de prueba.** Dos años, uno de ellos (2024) excepcional.
* **Riesgos éticos.** Datos agregados sin información personal; el riesgo de reidentificación es bajo. El principal
  riesgo es de equidad: si la notificación es peor en municipios pobres, el modelo puede subestimar su riesgo.

## Reproducibilidad

* **Semilla:** `SEMILLA = 42` en `src/config.py`, usada en todos los muestreos, permutaciones, bootstrap y modelos.
* **Entorno:** Python 3.9.25 con las versiones fijadas en `entregable1/requirements.txt`.
* **Código:** `entregable1/e1/` (datos, estadística, análisis espacial, modelo base y figuras) y `src/` (canal endémico y
  variables). Todo el código está comentado línea por línea.
* **Ejecución de principio a fin** (desde la raíz del proyecto):

```bash
python -m venv .venv
.venv/Scripts/python -m pip install -r entregable1/requirements.txt
cd entregable1/construir && ../../.venv/Scripts/python construir_todo.py && cd ../cuadernos
for n in *.ipynb; do ../../.venv/Scripts/jupyter nbconvert --to notebook --execute --inplace $n; done
cd ../.. && .venv/Scripts/jupyter-book build entregable1
```

## Entregables

| Entregable | Ubicación |
|---|---|
| Informe en Jupyter Book | `entregable1/_build/html/index.html` |
| Base de datos | `data/final_dataset.csv` (panel) y `data/divipola_municipios.csv` (centroides DANE, CC BY-SA 4.0) |
| Dependencias y semillas | `entregable1/requirements.txt` y `src/config.py` |
